# MediFlow 4차만 추가 Test · 기본~3차 결과 재사용

**GPU 선택 → 모두 실행 → Google Drive 연결** 순서로 실행합니다.
새 ZIP 업로드, Target 복호화 키 입력, 추가 학습은 필요 없습니다.
기존 공통 ZIP v3·원본 데이터 ZIP·1차 분할 목록과 **조정 전 4차** 저장 모델을 사용합니다.
기본·1·2·3차의 검증된 Test 결과는 이 노트북에 포함했으며 다시 추론하지 않습니다.
재설정 4차는 사용자 결정으로 실행하지 않습니다.

피부 700장·얼굴 400장·두피 1,252장의 원본 Test를 보존합니다.
4차 모델 3개에 원본·기존 조명·기존 확장·4차 조명·4차 확장을 적용해 **15개 조합만 새로 평가**합니다.
기본~3차 기존 36개 결과를 재사용해 기존 세 조건에서 비교합니다.
새 4차 변환 두 조건은 4차만 측정하므로 이전 모델 대비 변화량을 비워 둡니다.
난수 seed 1707, 클래스 순서, 입력 크기, float32 RGB 0–255를 유지합니다.
이전에 확인한 Test를 다시 쓰는 후속 비교이며 새로운 미관측 자료의 검증으로 표현하지 않습니다.

후속 후보는 4차 Validation 비교에서 피부·얼굴 4차, 두피 2차로 고정했습니다.
이 선정은 기본·2차·4차 범위이며 1차·3차는 이전 Test 기록으로 비교합니다.
Test 점수로 후보나 변환을 다시 정하지 않습니다.
실제 장비 질환 정확도는 정답 자료가 없어 측정하지 않습니다.

Drive 결과: `MediFlow_adaptation_runs/4ae6e07eb4af1225/stage4_test_dda96d8dc716`

진행 장수·실제 오류·로그를 표시하고 완료 분야는 같은 계획으로 재사용합니다.
비교표·클래스별 지표·혼동행렬·비교 그래프·사진별 모델 점수를 저장합니다.
완료하면 `MediFlow_Stage4_Test_Results_20261111.zip`을 다운로드합니다.
모델과 원본 이미지는 결과 ZIP에 중복 포함하지 않습니다.


In [ ]:
import hashlib
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

from google.colab import drive

print("MediFlow 4차만 Test · 코드 사본 검증 수정본", flush=True)
drive.mount("/content/drive")
my_drive = Path("/content/drive/MyDrive")
project = my_drive / "mediflow_Project"
bundle = project / "MediFlow_Adaptation_Common_v3.zip"
run_root = my_drive / "MediFlow_adaptation_runs" / "4ae6e07eb4af1225"
stages = [run_root / f"stage{n}" for n in (1, 2, 3)]
stages.append(run_root / 'measured_strength_492ee01afb45')
out = run_root / "stage4_test_dda96d8dc716"
if not all((stage / "bundle.json").is_file() for stage in stages[:1]) or not (stages[3] / "study.json").is_file():
    raise FileNotFoundError("Drive에서 1차 분할 목록과 조정 전 4차 완료 폴더를 확인하세요.")
log_root = out / "logs"
log_root.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex[:12]
log_path = log_root / f"run_{run_id}.log"
print("실행 로그 저장 위치", log_path, flush=True)

with tempfile.TemporaryDirectory(prefix="mediflow_final_test_", dir="/content") as temp:
    work = Path(temp)
    runtime = work / "runtime"
    local_bundle = work / bundle.name
    print("공통 ZIP v3 확인 중", flush=True)
    shutil.copyfile(bundle, local_bundle)
    with local_bundle.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe":
            raise ValueError("공통 ZIP v3의 해시가 일치하지 않습니다.")
    with ZipFile(local_bundle) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if member.is_dir() or not path.parts or path.parts[0] not in (
                "src", "configs", "references", "results"
            ):
                continue
            if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
                raise ValueError("공통 ZIP에 예상치 못한 경로가 있습니다.")
            archive.extract(member, runtime)
    local_bundle.unlink()
    for name, contents in {'adaptation_final_test.py': b'"""Run the locked camera-adaptation comparison on the original labeled Test split."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport io\nimport platform\nimport tempfile\nimport zipfile\nfrom contextlib import nullcontext\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport cv2\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import adaptation_stage1 as training\nfrom mediflow_datasets import camera_extended as extended\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as experiment\nfrom mediflow_datasets.adaptation_stage1_colab import copy_source\nfrom mediflow_datasets.domain_adaptation_batch import digest, find_source\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nCONDITIONS = ("clean", "lighting", "expanded")\nMODEL_LABELS = {\n    "initial": "Before adaptation",\n    "stage1_lighting": "Stage 1",\n    "stage2_expanded": "Stage 2",\n    "stage3_aligned": "Stage 3",\n}\nDOMAIN_LABELS = {"skin": "Microscope skin", "web_skin": "Webcam face", "hair": "Microscope scalp"}\n\n\ndef prepare_test(archive, task, lock, manifest):\n    if digest(manifest) != lock["manifest_sha256"]:\n        raise ValueError(f"{task[\'domain\']}: Test manifest SHA-256 mismatch")\n    rows, _ = data.source_index(archive, task["classes"], task["source_sha256"])\n    original_test = rows["original", "test"]\n    declared = data.read_json(manifest)["original"]["test"]\n    actual = [(row.path, row.label) for row in original_test]\n    expected = [(row["path"], row["label"]) for row in declared]\n    if actual != expected or len(actual) != lock["test_count"]:\n        raise ValueError(f"{task[\'domain\']}: original Test rows differ from the locked manifest")\n    return original_test\n\n\ndef evaluate_zip(archive_path, rows, task, spec, reference, predictors, seed):\n    """Read each Test image once; feed the exact same arrays to all model arms."""\n    scores = {name: {condition: [] for condition in CONDITIONS} for name in predictors}\n    size, batch_size = spec["size"], task["batch_size"]\n    with zipfile.ZipFile(archive_path) as archive:\n        for start in range(0, len(rows), batch_size):\n            batch = rows[start : start + batch_size]\n            clean, lighting, expanded = [], [], []\n            for row in batch:\n                image = experiment.image_tensor(archive.read(row.path), size, task["input_policy"])\n                item_seed = experiment.item_seed(row, seed)\n                clean.append(image)\n                lighting.append(\n                    experiment.transform_image(\n                        image, reference["extended_profile"]["lighting"], "target_guided", item_seed\n                    )\n                )\n                expanded.append(\n                    extended.transform(image, reference["extended_profile"], "expanded", item_seed)\n                )\n            for condition, inputs in zip(CONDITIONS, (clean, lighting, expanded), strict=True):\n                tensor = tf.convert_to_tensor(np.stack(inputs), dtype=tf.float32)\n                for name, predict in predictors.items():\n                    result = predict(tensor).numpy()\n                    if result.shape != (len(batch), len(task["classes"])):\n                        raise ValueError(f"{name}: unexpected model output shape {result.shape}")\n                    scores[name][condition].append(result)\n            done = start + len(batch)\n            if start == 0 or start % (batch_size * 20) == 0 or done == len(rows):\n                training.show(\n                    f"[{task[\'domain\']}] Test \xec\x98\x88\xec\xb8\xa1 \xc2\xb7 \xeb\xaa\xa8\xeb\x8d\xb8 4\xea\xb0\x9c \xc3\x97 \xec\xa1\xb0\xea\xb1\xb4 3\xea\xb0\x9c", done, len(rows)\n                )\n    return {\n        name: {\n            condition: experiment.classification(\n                rows, np.concatenate(batches), len(task["classes"])\n            )\n            for condition, batches in results.items()\n        }\n        for name, results in scores.items()\n    }\n\n\ndef domain_result(runtime, stages, domain, task, lock, seed, archive):\n    print(f"[{domain}] \xec\x9b\x90\xeb\xb3\xb8 ZIP \xed\x95\xb4\xec\x8b\x9c\xec\x99\x80 Test \xeb\xaa\xa9\xeb\xa1\x9d \xed\x99\x95\xec\x9d\xb8", flush=True)\n    rows = prepare_test(archive, task, lock, stages[0] / domain / "source_manifest.json")\n    reference = data.read_json(runtime / "references" / f"{domain}.json")\n    if training.object_hash(reference) != lock["reference_sha256"]:\n        raise ValueError(f"{domain}: camera profile mismatch")\n    spec = task["models"][lock["start_role"]]\n    predictors = {}\n    for name, model in lock["models"].items():\n        root = runtime if model["stage"] == 0 else stages[model["stage"] - 1]\n        path = data.local_path(root, model["path"])\n        if digest(path) != model["sha256"]:\n            raise ValueError(f"{domain}/{name}: checkpoint SHA-256 mismatch")\n        print(f"[{domain}] {name} \xeb\xaa\xa8\xeb\x8d\xb8 \xeb\xa1\x9c\xeb\x94\xa9", flush=True)\n        predictors[name] = training.predictor(keras.models.load_model(path, compile=False), spec)\n    metrics = evaluate_zip(archive, rows, task, spec, reference, predictors, seed)\n    return {\n        "domain": domain,\n        "classes": task["classes"],\n        "test_count": len(rows),\n        "source_sha256": task["source_sha256"],\n        "reference_sha256": lock["reference_sha256"],\n        "input_policy": task["input_policy"],\n        "selected_on_validation": lock["selected_on_validation"],\n        "metrics": metrics,\n    }\n\n\ndef write_comparison(output, results):\n    stream = io.StringIO()\n    writer = csv.writer(stream)\n    writer.writerow(\n        [\n            "domain",\n            "model",\n            "selected_on_validation",\n            "condition",\n            "count",\n            "accuracy",\n            "macro_f1",\n            "delta_accuracy_vs_initial",\n            "delta_macro_f1_vs_initial",\n        ]\n    )\n    for result in results:\n        for model, conditions in result["metrics"].items():\n            for condition, metrics in conditions.items():\n                baseline = result["metrics"]["initial"][condition]\n                writer.writerow(\n                    [\n                        result["domain"],\n                        model,\n                        model == result["selected_on_validation"],\n                        condition,\n                        metrics["labeled_count"],\n                        metrics["accuracy"],\n                        metrics["macro_f1"],\n                        metrics["accuracy"] - baseline["accuracy"],\n                        metrics["macro_f1"] - baseline["macro_f1"],\n                    ]\n                )\n    (output / "test_comparison.csv").write_text(stream.getvalue(), encoding="utf-8", newline="")\n\n\ndef write_class_metrics(output, results):\n    with (output / "class_metrics.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "domain",\n                "model",\n                "condition",\n                "class_index",\n                "class_name",\n                "support",\n                "precision",\n                "recall",\n                "f1",\n            ]\n        )\n        for result in results:\n            for model, conditions in result["metrics"].items():\n                for condition, metrics in conditions.items():\n                    for index, name in enumerate(result["classes"]):\n                        writer.writerow(\n                            [\n                                result["domain"],\n                                model,\n                                condition,\n                                index,\n                                name,\n                                metrics["support"][index],\n                                metrics["precision"][index],\n                                metrics["recall"][index],\n                                metrics["class_f1"][index],\n                            ]\n                        )\n\n\ndef write_plots(output, results):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    from matplotlib import pyplot as plt\n\n    plots = output / "plots"\n    plots.mkdir(exist_ok=True)\n    fig, axes = plt.subplots(3, 2, figsize=(13, 11), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            for column, metric in enumerate(("accuracy", "macro_f1")):\n                axis = axes[row, column]\n                for model, conditions in result["metrics"].items():\n                    label = MODEL_LABELS[model]\n                    if model == result["selected_on_validation"]:\n                        label += " (fixed on Validation)"\n                    axis.plot(\n                        range(3),\n                        [conditions[c][metric] for c in CONDITIONS],\n                        marker="o",\n                        label=label,\n                    )\n                axis.set_xticks(range(3), ["Original", "Lighting", "Expanded"])\n                axis.set_ylim(0, 1.01)\n                axis.set_title(f"{DOMAIN_LABELS[result[\'domain\']]} \xc2\xb7 {metric}")\n                axis.grid(alpha=0.2)\n                axis.legend(fontsize=8)\n        fig.suptitle("MediFlow \xc2\xb7 Original labeled Test comparison", fontsize=16)\n        fig.savefig(plots / "test_comparison.png", dpi=180)\n    finally:\n        plt.close(fig)\n    fig, axes = plt.subplots(3, 3, figsize=(16, 14), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            model = result["selected_on_validation"]\n            for column, condition in enumerate(CONDITIONS):\n                axis = axes[row, column]\n                matrix = np.asarray(result["metrics"][model][condition]["confusion_matrix"])\n                axis.imshow(matrix, cmap="Blues", vmin=0)\n                indexes = range(len(result["classes"]))\n                axis.set_xticks(indexes)\n                axis.set_yticks(indexes)\n                axis.set_xlabel("Predicted class index")\n                axis.set_ylabel("True class index")\n                axis.set_title(\n                    f"{DOMAIN_LABELS[result[\'domain\']]} \xc2\xb7 {MODEL_LABELS[model]} \xc2\xb7 {condition}"\n                )\n                for i, j in np.ndindex(matrix.shape):\n                    axis.text(\n                        j,\n                        i,\n                        str(matrix[i, j]),\n                        ha="center",\n                        va="center",\n                        fontsize=7,\n                        color="white" if matrix[i, j] > matrix.max() / 2 else "black",\n                    )\n        fig.suptitle("Validation-fixed candidates \xc2\xb7 Class names in class_metrics.csv", fontsize=16)\n        fig.savefig(plots / "selected_confusion_matrices.png", dpi=180)\n    finally:\n        plt.close(fig)\n\n\ndef run(runtime, stages, dataset_root, plan_path, output, work=None):\n    plan = data.read_json(plan_path)\n    config = data.read_json(runtime / "configs" / "adaptation_stage1.json")\n    if training.object_hash(config) != plan["configuration_sha256"]:\n        raise ValueError("Common bundle configuration differs from the locked plan")\n    if digest(training.__file__) != plan["predictor_code_sha256"]:\n        raise ValueError("Predictor differs from the Stage 3 validation code")\n    identity = {"plan_sha256": digest(plan_path), "evaluator_sha256": digest(__file__)}\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "evaluation_plan.json"\n    record = {**identity, "plan": plan}\n    if marker.exists() and data.read_json(marker) != record:\n        raise ValueError("Output belongs to another evaluation plan")\n    write_progress(marker, record)\n    code = output / "code"\n    code.mkdir(exist_ok=True)\n    for path in (Path(__file__), Path(training.__file__)):\n        (code / path.name).write_bytes(path.read_bytes())\n    results = []\n    for domain in training.DOMAINS:\n        task, lock = config["tasks"][domain], plan["domains"][domain]\n        destination = output / f"{domain}.json"\n        if destination.is_file():\n            result = data.read_json(destination)\n            if result["identity"] != identity:\n                raise ValueError(f"{domain}: saved result identity mismatch")\n            print(f"[{domain}] \xeb\x8f\x99\xec\x9d\xbc \xea\xb3\x84\xed\x9a\x8d\xec\x9d\x98 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        else:\n            source = find_source(dataset_root, task)\n            with tempfile.TemporaryDirectory(dir=work) if work else nullcontext() as scratch:\n                archive = source\n                if scratch:\n                    archive = Path(scratch) / "source.zip"\n                    copy_source(source, archive)\n                result = domain_result(\n                    runtime, stages, domain, task, lock, plan["transform_seed"], archive\n                )\n            result.update(\n                identity=identity,\n                environment={\n                    "python": platform.python_version(),\n                    "tensorflow": tf.__version__,\n                    "keras": keras.__version__,\n                    "numpy": np.__version__,\n                    "opencv": cv2.__version__,\n                    "matplotlib": version("matplotlib"),\n                    "gpu": [\n                        tf.config.experimental.get_device_details(d)\n                        for d in tf.config.list_physical_devices("GPU")\n                    ],\n                },\n            )\n            write_progress(destination, result)\n            keras.backend.clear_session()\n        results.append(result)\n    write_comparison(output, results)\n    write_class_metrics(output, results)\n    write_plots(output, results)\n    write_progress(\n        output / "summary.json",\n        {\n            **identity,\n            "completed_domains": [r["domain"] for r in results],\n            "comparisons": 36,\n            "source_test_evaluated": True,\n            "real_device_accuracy": None,\n        },\n    )\n    print(f"\xec\xb5\x9c\xec\xa2\x85 Test \xea\xb2\xb0\xea\xb3\xbc: {output / \'test_comparison.csv\'}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--runtime", type=Path, required=True)\n    parser.add_argument("--stages", type=Path, nargs=3, required=True)\n    parser.add_argument("--datasets", type=Path, required=True)\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--work", type=Path)\n    args = parser.parse_args()\n    run(args.runtime, args.stages, args.datasets, args.plan, args.output, args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1.py': b'"""Compare two eligible starting models per domain with the same lighting treatment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport os\nimport shutil\nimport time\nimport uuid\nimport zipfile\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import camera_extended, domain_alignment\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as exp\nfrom mediflow_datasets import domain_adaptation_resolution as resolution\nfrom mediflow_datasets.candidate_reproduction import ROOT\nfrom mediflow_datasets.domain_adaptation_io import write_progress\nfrom mediflow_datasets.domain_adaptation_training import object_hash\n\nDOMAINS = ("skin", "web_skin", "hair")\nARMS = ("basic", "selected")\nWORKERS = min(4, os.cpu_count() or 1)\n\n\ndef show(label, step, total, extra=""):\n    print(f"{label} | {step}/{total} ({100 * step / total:.1f}%) {extra}", flush=True)\n\n\ndef prepare_source(domain, archive_path, task, reference, out):\n    classes = task["classes"]\n    print(f"[{domain}] \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP \xed\x95\xb4\xec\x8b\x9c\xc2\xb7\xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4 \xed\x99\x95\xec\x9d\xb8", flush=True)\n    source, sha = data.source_index(archive_path, classes, task["source_sha256"])\n    retained = (\n        resolution.filtered_source(source, reference["quarantine"], classes, sha)\n        if reference.get("quarantine")\n        else {k: v for k, v in source.items() if k[0] == "original"}\n    )\n    # Skin\'s historical distinction is Original versus Augmented preparation. Use only\n    # linked descendants of retained originals. Adaptation itself uses identical originals.\n    train = retained["original", "train"]\n    originals = {r.path: r for r in train}\n    prepare_augmented = domain == "skin" and any(\n        spec.get("skin_augmented_preparation") for spec in task.get("models", {}).values()\n    )\n    descendants = [\n        r\n        for r in source["augmented", "train"]\n        if prepare_augmented\n        and r.origin in originals\n        and r.label == originals[r.origin].label\n        and r.path.rsplit("/", 1)[-1] != r.origin.rsplit("/", 1)[-1]\n    ]\n    # For the saved Hair model, include its previous augmented source in the leakage scan.\n    scanned = (\n        [r for rows in source.values() for r in rows]\n        if domain == "hair"\n        else [r for rows in retained.values() for r in rows] + descendants\n    )\n    label = f"[{domain}] \xec\xa4\x91\xeb\xb3\xb5\xc2\xb7\xeb\xb6\x84\xed\x95\xa0 \xed\x99\x95\xec\x9d\xb8 (\xec\x82\xac\xec\xa7\x84)"\n    entries = []\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def signature(row):\n            return row, *data.image_signatures(archive.read(row.path))\n\n        for start in range(0, len(scanned), 128):\n            entries.extend(pool.map(signature, scanned[start : start + 128]))\n            done = len(entries)\n            if start == 0 or done % 1024 == 0 or done == len(scanned):\n                show(label, done, len(scanned))\n    report = data.audit_groups(entries)\n    dropped = set()\n    if descendants:\n        involved = {p for pair in report["near_duplicate_review"] for p in pair}\n        involved.update(\n            p for leak in report["blocking_leaks_or_conflicts"] for p in leak.get("paths", ())\n        )\n        dropped = {r.path for r in descendants if r.path in involved}\n        entries = [entry for entry in entries if entry[0].path not in dropped]\n        report = data.audit_groups(entries)\n    report["excluded_augmented_descendants"] = sorted(dropped)\n    report["source_target_pixel_overlap"] = len(\n        {entry[2] for entry in entries} & set(reference["pixel_hashes"])\n    )\n    report["source_sha256"] = sha\n    report["retained_counts"] = {split: len(retained["original", split]) for split in data.SPLITS}\n    if domain == "hair":\n        consistency = {}\n        for split in data.SPLITS:\n            original, augmented = (\n                Counter(\n                    (row.label, pixel)\n                    for row, _, pixel, _ in entries\n                    if row.split == split and row.variant == variant\n                )\n                for variant in ("original", "augmented")\n            )\n            consistency[split] = (\n                not (original - augmented) if split == "train" else original == augmented\n            )\n        report["original_augmented_consistency"] = consistency\n    write_progress(out / "data_audit.json", report)\n    if (\n        report["blocking_leaks_or_conflicts"]\n        or report["near_duplicate_review"]\n        or report["source_target_pixel_overlap"]\n        or not all(report.get("original_augmented_consistency", {}).values())\n    ):\n        raise ValueError(f"{domain} \xeb\xb6\x84\xed\x95\xa0 \xec\xa0\x90\xea\xb2\x80 \xec\x8b\xa4\xed\x8c\xa8. data_audit.json\xec\x9d\x84 \xed\x99\x95\xec\x9d\xb8\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    pixel_by_path = {row.path: pixel for row, _, pixel, _ in entries}\n    paired = {}\n    for row in descendants:\n        if row.path not in dropped and pixel_by_path[row.path] != pixel_by_path[row.origin]:\n            paired.setdefault(row.origin, row)\n    if prepare_augmented and {r.label for r in paired.values()} != set(range(len(classes))):\n        raise ValueError("\xed\x94\xbc\xeb\xb6\x80 \xec\x84\xa0\xec\xa0\x95 \xeb\xb0\xa9\xec\x8b\x9d \xec\x9e\xac\xed\x98\x84\xec\x97\x90 \xed\x95\x84\xec\x9a\x94\xed\x95\x9c \xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4\xeb\xb3\x84 \xed\x8c\x8c\xec\x83\x9d\xeb\xb3\xb8\xec\x9d\xb4 \xed\x99\x95\xec\x9d\xb8\xeb\x90\x98\xec\xa7\x80 \xec\x95\x8a\xec\x95\x98\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(\n        out / "source_manifest.json",\n        {\n            "original": {\n                split: [asdict(r) for r in retained["original", split]] for split in data.SPLITS\n            },\n            "skin_preparation_pairs": {parent: asdict(row) for parent, row in paired.items()},\n        },\n    )\n    return train, retained["original", "val"], paired\n\n\ndef image_cache(folder, archive_path, train, val, pairs, task, size, profile, *, expanded=None):\n    """Cache exact float32 model inputs on local SSD once for each input resolution."""\n    rows = [*train, *val, *pairs.values()]\n    conditions = ("lighting", "expanded") if expanded is not None else ("lighting",)\n    size_bytes = (len(rows) + len(conditions) * len(val)) * size * size * 3 * 4\n    if shutil.disk_usage(folder).free < size_bytes + 1024**3:\n        raise RuntimeError(\n            f"{task[\'domain\']} \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xea\xb3\xb5\xea\xb0\x84 \xeb\xb6\x80\xec\xa1\xb1 (\xed\x95\x84\xec\x9a\x94 {size_bytes / 1e9:.1f} GB)."\n        )\n    shape = (size, size, 3)\n    cache = np.lib.format.open_memmap(\n        folder / f"inputs_{size}.npy", mode="w+", dtype=np.float32, shape=(len(rows), *shape)\n    )\n    shifted = {\n        name: np.lib.format.open_memmap(\n            folder / f"val_{name}_{size}.npy", mode="w+", dtype=np.float32, shape=(len(val), *shape)\n        )\n        for name in conditions\n    }\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def prepare(item):\n            i, row = item\n            with tf.device("/CPU:0"):\n                image = exp.image_tensor(archive.read(row.path), size, task["input_policy"])\n            cache[i] = image\n            if row.split == "val":\n                seed = exp.item_seed(row, 1707)\n                shifted["lighting"][i - len(train)] = exp.transform_image(\n                    image, profile, "target_guided", seed\n                )\n                if expanded is not None:\n                    shifted["expanded"][i - len(train)] = camera_extended.transform(\n                        image, expanded, "expanded", seed\n                    )\n\n        for start in range(0, len(rows), 128):\n            list(pool.map(prepare, enumerate(rows[start : start + 128], start)))\n            end = min(start + 128, len(rows))\n            if start == 0 or end % 512 == 0 or end == len(rows):\n                show(f"[{task[\'domain\']} \xc2\xb7 {size}px] \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xec\xa4\x80\xeb\xb9\x84 (\xec\x82\xac\xec\xa7\x84)", end, len(rows))\n    cache.flush()\n    for images in shifted.values():\n        images.flush()\n    lookup = {row.path: i for i, row in enumerate(rows)}\n    paired_indexes = [\n        lookup[pairs[r.path].path] if r.path in pairs else i for i, r in enumerate(train)\n    ]\n    return cache, shifted if expanded is not None else shifted["lighting"], paired_indexes\n\n\ndef pipeline(images, indexes, labels, batch, size, transform=None):\n    def read(index):\n        image = images[int(index)]\n        return image if transform is None else transform(int(index), image)\n\n    def get(index, label):\n        image = tf.numpy_function(read, [index], tf.float32)\n        return tf.ensure_shape(image, (size, size, 3)), label\n\n    ds = tf.data.Dataset.from_tensor_slices(\n        (np.asarray(indexes, np.int32), np.asarray(labels, np.int32))\n    )\n    options = tf.data.Options()\n    options.threading.private_threadpool_size = WORKERS\n    options.threading.max_intra_op_parallelism = 1\n    return ds.with_options(options).map(get, num_parallel_calls=WORKERS).batch(batch).prefetch(2)\n\n\ndef model_for(spec, count, weights):\n    if spec["architecture"] == "pmg":\n        from mediflow_datasets.web_skin_paper_suite import build_pmg_model\n\n        return build_pmg_model(count, spec["size"], weights=weights)\n    size = spec["size"]\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = keras.layers.GlobalAveragePooling2D()(backbone(inputs, training=False))\n    outputs = keras.layers.Dense(count, activation="softmax")(keras.layers.Dropout(0.3)(features))\n    return keras.Model(inputs, outputs)\n\n\ndef predictor(model, spec):\n    pmg = spec["architecture"] == "pmg"\n    size = spec["size"]\n\n    @tf.function(input_signature=[tf.TensorSpec([None, size, size, 3], tf.float32)])\n    def predict(images):\n        outputs = model(images, training=False)\n        scores = tf.nn.softmax(tf.add_n(outputs), axis=-1) if pmg else outputs\n        return tf.debugging.check_numerics(scores, "validation scores")\n\n    return predict\n\n\ndef steps(model, spec, optimizer, count, *, alignment=None):\n    pmg = spec["architecture"] == "pmg"\n    loss_fn = keras.losses.CategoricalCrossentropy(\n        from_logits=pmg, label_smoothing=spec["label_smoothing"]\n    )\n    variables = model.trainable_variables\n    optimizer.build(variables)\n    if pmg:\n        from mediflow_datasets.web_skin_paper_suite import _jigsaw\n    if alignment is not None:\n        adapter = "sum_logits_softmax" if pmg else "single_softmax"\n        extractor = exp.feature_model(model, adapter)\n        features = keras.Model(model.input, extractor.outputs[-1])\n\n    def update(images, labels, branch=None, factor=1.0, target=None):\n        with tf.GradientTape() as tape:\n            if target is None:\n                outputs = model(images, training=True)\n                logits = outputs[branch] if pmg else outputs\n                mmd = coral = tf.constant(0.0)\n            else:\n                outputs = extractor(images, training=True)\n                logits = outputs[branch if pmg else 0]\n                mmd, coral = domain_alignment.losses(\n                    outputs[-1], features(target, training=False), alignment["reference"]\n                )\n            ce = factor * loss_fn(labels, logits)\n            loss = ce\n            if target is not None:\n                loss = (\n                    loss + alignment["weights"]["mmd"] * mmd + alignment["weights"]["coral"] * coral\n                )\n            loss = tf.debugging.check_numerics(loss, "training loss")\n        grads = tape.gradient(loss, variables)\n        pairs = [\n            (grad, var) for grad, var in zip(grads, variables, strict=True) if grad is not None\n        ]\n        clipped, norm = tf.clip_by_global_norm([g for g, _ in pairs], 1.0)\n        with tf.control_dependencies([tf.debugging.check_numerics(norm, "gradient norm")]):\n            optimizer.apply_gradients(zip(clipped, [v for _, v in pairs], strict=True))\n        return loss, ce, mmd, coral\n\n    signature = [\n        tf.TensorSpec([None, spec["size"], spec["size"], 3], tf.float32),\n        tf.TensorSpec([None], tf.int32),\n    ]\n\n    def update_batch(images, indices, target=None):\n        labels = tf.one_hot(indices, count)\n        if pmg:\n            branch_losses = [\n                update(_jigsaw(images, grid), labels, i)[0] for i, grid in enumerate((8, 4, 2))\n            ]\n            loss, ce, mmd, coral = update(images, labels, 3, 2.0, target)\n            total = tf.add_n(branch_losses)\n            return (total + loss) / 4, (total + ce) / 4, mmd, coral\n        return update(images, labels, target=target)\n\n    @tf.function(input_signature=signature)\n    def train_step(images, indices):\n        return update_batch(images, indices)[0]\n\n    if alignment is not None:\n        aligned_step = tf.function(update_batch, input_signature=[*signature, signature[0]])\n        return train_step, predictor(model, spec), aligned_step\n\n    return train_step, predictor(model, spec)\n\n\ndef epoch_records(folder, epochs):\n    records = sorted(folder.glob("epoch_*.json"))\n    history = [data.read_json(p) for p in records]\n    if len(history) > epochs or [r["epoch"] for r in history] != list(range(1, len(history) + 1)):\n        raise ValueError(f"Incomplete or inconsistent epoch records: {folder}")\n    return history\n\n\ndef save_epoch(checkpoint, folder, local, entry):\n    attempt = "snapshot_" + uuid.uuid4().hex\n    destination = folder / attempt\n    destination.mkdir()\n    local.mkdir(exist_ok=True)\n    local_prefix = checkpoint.write(str(local / "state"))\n    record = {**entry, "snapshot": attempt, "files": {}}\n    for path in sorted(Path(local_prefix).parent.glob("state.*")):\n        shutil.copyfile(path, destination / path.name)\n        record["files"][path.name] = data.sha256(path)\n    if set(record["files"]) != {"state.index", "state.data-00000-of-00001"}:\n        raise RuntimeError("Unexpected checkpoint shard layout")\n    write_progress(folder / f"epoch_{entry[\'epoch\']:03}.json", record)\n    return record\n\n\ndef restore_epoch(checkpoint, folder, record):\n    for name, sha in record["files"].items():\n        if data.sha256(data.local_path(folder, f"{record[\'snapshot\']}/{name}")) != sha:\n            raise ValueError("Epoch checkpoint failed integrity verification")\n    checkpoint.read(str(folder / record["snapshot"] / "state")).assert_consumed()\n\n\ndef publish_model(source, directory, name):\n    temporary = directory / (uuid.uuid4().hex + ".keras.tmp")\n    shutil.copyfile(source, temporary)\n    destination = directory / f"{name}.keras"\n    temporary.replace(destination)\n    return {"file": destination.name, "sha256": data.sha256(source)}\n\n\ndef export_model(model, directory, name, local):\n    local.mkdir(exist_ok=True)\n    scratch = local / f"{name}.keras"\n    model.save(scratch)\n    return publish_model(scratch, directory, name)\n\n\ndef validation_pipelines(cache, shifted, n, val, batch, size):\n    labels = [row.label for row in val]\n    return {\n        "clean": pipeline(cache, range(n, n + len(val)), labels, batch, size),\n        **{\n            name: pipeline(images, range(len(val)), labels, batch, size)\n            for name, images in shifted.items()\n        },\n    }\n\n\ndef validate(predict, datasets, rows, class_count, label):\n    metrics = {}\n    for name, ds in datasets.items():\n        print(f"{label} Validation {name} \xc2\xb7 {len(rows):,}\xec\x9e\xa5", flush=True)\n        scores = np.concatenate([predict(images).numpy() for images, _ in ds])\n        metrics[name] = exp.classification(rows, scores, class_count)\n    return metrics\n\n\ndef phase(model, context, folder, local, spec, phase_name, epochs, rate, *, alignment=None):\n    config, task, train, val, profile, cache, shifted, pairs = context\n    folder.mkdir(exist_ok=True)\n    history = epoch_records(folder, epochs)\n    optimizer = keras.optimizers.Adam(rate)\n    step, predict, *extra_steps = steps(\n        model, spec, optimizer, len(task["classes"]), alignment=alignment\n    )\n    if (phase_name == "aligned") != (alignment is not None):\n        raise ValueError("Alignment inputs must be present exactly for the aligned phase")\n    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)\n    tag = f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']} \xc2\xb7 {phase_name}]"\n    if history:\n        restore_epoch(checkpoint, folder, history[-1])\n        print(f"{tag} Drive\xec\x97\x90\xec\x84\x9c {len(history)}/{epochs} epoch \xeb\xb3\xb5\xec\x9b\x90", flush=True)\n    n, batch, size = len(train), task["batch_size"], spec["size"]\n    expanded = phase_name in ("expanded", "aligned")\n    condition = "expanded" if expanded else "lighting"\n    datasets = validation_pipelines(\n        cache, shifted if expanded else {"lighting": shifted}, n, val, batch, size\n    )\n    for epoch in range(len(history), epochs):\n        started = time.monotonic()\n        keras.utils.set_random_seed(config["seed"] + epoch)\n        order = np.random.default_rng(config["seed"] + epoch).permutation(n)\n        indexes = order\n        if phase_name.startswith("prepare") and spec.get("skin_augmented_preparation"):\n            indexes = [pairs[i] if (i + epoch) % 2 == 0 else i for i in order]\n        transform = None\n        if expanded or phase_name == "lighting":\n            seeds = [exp.item_seed(row, config["seed"], epoch) for row in train]\n            apply = [exp.augmentation_applies(s, config["augmentation_probability"]) for s in seeds]\n\n            def transform(i, image, seeds=seeds, apply=apply):\n                if not apply[i]:\n                    return image\n                if expanded:\n                    return camera_extended.transform(image, profile, "expanded", seeds[i])\n                return exp.transform_image(image, profile, "target_guided", seeds[i])\n\n        label = f"{tag} Epoch {epoch + 1}/{epochs}"\n        batches = math.ceil(n / batch)\n        print(f"{label} \xec\x8b\x9c\xec\x9e\x91 \xc2\xb7 \xed\x95\x99\xec\x8a\xb5 {n:,}\xec\x9e\xa5 \xc2\xb7 {batches} \xeb\xb0\xb0\xec\xb9\x98", flush=True)\n        train_data = pipeline(\n            cache, indexes, [train[i].label for i in order], batch, size, transform\n        )\n        losses = []\n        aligned_metrics = []\n        schedule = {}\n        if alignment is not None:\n            eligible = range(batches - int(n % batch == 1))\n            schedule = domain_alignment.balanced_schedule(\n                alignment["rows"], eligible, config["seed"], epoch\n            )\n            if not schedule:\n                raise ValueError("No Source batch is eligible for feature alignment")\n        for i, (images, labels) in enumerate(train_data, 1):\n            if i - 1 in schedule:\n                target = alignment["images"][schedule[i - 1]]\n                loss, ce, mmd, coral = map(float, extra_steps[0](images, labels, target))\n                losses.append(loss)\n                aligned_metrics.append((ce, mmd, coral))\n            else:\n                losses.append(float(step(images, labels)))\n            if i == 1 or i == batches or i % max(5, batches // 15) == 0:\n                elapsed = time.monotonic() - started\n                remaining = elapsed * (batches / i - 1)\n                show(\n                    label + " \xed\x95\x99\xec\x8a\xb5",\n                    i,\n                    batches,\n                    f"loss={np.mean(losses):.5f} \xc2\xb7 \xeb\x82\xa8\xec\x9d\x80 \xed\x95\x99\xec\x8a\xb5 \xec\x95\xbd {remaining:.0f}\xec\xb4\x88",\n                )\n        train_seconds = time.monotonic() - started\n        metrics = validate(predict, datasets, val, len(task["classes"]), label)\n        validation_seconds = time.monotonic() - started - train_seconds\n        entry = {\n            "epoch": epoch + 1,\n            "train_loss_mean_per_update": float(np.mean(losses)),\n            "training_batches": batches,\n            "optimizer_updates": batches * (4 if spec["architecture"] == "pmg" else 1),\n            "train_seconds": train_seconds,\n            "validation_seconds": validation_seconds,\n            **metrics,\n        }\n        if aligned_metrics:\n            ce, mmd, coral = np.asarray(aligned_metrics).mean(0).tolist()\n            entry["alignment"] = {\n                "updates": len(aligned_metrics),\n                "target_images_used": sum(map(len, schedule.values())),\n                "classification_loss_on_aligned_batches": ce,\n                "mmd_mean": mmd,\n                "coral_mean": coral,\n            }\n            print(\n                f"{label} \xec\xa0\x95\xeb\xa0\xac {len(aligned_metrics)}\xeb\xb0\xb0\xec\xb9\x98 \xc2\xb7 MMD={mmd:.6g} \xc2\xb7 CORAL={coral:.6g}",\n                flush=True,\n            )\n        print(f"{label} Drive \xec\xb2\xb4\xed\x81\xac\xed\x8f\xac\xec\x9d\xb8\xed\x8a\xb8 \xec\xa0\x80\xec\x9e\xa5", flush=True)\n        history.append(save_epoch(checkpoint, folder, local, entry))\n        print(\n            f"{label} \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 loss={entry[\'train_loss_mean_per_update\']:.5f} \xc2\xb7 "\n            f"\xec\x9b\x90\xeb\xb3\xb8 Val F1={metrics[\'clean\'][\'macro_f1\']:.5f} \xc2\xb7 "\n            f"\xec\xa1\xb0\xeb\xaa\x85 Val F1={metrics[\'lighting\'][\'macro_f1\']:.5f}"\n            + (f" \xc2\xb7 \xed\x99\x95\xec\x9e\xa5 Val F1={metrics[\'expanded\'][\'macro_f1\']:.5f}" if expanded else ""),\n            flush=True,\n        )\n    best = max(history, key=lambda r: score(r, expanded or phase_name == "lighting", condition))\n    restore_epoch(checkpoint, folder, best)\n    write_progress(folder / "history.json", history)\n    return best\n\n\ndef score(record, lighting=True, condition="lighting"):\n    clean, shifted = record["clean"]["macro_f1"], record[condition]["macro_f1"]\n    return (min(clean, shifted), clean) if lighting else (clean,)\n\n\ndef trial(context, output, local, spec, job):\n    config, task = context[:2]\n    output.mkdir(exist_ok=True)\n    completed = output / "complete.json"\n    if completed.exists():\n        saved = data.read_json(completed)\n        verify_models(output, saved)\n        print(f"[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        return saved\n    print(f"\\n[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x8b\xa4\xed\x97\x98 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(config["seed"])\n    count = len(task["classes"])\n    initial_path = output / "initial.json"\n    if initial_path.exists():\n        initial = data.read_json(initial_path)\n        path = data.local_path(output, initial["file"])\n        if data.sha256(path) != initial["sha256"]:\n            raise ValueError("Starting checkpoint changed")\n        model = keras.models.load_model(path, compile=False)\n    else:\n        if spec.get("model"):\n            path = data.local_path(ROOT, spec["model"])\n            if data.sha256(path) != spec["model_sha256"]:\n                raise ValueError("Selected checkpoint changed")\n            model = keras.models.load_model(path, compile=False)\n            initial = publish_model(path, output, "initial")\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\xa0\x80\xec\x9e\xa5 \xea\xb0\x80\xec\xa4\x91\xec\xb9\x98\xeb\xa1\x9c \xec\xa1\xb0\xeb\xaa\x85 \xec\xa0\x81\xec\x9d\x91 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n        else:\n            weights = None if (output / "prepare_head/epoch_001.json").exists() else "imagenet"\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xed\x98\x84\xec\x9e\xac Train\xec\x9c\xbc\xeb\xa1\x9c \xec\xb6\x9c\xeb\xb0\x9c \xeb\xaa\xa8\xeb\x8d\xb8 \xec\xa4\x80\xeb\xb9\x84", flush=True)\n            model = model_for(spec, count, weights)\n            phase(\n                model,\n                context,\n                output / "prepare_head",\n                local,\n                spec,\n                "prepare_head",\n                config["prepare_head_epochs"],\n                config["prepare_head_lr"],\n            )\n            exp.configure_partial(model)\n            phase(\n                model,\n                context,\n                output / "prepare_partial",\n                local,\n                spec,\n                "prepare_partial",\n                config["prepare_partial_epochs"],\n                config["prepare_partial_lr"],\n            )\n            initial = export_model(model, output, "initial", local)\n        write_progress(initial_path, initial)\n    if tuple(model.input_shape[1:]) != (spec["size"], spec["size"], 3):\n        raise ValueError("Initial input shape differs from configuration")\n    if len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1) or any(\n        layer.shape[-1] != count for layer in model.outputs\n    ):\n        raise ValueError("Initial output contract differs from class mapping")\n    exp.configure_partial(model)\n    best = phase(\n        model,\n        context,\n        output / "lighting",\n        local,\n        spec,\n        "lighting",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved = {\n        "initial": initial,\n        "best_model": export_model(model, output, "best", local),\n        "model_config": spec,\n        "validation": best,\n        "source_test_evaluated": False,\n        "validation_scope": config.get("validation_scope", "current_source_validation"),\n    }\n    write_progress(completed, saved)\n    return saved\n\n\ndef verify_models(folder, result):\n    for key in ("initial", "best_model"):\n        item = result[key]\n        if data.sha256(data.local_path(folder, item["file"])) != item["sha256"]:\n            raise ValueError(f"Completed checkpoint changed: {folder}/{item[\'file\']}")\n\n\ndef run_domain(domain, source_zip, output, local):\n    config = data.read_json(ROOT / "configs/adaptation_stage1.json")\n    task = config["tasks"][domain]\n    if config.get("initialization_policy") == "saved_only" and any(\n        not spec.get("model") or not spec.get("model_sha256") for spec in task["models"].values()\n    ):\n        raise ValueError("A starting checkpoint is missing from the saved-only bundle")\n    if config.get("validation_scope"):\n        print(\n            f"[{domain}] \xea\xb3\xbc\xea\xb1\xb0 \xed\x95\x99\xec\x8a\xb5 \xec\x9d\xb4\xeb\xa0\xa5\xec\x9d\x98 \xea\xb2\xb9\xec\xb9\xa8 \xec\x97\xac\xeb\xb6\x80 \xeb\xaf\xb8\xed\x99\x95\xec\x9d\xb8 \xc2\xb7 Validation\xec\x9d\x80 \xed\x83\x90\xec\x83\x89 \xeb\xb9\x84\xea\xb5\x90\xec\x9a\xa9", flush=True\n        )\n    reference = data.read_json(ROOT / f"references/{domain}.json")\n    profile = reference["extended_profile"]["lighting"]\n    output.mkdir(parents=True, exist_ok=True)\n    identity = {\n        "config_sha256": object_hash(config),\n        "reference_sha256": object_hash(reference),\n        "code_sha256": data.sha256(Path(__file__)),\n    }\n    marker = output / "identity.json"\n    if marker.exists() and data.read_json(marker) != identity:\n        raise ValueError("Different inputs cannot resume this output")\n    write_progress(marker, identity)\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "identity": identity,\n            "configuration": config,\n            "profile": profile,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": np.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(device)\n                for device in tf.config.list_physical_devices("GPU")\n            ],\n            "code_provenance": data.read_json(ROOT / "bundle_provenance.json"),\n        },\n    )\n    if all((output / role / "complete.json").is_file() for role in ARMS):\n        results = {role: data.read_json(output / role / "complete.json") for role in ARMS}\n        for role, result in results.items():\n            verify_models(output / role, result)\n        print(f"[{domain}] \xeb\x91\x90 \xed\x95\x99\xec\x8a\xb5 \xeb\xaa\xa8\xeb\x91\x90 \xec\x99\x84\xeb\xa3\x8c\xeb\x90\xa8", flush=True)\n        return results\n    train, val, pairs = prepare_source(domain, source_zip, task, reference, output)\n    caches = {}\n    results = {}\n    for i, role in enumerate(ARMS, 1):\n        existing = output / role / "complete.json"\n        if existing.is_file():\n            results[role] = data.read_json(existing)\n            verify_models(output / role, results[role])\n            print(\n                f"[{DOMAINS.index(domain) * 2 + i}/6 \xc2\xb7 {domain} \xc2\xb7 {role}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9",\n                flush=True,\n            )\n            continue\n        spec = {**task["models"][role], "role": role}\n        size = spec["size"]\n        if size not in caches:\n            caches[size] = image_cache(local, source_zip, train, val, pairs, task, size, profile)\n        context = (config, task, train, val, profile, *caches[size])\n        results[role] = trial(\n            context, output / role, local / "saving", spec, DOMAINS.index(domain) * 2 + i\n        )\n    return results\n\n\ndef write_selection(root):\n    selected, table = {}, []\n    for domain in DOMAINS:\n        records = {role: data.read_json(root / domain / role / "complete.json") for role in ARMS}\n        for role, record in records.items():\n            verify_models(root / domain / role, record)\n            table.append(\n                {\n                    "domain": domain,\n                    "role": role,\n                    "best_epoch": record["validation"]["epoch"],\n                    "clean_macro_f1": record["validation"]["clean"]["macro_f1"],\n                    "lighting_macro_f1": record["validation"]["lighting"]["macro_f1"],\n                    "clean_accuracy": record["validation"]["clean"]["accuracy"],\n                    "lighting_accuracy": record["validation"]["lighting"]["accuracy"],\n                    "validation_scope": record.get("validation_scope", "current_source_validation"),\n                }\n            )\n        role = max(ARMS, key=lambda name: score(records[name]["validation"]))\n        record = records[role]\n        selected[domain] = {\n            "role": role,\n            "model_config": record["model_config"],\n            "initial_checkpoint": {\n                "path": f"{domain}/{role}/initial.keras",\n                "sha256": record["initial"]["sha256"],\n            },\n            "stage1_checkpoint": {\n                "path": f"{domain}/{role}/best.keras",\n                "sha256": record["best_model"]["sha256"],\n            },\n        }\n    selection = {\n        "schema": "mediflow_adaptation_stage1_selection_v1",\n        "selected": selected,\n        "criteria": "Maximize worst Val Macro-F1. Break ties by clean Val, then basic.",\n        "source_test_evaluated": False,\n        "target_accuracy": None,\n    }\n    if any(row["validation_scope"].startswith("exploratory_") for row in table):\n        selection["selection_scope"] = "provisional_followup_candidate"\n        selection["historical_train_evaluation_independence_verified"] = False\n    path = root / "selection.json"\n    if path.exists() and data.read_json(path) != selection:\n        raise ValueError("Existing Validation selection differs")\n    with (root / "validation_comparison.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=table[0].keys())\n        writer.writeheader()\n        writer.writerows(table)\n    write_progress(path, selection)\n    for row in table:\n        print(row, flush=True)\n    print("1\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 \xeb\xb6\x84\xec\x95\xbc\xeb\xb3\x84 \xec\x84\xa0\xed\x83\x9d:", {d: s["role"] for d, s in selected.items()}, flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--select", type=Path)\n    parser.add_argument("--domain", choices=DOMAINS)\n    parser.add_argument("--source-zip", type=Path)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--local", type=Path)\n    args = parser.parse_args()\n    if args.select:\n        write_selection(args.select)\n    elif all((args.domain, args.source_zip, args.output, args.local)):\n        if not tf.config.list_physical_devices("GPU"):\n            raise RuntimeError("TensorFlow GPU\xeb\xa5\xbc \xec\x82\xac\xec\x9a\xa9\xed\x95\xa0 \xec\x88\x98 \xec\x97\x86\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        run_domain(args.domain, args.source_zip, args.output, args.local)\n    else:\n        parser.error("Provide --select or all domain run parameters")\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1_colab.py': b'"""Drive-based stage 1 launcher with live output and scoped interruption."""\n\nfrom __future__ import annotations\n\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport uuid\nfrom pathlib import Path\n\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nDOMAINS = ("skin", "web_skin", "hair")\n\n\ndef run_live(command, env, log_path, *, input_text=None):\n    with log_path.open("x", encoding="utf-8") as log:\n        process = subprocess.Popen(\n            command,\n            env=env,\n            stdin=subprocess.PIPE if input_text is not None else None,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            encoding="utf-8",\n            errors="replace",\n            bufsize=1,\n        )\n        try:\n            if input_text is not None:\n                with process.stdin:\n                    process.stdin.write(input_text)\n            for line in process.stdout:\n                log.write(line)\n                log.flush()\n                print(line, end="", flush=True)\n            if process.wait():\n                raise subprocess.CalledProcessError(process.returncode, command)\n        finally:\n            process.stdout.close()\n            if process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait()\n\n\ndef copy_source(source, destination):\n    total = source.stat().st_size\n    if shutil.disk_usage(destination.parent).free < total + 1024**3:\n        raise RuntimeError("Colab \xec\x9e\x84\xec\x8b\x9c \xeb\x94\x94\xec\x8a\xa4\xed\x81\xac\xec\x97\x90 \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP\xec\x9d\x84 \xeb\xb3\xb5\xec\x82\xac\xed\x95\xa0 \xea\xb3\xb5\xea\xb0\x84\xec\x9d\xb4 \xeb\xb6\x80\xec\xa1\xb1\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4.")\n    with source.open("rb") as reader, destination.open("xb") as writer:\n        done, last = 0, -1\n        while block := reader.read(16 * 1024**2):\n            writer.write(block)\n            done += len(block)\n            progress = min(100, 100 * done // total)\n            if progress >= last + 5 or done == total:\n                print(f"  Drive \xe2\x86\x92 Colab \xeb\xa1\x9c\xec\xbb\xac \xeb\xb3\xb5\xec\x82\xac {progress}% ({done / 1e9:.2f} GB)", flush=True)\n                last = progress\n    if done != total:\n        raise OSError("Drive ZIP \xeb\xb3\xb5\xec\x82\xac\xea\xb0\x80 \xec\xa4\x91\xeb\x8b\xa8\xeb\x90\x90\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n\n\ndef run(runtime, work, project, bundle_sha, my_drive):\n    config = json.loads((runtime / "configs/adaptation_stage1.json").read_text(encoding="utf-8"))\n    python, versions = batch.prepare_python(work)\n    if not versions["gpu"]:\n        raise RuntimeError("Colab \xeb\x9f\xb0\xed\x83\x80\xec\x9e\x84 \xec\x9c\xa0\xed\x98\x95\xec\x97\x90\xec\x84\x9c GPU\xeb\xa5\xbc \xec\x84\xa0\xed\x83\x9d\xed\x95\x9c \xeb\x92\xa4 \xeb\x8b\xa4\xec\x8b\x9c \xec\x8b\xa4\xed\x96\x89\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    env = batch.python_environment(python, runtime)\n    env.update(PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n    destination = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16] / "stage1"\n    destination.mkdir(parents=True, exist_ok=True)\n    marker = destination / "bundle.json"\n    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8"))["sha256"] != bundle_sha:\n        raise RuntimeError("\xec\x9d\xb4 \xea\xb2\xb0\xea\xb3\xbc \xed\x8f\xb4\xeb\x8d\x94\xeb\x8a\x94 \xeb\x8b\xa4\xeb\xa5\xb8 \xeb\xb2\x84\xec\xa0\x84\xec\x9d\x98 \xea\xb3\xb5\xed\x86\xb5 ZIP\xec\x97\x90 \xec\x97\xb0\xea\xb2\xb0\xeb\x8f\xbc \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(marker, {"sha256": bundle_sha, "stage": 1, "environment": versions})\n    print("1\xec\xb0\xa8 \xea\xb2\xb0\xea\xb3\xbc \xec\xa0\x80\xec\x9e\xa5:", destination, flush=True)\n    failures = {}\n    for domain in DOMAINS:\n        output = destination / domain\n        log = destination / f"{domain}_{uuid.uuid4().hex[:10]}.log"\n        command = [str(python), "-m", "mediflow_datasets.adaptation_stage1"]\n        try:\n            with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as folder:\n                local = Path(folder)\n                data_zip = local / "source.zip"\n                if not all(\n                    (output / arm / "complete.json").is_file() for arm in ("basic", "selected")\n                ):\n                    print(f"\\n[{domain}] Drive \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0\xec\x85\x8b \xec\xa4\x80\xeb\xb9\x84", flush=True)\n                    source = batch.find_source(project / "datasets", config["tasks"][domain])\n                    copy_source(source, data_zip)\n                run_live(\n                    command\n                    + [\n                        "--domain",\n                        domain,\n                        "--source-zip",\n                        str(data_zip),\n                        "--output",\n                        str(output),\n                        "--local",\n                        str(local),\n                    ],\n                    env,\n                    log,\n                )\n        except Exception as error:\n            failures[domain] = {"error": str(error), "log": log.name if log.is_file() else None}\n            print(f"[{domain}] \xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c: {error}", flush=True)\n    completed = [\n        f"{d}/{a}"\n        for d in DOMAINS\n        for a in ("basic", "selected")\n        if (destination / d / a / "complete.json").is_file() and d not in failures\n    ]\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": False},\n    )\n    if failures:\n        raise RuntimeError(\n            f"\xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc\xea\xb0\x80 \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4. \xec\x99\x84\xeb\xa3\x8c\xeb\x90\x9c \xea\xb8\xb0\xeb\xa1\x9d\xec\x9d\x80 Drive\xec\x97\x90 \xeb\xb3\xb4\xec\xa1\xb4\xed\x96\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4: {destination}"\n        )\n    run_live(\n        [str(python), "-m", "mediflow_datasets.adaptation_stage1", "--select", str(destination)],\n        env,\n        destination / f"selection_{uuid.uuid4().hex[:10]}.log",\n    )\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": True},\n    )\n    print("\xeb\xb9\x84\xea\xb5\x90\xed\x91\x9c:", destination / "validation_comparison.csv", flush=True)\n    return destination\n\n\ndef launch(runtime, work, project, bundle_sha, my_drive):\n    import fcntl\n\n    # Keep two cells in the same Colab VM from touching the same run concurrently.\n    lock = Path("/content") / f"mediflow_stage1_{bundle_sha}.lock"\n    with lock.open("a") as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as error:\n            raise RuntimeError("\xea\xb0\x99\xec\x9d\x80 Colab \xec\x84\xb8\xec\x85\x98\xec\x97\x90\xec\x84\x9c \xec\x9d\xb4\xeb\xaf\xb8 \xec\x8b\xa4\xed\x96\x89 \xec\xa4\x91\xec\x9e\x85\xeb\x8b\x88\xeb\x8b\xa4.") from error\n        return run(runtime, work, project, bundle_sha, my_drive)\n\n\nif __name__ == "__main__":\n    launch(*(Path(arg) if index != 3 else arg for index, arg in enumerate(sys.argv[1:])))\n'}.items():
        (runtime / "src/mediflow_datasets" / name).write_bytes(contents)
    (runtime / "src/mediflow_datasets/adaptation_stage4_test.py").write_bytes(b'"""Evaluate Stage 4 only; reuse verified historical Test metrics without new inference."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport io\nimport platform\nimport tempfile\nimport zipfile\nfrom contextlib import nullcontext\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport cv2\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import adaptation_final_test as original\nfrom mediflow_datasets import adaptation_stage1 as training\nfrom mediflow_datasets import camera_extended as extended\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as experiment\nfrom mediflow_datasets.adaptation_stage1_colab import copy_source\nfrom mediflow_datasets.domain_adaptation_batch import digest, find_source\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nCONDITIONS = ("clean", "lighting", "expanded", "stage4_lighting", "stage4_expanded")\nMODEL_LABELS = {**original.MODEL_LABELS, "stage4_revised": "Stage 4 (before refit)"}\nCONDITION_LABELS = (\n    "Original",\n    "Old lighting",\n    "Old expanded",\n    "Stage 4 lighting",\n    "Stage 4 expanded",\n)\n\n\ndef evaluate_zip(archive_path, rows, task, spec, old_profile, stage4_profile, predictors, seed):\n    """Keep the same pixel arrays, class order and item seed for every checkpoint."""\n    scores = {name: {condition: [] for condition in CONDITIONS} for name in predictors}\n    batch_size = task["batch_size"]\n    with zipfile.ZipFile(archive_path) as archive:\n        for start in range(0, len(rows), batch_size):\n            batch = rows[start : start + batch_size]\n            inputs = {condition: [] for condition in CONDITIONS}\n            for row in batch:\n                image = experiment.image_tensor(\n                    archive.read(row.path), spec["size"], task["input_policy"]\n                )\n                item_seed = experiment.item_seed(row, seed)\n                inputs["clean"].append(image)\n                for name, profile, mode in (\n                    ("lighting", old_profile, "lighting"),\n                    ("expanded", old_profile, "expanded"),\n                    ("stage4_lighting", stage4_profile, "lighting"),\n                    ("stage4_expanded", stage4_profile, "expanded"),\n                ):\n                    inputs[name].append(extended.transform(image, profile, mode, item_seed))\n            for condition, arrays in inputs.items():\n                tensor = tf.convert_to_tensor(np.stack(arrays), dtype=tf.float32)\n                for name, predict in predictors.items():\n                    result = np.asarray(predict(tensor).numpy(), dtype=np.float32)\n                    if result.shape != (len(batch), len(task["classes"])):\n                        raise ValueError(f"{name}: unexpected model output shape {result.shape}")\n                    if not np.isfinite(result).all():\n                        raise ValueError(f"{name}: nonfinite model output")\n                    scores[name][condition].append(result)\n            done = start + len(batch)\n            if start == 0 or start % (batch_size * 20) == 0 or done == len(rows):\n                training.show(\n                    f"[{task[\'domain\']}] Test \xec\x98\x88\xec\xb8\xa1 \xc2\xb7 \xeb\xaa\xa8\xeb\x8d\xb8 {len(predictors)}\xea\xb0\x9c \xc3\x97 \xec\xa1\xb0\xea\xb1\xb4 5\xea\xb0\x9c",\n                    done,\n                    len(rows),\n                )\n    arrays = {\n        name: {condition: np.concatenate(parts) for condition, parts in values.items()}\n        for name, values in scores.items()\n    }\n    metrics = {\n        name: {\n            condition: experiment.classification(rows, values, len(task["classes"]))\n            for condition, values in conditions.items()\n        }\n        for name, conditions in arrays.items()\n    }\n    return metrics, arrays\n\n\ndef verify_inputs(runtime, stages, plan, domain):\n    lock = plan["domains"][domain]\n    task = data.read_json(runtime / "configs/adaptation_stage1.json")["tasks"][domain]\n    reference = data.read_json(runtime / "references" / f"{domain}.json")\n    if training.object_hash(reference) != lock["reference_sha256"]:\n        raise ValueError(f"{domain}: original camera profile changed")\n    manifest = stages[0] / domain / "source_manifest.json"\n    if digest(manifest) != lock["manifest_sha256"]:\n        raise ValueError(f"{domain}: split manifest changed")\n    study = data.read_json(stages[3] / "study.json")\n    if (\n        digest(stages[3] / "study.json") != plan["stage4_study_file_sha256"]\n        or study["study_sha256"] != plan["stage4_study_sha256"]\n        or study["study"]["domains"][domain]["profile"] != plan["stage4_profiles"][domain]\n    ):\n        raise ValueError(f"{domain}: expected completed pre-refit Stage 4 study")\n    spec = task["models"][lock["start_role"]]\n    if task["classes"] != lock["classes"] or spec != lock["model_spec"]:\n        raise ValueError(f"{domain}: class order or input/model configuration changed")\n    paths = {}\n    for name, model in lock["models"].items():\n        if name not in plan["evaluate_arms"]:\n            continue\n        root = runtime if model["stage"] == 0 else stages[model["stage"] - 1]\n        path = data.local_path(root, model["path"])\n        if digest(path) != model["sha256"]:\n            raise ValueError(f"{domain}/{name}: checkpoint SHA-256 mismatch")\n        paths[name] = path\n    return task, spec, reference["extended_profile"], paths\n\n\ndef previous_result(runtime, output, plan, domain):\n    """Keep the original result bytes and reject mismatched comparison inputs."""\n    prior = plan["previous_test"]\n    folder = output / "previous_test"\n    folder.mkdir(exist_ok=True)\n    for name in ("evaluation_plan.json", "summary.json", f"{domain}.json"):\n        source = runtime / "previous_test" / name\n        expected = prior["files"][name]\n        if digest(source) != expected:\n            raise ValueError(f"Previous Test result bytes changed: {name}")\n        destination = folder / name\n        if destination.exists() and digest(destination) != expected:\n            raise ValueError(f"Saved previous Test result bytes changed: {name}")\n        if not destination.exists():\n            destination.write_bytes(source.read_bytes())\n    recorded_plan = data.read_json(folder / "evaluation_plan.json")\n    result = data.read_json(folder / f"{domain}.json")\n    lock = plan["domains"][domain]\n    if (\n        recorded_plan["plan_sha256"] != prior["plan_sha256"]\n        or result["identity"]["plan_sha256"] != prior["plan_sha256"]\n        or result["classes"] != lock["classes"]\n        or result["test_count"] != lock["test_count"]\n        or result["source_sha256"] != lock["source_sha256"]\n        or result["reference_sha256"] != lock["reference_sha256"]\n        or result["input_policy"] != plan["input_policies"][domain]\n        or recorded_plan["plan"]["transform_seed"] != plan["transform_seed"]\n        or recorded_plan["plan"]["domains"][domain]["manifest_sha256"] != lock["manifest_sha256"]\n        or set(result["metrics"]) != set(original.MODEL_LABELS)\n        or any(set(values) != set(original.CONDITIONS) for values in result["metrics"].values())\n    ):\n        raise ValueError(f"{domain}: previous Test conditions or data changed")\n    return result\n\n\ndef combine_results(fresh, prior):\n    return {\n        **fresh,\n        "metrics": {**prior["metrics"], **fresh["metrics"]},\n        "metric_origins": {\n            **dict.fromkeys(prior["metrics"], "reused_previous_test"),\n            **dict.fromkeys(fresh["metrics"], "new_stage4_inference"),\n        },\n    }\n\n\ndef write_comparison(output, results):\n    with (output / "test_comparison.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "domain",\n                "model",\n                "measurement_source",\n                "selected_on_validation",\n                "condition",\n                "count",\n                "accuracy",\n                "macro_f1",\n                "delta_accuracy_vs_initial",\n                "delta_macro_f1_vs_initial",\n            ]\n        )\n        for result in results:\n            for name, conditions in result["metrics"].items():\n                for condition, metrics in conditions.items():\n                    baseline = result["metrics"]["initial"].get(condition)\n                    writer.writerow(\n                        [\n                            result["domain"],\n                            name,\n                            result["metric_origins"][name],\n                            name == result["selected_on_validation"],\n                            condition,\n                            metrics["labeled_count"],\n                            metrics["accuracy"],\n                            metrics["macro_f1"],\n                            metrics["accuracy"] - baseline["accuracy"] if baseline else "",\n                            metrics["macro_f1"] - baseline["macro_f1"] if baseline else "",\n                        ]\n                    )\n\n\ndef domain_result(runtime, stages, plan, domain, archive, output):\n    task, spec, old_profile, paths = verify_inputs(runtime, stages, plan, domain)\n    lock = plan["domains"][domain]\n    rows = original.prepare_test(archive, task, lock, stages[0] / domain / "source_manifest.json")\n    predictors = {}\n    for name, path in paths.items():\n        print(f"[{domain}] {MODEL_LABELS[name]} \xeb\xaa\xa8\xeb\x8d\xb8 \xeb\xa1\x9c\xeb\x94\xa9", flush=True)\n        predictors[name] = training.predictor(keras.models.load_model(path, compile=False), spec)\n    metrics, scores = evaluate_zip(\n        archive,\n        rows,\n        task,\n        spec,\n        old_profile,\n        plan["stage4_profiles"][domain],\n        predictors,\n        plan["transform_seed"],\n    )\n    predictions = output / f"{domain}_predictions.npz"\n    payload = io.BytesIO()\n    np.savez_compressed(\n        payload,\n        **{\n            f"{name}__{condition}": values\n            for name, c in scores.items()\n            for condition, values in c.items()\n        },\n        labels=np.asarray([row.label for row in rows], dtype=np.int32),\n    )\n    predictions.write_bytes(payload.getvalue())\n    row_path = output / f"{domain}_test_rows.json"\n    write_progress(row_path, [{"path": row.path, "label": row.label} for row in rows])\n    return {\n        "domain": domain,\n        "classes": task["classes"],\n        "test_count": len(rows),\n        "source_sha256": task["source_sha256"],\n        "reference_sha256": lock["reference_sha256"],\n        "stage4_study_sha256": plan["stage4_study_sha256"],\n        "input_policy": task["input_policy"],\n        "selected_on_validation": lock["selected_on_validation"],\n        "selection_scope": plan["selection_scope"],\n        "model_sha256": {name: value["sha256"] for name, value in lock["models"].items()},\n        "prediction_files": {p.name: digest(p) for p in (predictions, row_path)},\n        "metrics": metrics,\n    }\n\n\ndef write_plots(output, results):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    from matplotlib import pyplot as plt\n\n    plots = output / "plots"\n    plots.mkdir(exist_ok=True)\n    fig, axes = plt.subplots(3, 2, figsize=(15, 12), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            for column, metric in enumerate(("accuracy", "macro_f1")):\n                axis = axes[row, column]\n                for name, conditions in result["metrics"].items():\n                    axis.plot(\n                        range(5),\n                        [conditions.get(c, {}).get(metric, np.nan) for c in CONDITIONS],\n                        marker="o",\n                        label=MODEL_LABELS[name],\n                    )\n                axis.set_xticks(range(5), CONDITION_LABELS, rotation=15, ha="right")\n                axis.set_ylim(0, 1.01)\n                axis.set_title(f"{original.DOMAIN_LABELS[result[\'domain\']]} \xc2\xb7 {metric}")\n                axis.legend(fontsize=7)\n                axis.grid(alpha=0.2)\n        fig.suptitle("Stage 4 Test + reused historical results (missing conditions not measured)")\n        fig.savefig(plots / "test_comparison.png", dpi=180)\n    finally:\n        plt.close(fig)\n    fig, axes = plt.subplots(3, 3, figsize=(16, 14), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            selected = "stage4_revised"\n            for column, condition in enumerate(("clean", "expanded", "stage4_expanded")):\n                axis = axes[row, column]\n                matrix = np.asarray(result["metrics"][selected][condition]["confusion_matrix"])\n                axis.imshow(matrix, cmap="Blues", vmin=0)\n                axis.set_xticks(range(len(matrix)))\n                axis.set_yticks(range(len(matrix)))\n                axis.set_xlabel("Predicted class index")\n                axis.set_ylabel("True class index")\n                axis.set_title(f"{result[\'domain\']} \xc2\xb7 {MODEL_LABELS[selected]} \xc2\xb7 {condition}")\n                for i, j in np.ndindex(matrix.shape):\n                    axis.text(\n                        j,\n                        i,\n                        str(matrix[i, j]),\n                        ha="center",\n                        va="center",\n                        fontsize=7,\n                        color="white" if matrix[i, j] > matrix.max() / 2 else "black",\n                    )\n        fig.suptitle("New Stage 4 Test \xc2\xb7 Class names in class_metrics.csv")\n        fig.savefig(plots / "stage4_confusion_matrices.png", dpi=180)\n    finally:\n        plt.close(fig)\n\n\ndef run(runtime, stages, dataset_root, plan_path, output, work=None):\n    plan = data.read_json(plan_path)\n    config = data.read_json(runtime / "configs/adaptation_stage1.json")\n    if training.object_hash(config) != plan["configuration_sha256"]:\n        raise ValueError("Common bundle configuration differs from locked plan")\n    for name, expected in plan["runtime_sha256"].items():\n        if digest(Path(__file__).parent / name) != expected:\n            raise ValueError(f"Evaluation code changed: {name}")\n    identity = {"plan_sha256": digest(plan_path), "evaluator_sha256": digest(__file__)}\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "evaluation_plan.json"\n    record = {**identity, "plan": plan}\n    if marker.exists() and data.read_json(marker) != record:\n        raise ValueError("Output belongs to another evaluation plan")\n    write_progress(marker, record)\n    code = output / "code"\n    code.mkdir(exist_ok=True)\n    for name in plan["runtime_sha256"]:\n        (code / name).write_bytes((Path(__file__).parent / name).read_bytes())\n    results = []\n    for domain in training.DOMAINS:\n        verify_inputs(runtime, stages, plan, domain)\n        prior = previous_result(runtime, output, plan, domain)\n        destination = output / f"{domain}.json"\n        if destination.exists():\n            result = data.read_json(destination)\n            if result["identity"] != identity or any(\n                digest(output / name) != expected\n                for name, expected in result["prediction_files"].items()\n            ):\n                raise ValueError(f"{domain}: saved evaluation identity or prediction file changed")\n            print(f"[{domain}] \xeb\x8f\x99\xec\x9d\xbc \xea\xb3\x84\xed\x9a\x8d\xec\x9d\x98 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        else:\n            source = find_source(dataset_root, config["tasks"][domain])\n            with tempfile.TemporaryDirectory(dir=work) if work else nullcontext() as scratch:\n                archive = source\n                if scratch:\n                    archive = Path(scratch) / "source.zip"\n                    copy_source(source, archive)\n                result = domain_result(runtime, stages, plan, domain, archive, output)\n            result.update(\n                identity=identity,\n                environment={\n                    "python": platform.python_version(),\n                    "tensorflow": tf.__version__,\n                    "keras": keras.__version__,\n                    "numpy": np.__version__,\n                    "opencv": cv2.__version__,\n                    "matplotlib": version("matplotlib"),\n                    "gpu": [\n                        tf.config.experimental.get_device_details(d)\n                        for d in tf.config.list_physical_devices("GPU")\n                    ],\n                },\n            )\n            write_progress(destination, result)\n            keras.backend.clear_session()\n        results.append(combine_results(result, prior))\n    write_comparison(output, results)\n    original.write_class_metrics(output, results)\n    write_plots(output, results)\n    write_progress(\n        output / "summary.json",\n        {\n            **identity,\n            "completed_domains": [r["domain"] for r in results],\n            "comparisons": sum(\n                len(r["metrics"][arm]) for r in results for arm in plan["evaluate_arms"]\n            ),\n            "new_inference_arms": plan["evaluate_arms"],\n            "reused_comparisons": 36,\n            "combined_comparisons": sum(len(c) for r in results for c in r["metrics"].values()),\n            "source_test_evaluated": True,\n            "real_device_accuracy": None,\n            "prior_source_test_inspected": True,\n            "training_executed": False,\n            "test_used_for_model_selection": False,\n        },\n    )\n    print(f"\xec\x99\x84\xeb\xa3\x8c: {output / \'test_comparison.csv\'}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--runtime", type=Path, required=True)\n    parser.add_argument("--stages", type=Path, nargs=4, required=True)\n    parser.add_argument("--datasets", type=Path, required=True)\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--work", type=Path)\n    args = parser.parse_args()\n    run(args.runtime, args.stages, args.datasets, args.plan, args.output, args.work)\n\n\nif __name__ == "__main__":\n    main()\n')
    prior = runtime / "previous_test"
    prior.mkdir()
    for name, body in {'evaluation_plan.json': b'{\n  "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n  "evaluator_sha256": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6",\n  "plan": {\n    "scope": "source_test_only_historical_checkpoint_independence_unverified",\n    "bundle_sha256": "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe",\n    "validation_sha256": "5bd16693270fa0962825588a95508412677da866e6d15a745b535bf5d84a32f4",\n    "selection_rule": "Maximize minimum of clean, lighting and expanded Validation Macro-F1; then clean Validation Macro-F1; then earlier arm.",\n    "transform_seed": 1707,\n    "conditions": [\n      "clean",\n      "lighting",\n      "expanded"\n    ],\n    "git_commit": "ffd71eca68ff1e21c145ed56158f4cfa77f7f037",\n    "domains": {\n      "skin": {\n        "start_role": "basic",\n        "models": {\n          "initial": {\n            "stage": 0,\n            "path": "results/skin/1_training/original/best_model.keras",\n            "sha256": "1b146a6a53742226ec677865aa71bd842a28c9db9c3174b352dd8bc87270770b"\n          },\n          "stage1_lighting": {\n            "stage": 1,\n            "path": "skin/basic/best.keras",\n            "sha256": "4bca967a479f40745e2d48e81579d442831e955a6193934b5840364fc1e419bb"\n          },\n          "stage2_expanded": {\n            "stage": 2,\n            "path": "skin/best.keras",\n            "sha256": "9687fa503d8287b880e58e5014cae0a0efd47e750f5ececcc11d813367e7ce6b"\n          },\n          "stage3_aligned": {\n            "stage": 3,\n            "path": "skin/best.keras",\n            "sha256": "98d9de1af93d31cf2a07cb828d6c29b33b7166f7169da14ca1bc53b7bc2f6f3f"\n          }\n        },\n        "selected_on_validation": "stage1_lighting",\n        "validation_macro_f1": {\n          "initial": {\n            "clean": 0.9965963674000184,\n            "lighting": 0.9976017479221756,\n            "expanded": 0.9976017479221756\n          },\n          "stage1_lighting": {\n            "clean": 0.9976017479221756,\n            "lighting": 0.9976017479221756,\n            "expanded": 0.9976017479221756\n          },\n          "stage2_expanded": {\n            "clean": 0.9976017479221756,\n            "lighting": 0.9976017479221756,\n            "expanded": 0.9976017479221756\n          },\n          "stage3_aligned": {\n            "clean": 0.9976017479221756,\n            "lighting": 0.9976017479221756,\n            "expanded": 0.9976017479221756\n          }\n        },\n        "reference_sha256": "1837fd554de1363d4695b01190d0625420e5fdde9775313d732fb93e69876172",\n        "manifest_sha256": "36f79b61a42e9ed634ffa7d4c490efefe5e5031a8bbdce1daf630b1e01e7e27d",\n        "test_count": 700\n      },\n      "web_skin": {\n        "start_role": "selected",\n        "models": {\n          "initial": {\n            "stage": 0,\n            "path": "results/web_skin/selected_models/v2/web_skin_model.keras",\n            "sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4"\n          },\n          "stage1_lighting": {\n            "stage": 1,\n            "path": "web_skin/selected/best.keras",\n            "sha256": "beb992fa2bf51c28c7b2e760b1c7000bb5b65decef47c0e196ed07c9a62070e5"\n          },\n          "stage2_expanded": {\n            "stage": 2,\n            "path": "web_skin/best.keras",\n            "sha256": "5e181c1ff064eb1d501c57ccecba91703cfe59b14d02b258152b3ca737c46981"\n          },\n          "stage3_aligned": {\n            "stage": 3,\n            "path": "web_skin/best.keras",\n            "sha256": "9df65bdf905434e8671afdc034e832170c667193ba803213771d11d5afa5ff71"\n          }\n        },\n        "selected_on_validation": "stage2_expanded",\n        "validation_macro_f1": {\n          "initial": {\n            "clean": 0.8471375958306038,\n            "lighting": 0.8264309294807919,\n            "expanded": 0.8330199818760329\n          },\n          "stage1_lighting": {\n            "clean": 0.8436210626904069,\n            "lighting": 0.8354160598851343,\n            "expanded": 0.8197582784927677\n          },\n          "stage2_expanded": {\n            "clean": 0.8452139236284577,\n            "lighting": 0.8414941635470983,\n            "expanded": 0.8297960345454877\n          },\n          "stage3_aligned": {\n            "clean": 0.8463380823691281,\n            "lighting": 0.8360852895678168,\n            "expanded": 0.8261149876826673\n          }\n        },\n        "reference_sha256": "239245cc7a0804c931ddd45b5b9408474fb8cce5838a00b00d65471b87221b4e",\n        "manifest_sha256": "7c8b41878883193e5dd340ab8a4b7f804a3ab7f37c8159093b663937d997a738",\n        "test_count": 400\n      },\n      "hair": {\n        "start_role": "selected",\n        "models": {\n          "initial": {\n            "stage": 0,\n            "path": "results/hair/selected_models/v2/hair_model.keras",\n            "sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2"\n          },\n          "stage1_lighting": {\n            "stage": 1,\n            "path": "hair/selected/best.keras",\n            "sha256": "983001b0b116cd913b4cf1e349b47d11ab6246cf78fbaa607dc75b4578108b7f"\n          },\n          "stage2_expanded": {\n            "stage": 2,\n            "path": "hair/best.keras",\n            "sha256": "fa2c59ae6a0b1342b8c693014d0a951c59064a4c8439a313f2520e1ce8c224fd"\n          },\n          "stage3_aligned": {\n            "stage": 3,\n            "path": "hair/best.keras",\n            "sha256": "940b398c7bfdba033df58e99b2639eef8e8eabb0c8c19945dd8f89cfd265e1d7"\n          }\n        },\n        "selected_on_validation": "stage2_expanded",\n        "validation_macro_f1": {\n          "initial": {\n            "clean": 0.7948815320131865,\n            "lighting": 0.7932265946320327,\n            "expanded": 0.7812966127563242\n          },\n          "stage1_lighting": {\n            "clean": 0.7940808319964521,\n            "lighting": 0.790851219358226,\n            "expanded": 0.7717843351060846\n          },\n          "stage2_expanded": {\n            "clean": 0.7933089589962153,\n            "lighting": 0.7891828841548413,\n            "expanded": 0.7892573288809733\n          },\n          "stage3_aligned": {\n            "clean": 0.7924600237364962,\n            "lighting": 0.7891245054928616,\n            "expanded": 0.7884731261645151\n          }\n        },\n        "reference_sha256": "1a44a143fe6f5ef3021865b75f4797ffff16b1e21bae7dc7ca838eb70a09a81f",\n        "manifest_sha256": "c6320fa7d2ef25332d508b3da868d2731d61ce7c8cd3ed2ebc505e6cd0b54717",\n        "test_count": 1252\n      }\n    },\n    "configuration_sha256": "ee9604c652ace8a6d7d0df5656b70ef7512f0ae40f7ef2211f081432b75d9366",\n    "predictor_code_sha256": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a"\n  }\n}', 'summary.json': b'{\n  "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n  "evaluator_sha256": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6",\n  "completed_domains": [\n    "skin",\n    "web_skin",\n    "hair"\n  ],\n  "comparisons": 36,\n  "source_test_evaluated": true,\n  "real_device_accuracy": null\n}', 'skin.json': b'{\n  "domain": "skin",\n  "classes": [\n    "\xea\xb4\x91\xec\x84\xa0\xea\xb0\x81\xed\x99\x94\xec\xa6\x9d",\n    "\xea\xb8\xb0\xec\xa0\x80\xec\x84\xb8\xed\x8f\xac\xec\x95\x94",\n    "\xeb\xb3\xb4\xec\x9b\xac\xeb\xb3\x91",\n    "\xec\x82\xac\xeb\xa7\x88\xea\xb7\x80",\n    "\xec\xa7\x80\xeb\xa3\xa8\xea\xb0\x81\xed\x99\x94\xec\xa6\x9d",\n    "\xed\x8e\xb8\xed\x8f\x89\xec\x84\xb8\xed\x8f\xac\xec\x95\x94",\n    "\xed\x91\x9c\xed\x94\xbc\xeb\x82\xad\xec\xa2\x85",\n    "\xed\x94\xbc\xeb\xb6\x80\xec\x84\xac\xec\x9c\xa0\xec\xa2\x85",\n    "\xed\x98\x88\xea\xb4\x80\xec\xa2\x85",\n    "\xed\x9d\x91\xec\x83\x89\xec\xa0\x90"\n  ],\n  "test_count": 700,\n  "source_sha256": "db6cf59087b68d4534ff857bdb0eacc1a1a09ed5703772fefc75f038d4570e4b",\n  "reference_sha256": "1837fd554de1363d4695b01190d0625420e5fdde9775313d732fb93e69876172",\n  "input_policy": "legacy_resize",\n  "selected_on_validation": "stage1_lighting",\n  "metrics": {\n    "initial": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      }\n    },\n    "stage1_lighting": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      }\n    },\n    "stage2_expanded": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      }\n    },\n    "stage3_aligned": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 700,\n        "accuracy": 0.9985714285714286,\n        "macro_f1": 0.9985713556814122,\n        "class_f1": [\n          1.0,\n          1.0,\n          1.0,\n          0.9928057553956835,\n          1.0,\n          0.9929078014184397,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "precision": [\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          0.9859154929577465,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "recall": [\n          1.0,\n          1.0,\n          1.0,\n          0.9857142857142858,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0,\n          1.0\n        ],\n        "support": [\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70,\n          70\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            69,\n            0,\n            1,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70,\n            0\n          ],\n          [\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            0,\n            70\n          ]\n        ],\n        "count": 700\n      }\n    }\n  },\n  "identity": {\n    "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n    "evaluator_sha256": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6"\n  },\n  "environment": {\n    "python": "3.13.16",\n    "tensorflow": "2.20.0",\n    "keras": "3.13.2",\n    "numpy": "2.5.4",\n    "opencv": "4.14.0",\n    "matplotlib": "3.11.2",\n    "gpu": [\n      {\n        "compute_capability": [\n          8,\n          9\n        ],\n        "device_name": "NVIDIA L4"\n      }\n    ]\n  }\n}', 'web_skin.json': b'{\n  "domain": "web_skin",\n  "classes": [\n    "\xea\xb1\xb4\xec\x84\xa0",\n    "\xec\x95\x84\xed\x86\xa0\xed\x94\xbc",\n    "\xec\x97\xac\xeb\x93\x9c\xeb\xa6\x84",\n    "\xec\xa0\x95\xec\x83\x81",\n    "\xec\xa3\xbc\xec\x82\xac"\n  ],\n  "test_count": 400,\n  "source_sha256": "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d",\n  "reference_sha256": "239245cc7a0804c931ddd45b5b9408474fb8cce5838a00b00d65471b87221b4e",\n  "input_policy": "center_square",\n  "selected_on_validation": "stage2_expanded",\n  "metrics": {\n    "initial": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.915,\n        "macro_f1": 0.9141049081029712,\n        "class_f1": [\n          0.8627450980392157,\n          0.8860759493670887,\n          0.888888888888889,\n          0.9815950920245399,\n          0.951219512195122\n        ],\n        "precision": [\n          0.9041095890410958,\n          0.8974358974358975,\n          0.8780487804878049,\n          0.963855421686747,\n          0.9285714285714286\n        ],\n        "recall": [\n          0.825,\n          0.875,\n          0.9,\n          1.0,\n          0.975\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            66,\n            6,\n            6,\n            0,\n            2\n          ],\n          [\n            3,\n            70,\n            4,\n            2,\n            1\n          ],\n          [\n            3,\n            1,\n            72,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            1,\n            0,\n            0,\n            78\n          ]\n        ],\n        "count": 400\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.8975,\n        "macro_f1": 0.8962809484925597,\n        "class_f1": [\n          0.8481012658227848,\n          0.8513513513513514,\n          0.8674698795180723,\n          0.9696969696969697,\n          0.9447852760736196\n        ],\n        "precision": [\n          0.8589743589743589,\n          0.9264705882352942,\n          0.8372093023255814,\n          0.9411764705882353,\n          0.927710843373494\n        ],\n        "recall": [\n          0.8375,\n          0.7875,\n          0.9,\n          1.0,\n          0.9625\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            4,\n            7,\n            0,\n            2\n          ],\n          [\n            6,\n            63,\n            6,\n            4,\n            1\n          ],\n          [\n            4,\n            0,\n            72,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            1,\n            1,\n            0,\n            77\n          ]\n        ],\n        "count": 400\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.8975,\n        "macro_f1": 0.8963901626478314,\n        "class_f1": [\n          0.8589743589743589,\n          0.8513513513513514,\n          0.8571428571428572,\n          0.9696969696969697,\n          0.9447852760736196\n        ],\n        "precision": [\n          0.881578947368421,\n          0.9264705882352942,\n          0.8181818181818182,\n          0.9411764705882353,\n          0.927710843373494\n        ],\n        "recall": [\n          0.8375,\n          0.7875,\n          0.9,\n          1.0,\n          0.9625\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            4,\n            7,\n            0,\n            2\n          ],\n          [\n            4,\n            63,\n            8,\n            4,\n            1\n          ],\n          [\n            4,\n            0,\n            72,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            1,\n            1,\n            0,\n            77\n          ]\n        ],\n        "count": 400\n      }\n    },\n    "stage1_lighting": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.9025,\n        "macro_f1": 0.902391120932396,\n        "class_f1": [\n          0.8627450980392157,\n          0.875,\n          0.8554216867469879,\n          0.9937888198757764,\n          0.925\n        ],\n        "precision": [\n          0.9041095890410958,\n          0.875,\n          0.8255813953488372,\n          0.9876543209876543,\n          0.925\n        ],\n        "recall": [\n          0.825,\n          0.875,\n          0.8875,\n          1.0,\n          0.925\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            66,\n            6,\n            7,\n            0,\n            1\n          ],\n          [\n            3,\n            70,\n            5,\n            1,\n            1\n          ],\n          [\n            3,\n            2,\n            71,\n            0,\n            4\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            2,\n            3,\n            0,\n            74\n          ]\n        ],\n        "count": 400\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.9,\n        "macro_f1": 0.8999783814868859,\n        "class_f1": [\n          0.8553459119496856,\n          0.8774193548387097,\n          0.8554216867469879,\n          0.9876543209876543,\n          0.9240506329113924\n        ],\n        "precision": [\n          0.8607594936708861,\n          0.9066666666666666,\n          0.8255813953488372,\n          0.975609756097561,\n          0.9358974358974359\n        ],\n        "recall": [\n          0.85,\n          0.85,\n          0.8875,\n          1.0,\n          0.9125\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            68,\n            5,\n            6,\n            0,\n            1\n          ],\n          [\n            4,\n            68,\n            6,\n            1,\n            1\n          ],\n          [\n            5,\n            0,\n            71,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            2,\n            2,\n            3,\n            0,\n            73\n          ]\n        ],\n        "count": 400\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.895,\n        "macro_f1": 0.8952042407093288,\n        "class_f1": [\n          0.8481012658227848,\n          0.8701298701298702,\n          0.8470588235294118,\n          0.9876543209876543,\n          0.9230769230769231\n        ],\n        "precision": [\n          0.8589743589743589,\n          0.9054054054054054,\n          0.8,\n          0.975609756097561,\n          0.9473684210526315\n        ],\n        "recall": [\n          0.8375,\n          0.8375,\n          0.9,\n          1.0,\n          0.9\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            5,\n            7,\n            0,\n            1\n          ],\n          [\n            4,\n            67,\n            7,\n            1,\n            1\n          ],\n          [\n            5,\n            0,\n            72,\n            1,\n            2\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            2,\n            2,\n            4,\n            0,\n            72\n          ]\n        ],\n        "count": 400\n      }\n    },\n    "stage2_expanded": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.9075,\n        "macro_f1": 0.9074049501223416,\n        "class_f1": [\n          0.8701298701298702,\n          0.8875,\n          0.8606060606060607,\n          0.9937888198757764,\n          0.925\n        ],\n        "precision": [\n          0.9054054054054054,\n          0.8875,\n          0.8352941176470589,\n          0.9876543209876543,\n          0.925\n        ],\n        "recall": [\n          0.8375,\n          0.8875,\n          0.8875,\n          1.0,\n          0.925\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            5,\n            7,\n            0,\n            1\n          ],\n          [\n            3,\n            71,\n            4,\n            1,\n            1\n          ],\n          [\n            3,\n            2,\n            71,\n            0,\n            4\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            2,\n            3,\n            0,\n            74\n          ]\n        ],\n        "count": 400\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.8975,\n        "macro_f1": 0.8975250542584641,\n        "class_f1": [\n          0.8535031847133757,\n          0.8717948717948718,\n          0.8502994011976048,\n          0.9937888198757764,\n          0.9182389937106918\n        ],\n        "precision": [\n          0.8701298701298701,\n          0.8947368421052632,\n          0.8160919540229885,\n          0.9876543209876543,\n          0.9240506329113924\n        ],\n        "recall": [\n          0.8375,\n          0.85,\n          0.8875,\n          1.0,\n          0.9125\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            5,\n            7,\n            0,\n            1\n          ],\n          [\n            4,\n            68,\n            6,\n            1,\n            1\n          ],\n          [\n            4,\n            1,\n            71,\n            0,\n            4\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            2,\n            2,\n            3,\n            0,\n            73\n          ]\n        ],\n        "count": 400\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.895,\n        "macro_f1": 0.8950348370179594,\n        "class_f1": [\n          0.8481012658227848,\n          0.8701298701298702,\n          0.8452380952380951,\n          0.9876543209876543,\n          0.9240506329113924\n        ],\n        "precision": [\n          0.8589743589743589,\n          0.9054054054054054,\n          0.8068181818181818,\n          0.975609756097561,\n          0.9358974358974359\n        ],\n        "recall": [\n          0.8375,\n          0.8375,\n          0.8875,\n          1.0,\n          0.9125\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            67,\n            5,\n            7,\n            0,\n            1\n          ],\n          [\n            4,\n            67,\n            7,\n            1,\n            1\n          ],\n          [\n            5,\n            0,\n            71,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            2,\n            2,\n            3,\n            0,\n            73\n          ]\n        ],\n        "count": 400\n      }\n    },\n    "stage3_aligned": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.91,\n        "macro_f1": 0.9095171874784814,\n        "class_f1": [\n          0.8571428571428572,\n          0.8701298701298702,\n          0.8875,\n          0.9937888198757764,\n          0.9390243902439025\n        ],\n        "precision": [\n          0.8518518518518519,\n          0.9054054054054054,\n          0.8875,\n          0.9876543209876543,\n          0.9166666666666666\n        ],\n        "recall": [\n          0.8625,\n          0.8375,\n          0.8875,\n          1.0,\n          0.9625\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            69,\n            5,\n            4,\n            0,\n            2\n          ],\n          [\n            6,\n            67,\n            5,\n            1,\n            1\n          ],\n          [\n            5,\n            0,\n            71,\n            0,\n            4\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            2,\n            0,\n            0,\n            77\n          ]\n        ],\n        "count": 400\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.9025,\n        "macro_f1": 0.9014455597953545,\n        "class_f1": [\n          0.8641975308641976,\n          0.8456375838926175,\n          0.8711656441717791,\n          0.975609756097561,\n          0.9506172839506174\n        ],\n        "precision": [\n          0.8536585365853658,\n          0.9130434782608695,\n          0.8554216867469879,\n          0.9523809523809523,\n          0.9390243902439024\n        ],\n        "recall": [\n          0.875,\n          0.7875,\n          0.8875,\n          1.0,\n          0.9625\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            4,\n            5,\n            0,\n            1\n          ],\n          [\n            6,\n            63,\n            7,\n            3,\n            1\n          ],\n          [\n            5,\n            0,\n            71,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            2,\n            0,\n            0,\n            77\n          ]\n        ],\n        "count": 400\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 400,\n        "accuracy": 0.9075,\n        "macro_f1": 0.9066525239267813,\n        "class_f1": [\n          0.8695652173913043,\n          0.8609271523178808,\n          0.8765432098765432,\n          0.975609756097561,\n          0.9506172839506174\n        ],\n        "precision": [\n          0.8641975308641975,\n          0.9154929577464789,\n          0.8658536585365854,\n          0.9523809523809523,\n          0.9390243902439024\n        ],\n        "recall": [\n          0.875,\n          0.8125,\n          0.8875,\n          1.0,\n          0.9625\n        ],\n        "support": [\n          80,\n          80,\n          80,\n          80,\n          80\n        ],\n        "confusion_matrix": [\n          [\n            70,\n            4,\n            5,\n            0,\n            1\n          ],\n          [\n            5,\n            65,\n            6,\n            3,\n            1\n          ],\n          [\n            5,\n            0,\n            71,\n            1,\n            3\n          ],\n          [\n            0,\n            0,\n            0,\n            80,\n            0\n          ],\n          [\n            1,\n            2,\n            0,\n            0,\n            77\n          ]\n        ],\n        "count": 400\n      }\n    }\n  },\n  "identity": {\n    "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n    "evaluator_sha256": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6"\n  },\n  "environment": {\n    "python": "3.13.16",\n    "tensorflow": "2.20.0",\n    "keras": "3.13.2",\n    "numpy": "2.5.4",\n    "opencv": "4.14.0",\n    "matplotlib": "3.11.2",\n    "gpu": [\n      {\n        "compute_capability": [\n          8,\n          9\n        ],\n        "device_name": "NVIDIA L4"\n      }\n    ]\n  }\n}', 'hair.json': b'{\n  "domain": "hair",\n  "classes": [\n    "\xeb\xaa\xa8\xeb\x82\xad\xec\x82\xac\xec\x9d\xb4\xed\x99\x8d\xeb\xb0\x98",\n    "\xeb\xaf\xb8\xec\x84\xb8\xea\xb0\x81\xec\xa7\x88",\n    "\xeb\xb9\x84\xeb\x93\xac",\n    "\xed\x83\x88\xeb\xaa\xa8",\n    "\xed\x94\xbc\xec\xa7\x80\xea\xb3\xbc\xeb\x8b\xa4"\n  ],\n  "test_count": 1252,\n  "source_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156",\n  "reference_sha256": "1a44a143fe6f5ef3021865b75f4797ffff16b1e21bae7dc7ca838eb70a09a81f",\n  "input_policy": "legacy_resize",\n  "selected_on_validation": "stage2_expanded",\n  "metrics": {\n    "initial": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.8003194888178914,\n        "macro_f1": 0.8002222282821301,\n        "class_f1": [\n          0.8275862068965518,\n          0.7807692307692308,\n          0.7499999999999999,\n          0.8442622950819672,\n          0.7984934086629002\n        ],\n        "precision": [\n          0.8326530612244898,\n          0.7689393939393939,\n          0.7662337662337663,\n          0.8477366255144033,\n          0.7881040892193308\n        ],\n        "recall": [\n          0.8225806451612904,\n          0.79296875,\n          0.7344398340248963,\n          0.8408163265306122,\n          0.8091603053435115\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            204,\n            11,\n            8,\n            18,\n            7\n          ],\n          [\n            10,\n            203,\n            27,\n            3,\n            13\n          ],\n          [\n            1,\n            40,\n            177,\n            1,\n            22\n          ],\n          [\n            14,\n            4,\n            6,\n            206,\n            15\n          ],\n          [\n            16,\n            6,\n            13,\n            15,\n            212\n          ]\n        ],\n        "count": 1252\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7963258785942492,\n        "macro_f1": 0.7960594412980679,\n        "class_f1": [\n          0.8204081632653061,\n          0.7748091603053434,\n          0.7483870967741936,\n          0.842741935483871,\n          0.7939508506616257\n        ],\n        "precision": [\n          0.8305785123966942,\n          0.7574626865671642,\n          0.7767857142857143,\n          0.8326693227091634,\n          0.7865168539325843\n        ],\n        "recall": [\n          0.8104838709677419,\n          0.79296875,\n          0.7219917012448133,\n          0.8530612244897959,\n          0.8015267175572519\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            201,\n            14,\n            5,\n            20,\n            8\n          ],\n          [\n            11,\n            203,\n            27,\n            3,\n            12\n          ],\n          [\n            1,\n            42,\n            174,\n            2,\n            22\n          ],\n          [\n            13,\n            3,\n            5,\n            209,\n            15\n          ],\n          [\n            16,\n            6,\n            13,\n            17,\n            210\n          ]\n        ],\n        "count": 1252\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7939297124600639,\n        "macro_f1": 0.794078156187424,\n        "class_f1": [\n          0.8225469728601251,\n          0.7747747747747747,\n          0.7581699346405228,\n          0.8218623481781376,\n          0.793036750483559\n        ],\n        "precision": [\n          0.8528138528138528,\n          0.7190635451505016,\n          0.7981651376146789,\n          0.8152610441767069,\n          0.803921568627451\n        ],\n        "recall": [\n          0.7943548387096774,\n          0.83984375,\n          0.7219917012448133,\n          0.8285714285714286,\n          0.7824427480916031\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            197,\n            16,\n            5,\n            23,\n            7\n          ],\n          [\n            9,\n            215,\n            21,\n            4,\n            7\n          ],\n          [\n            1,\n            45,\n            174,\n            3,\n            18\n          ],\n          [\n            12,\n            9,\n            3,\n            203,\n            18\n          ],\n          [\n            12,\n            14,\n            15,\n            16,\n            205\n          ]\n        ],\n        "count": 1252\n      }\n    },\n    "stage1_lighting": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7971246006389776,\n        "macro_f1": 0.7971826536894018,\n        "class_f1": [\n          0.8253968253968254,\n          0.783300198807157,\n          0.7605633802816901,\n          0.8316430020283976,\n          0.7850098619329389\n        ],\n        "precision": [\n          0.8125,\n          0.7975708502024291,\n          0.73828125,\n          0.8266129032258065,\n          0.8122448979591836\n        ],\n        "recall": [\n          0.8387096774193549,\n          0.76953125,\n          0.7842323651452282,\n          0.8367346938775511,\n          0.7595419847328244\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            208,\n            8,\n            9,\n            17,\n            6\n          ],\n          [\n            12,\n            197,\n            32,\n            6,\n            9\n          ],\n          [\n            1,\n            33,\n            189,\n            1,\n            17\n          ],\n          [\n            17,\n            3,\n            6,\n            205,\n            14\n          ],\n          [\n            18,\n            6,\n            20,\n            19,\n            199\n          ]\n        ],\n        "count": 1252\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7963258785942492,\n        "macro_f1": 0.7961773287158287,\n        "class_f1": [\n          0.8232931726907632,\n          0.7821011673151751,\n          0.756198347107438,\n          0.8303393213572854,\n          0.7889546351084814\n        ],\n        "precision": [\n          0.82,\n          0.7790697674418605,\n          0.7530864197530864,\n          0.8125,\n          0.8163265306122449\n        ],\n        "recall": [\n          0.8266129032258065,\n          0.78515625,\n          0.7593360995850622,\n          0.8489795918367347,\n          0.7633587786259542\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            205,\n            10,\n            7,\n            20,\n            6\n          ],\n          [\n            11,\n            201,\n            30,\n            6,\n            8\n          ],\n          [\n            1,\n            38,\n            183,\n            2,\n            17\n          ],\n          [\n            15,\n            3,\n            5,\n            208,\n            14\n          ],\n          [\n            18,\n            6,\n            18,\n            20,\n            200\n          ]\n        ],\n        "count": 1252\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7915335463258786,\n        "macro_f1": 0.7915936209173309,\n        "class_f1": [\n          0.8163265306122448,\n          0.7794117647058824,\n          0.7615062761506275,\n          0.8209255533199196,\n          0.7797979797979797\n        ],\n        "precision": [\n          0.8264462809917356,\n          0.7361111111111112,\n          0.7679324894514767,\n          0.8095238095238095,\n          0.8283261802575107\n        ],\n        "recall": [\n          0.8064516129032258,\n          0.828125,\n          0.7551867219917012,\n          0.8326530612244898,\n          0.7366412213740458\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            200,\n            14,\n            6,\n            22,\n            6\n          ],\n          [\n            10,\n            212,\n            25,\n            3,\n            6\n          ],\n          [\n            1,\n            40,\n            182,\n            3,\n            15\n          ],\n          [\n            15,\n            9,\n            4,\n            204,\n            13\n          ],\n          [\n            16,\n            13,\n            20,\n            20,\n            193\n          ]\n        ],\n        "count": 1252\n      }\n    },\n    "stage2_expanded": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7947284345047924,\n        "macro_f1": 0.7951323060366875,\n        "class_f1": [\n          0.8140495867768595,\n          0.7847619047619048,\n          0.7515151515151517,\n          0.8322981366459626,\n          0.793036750483559\n        ],\n        "precision": [\n          0.8347457627118644,\n          0.7657992565055762,\n          0.7322834645669292,\n          0.8445378151260504,\n          0.803921568627451\n        ],\n        "recall": [\n          0.7943548387096774,\n          0.8046875,\n          0.7717842323651453,\n          0.8204081632653061,\n          0.7824427480916031\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            197,\n            15,\n            9,\n            19,\n            8\n          ],\n          [\n            8,\n            206,\n            30,\n            2,\n            10\n          ],\n          [\n            1,\n            35,\n            186,\n            1,\n            18\n          ],\n          [\n            15,\n            7,\n            8,\n            201,\n            14\n          ],\n          [\n            15,\n            6,\n            21,\n            15,\n            205\n          ]\n        ],\n        "count": 1252\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7947284345047924,\n        "macro_f1": 0.7947964808800927,\n        "class_f1": [\n          0.8083333333333332,\n          0.7849056603773584,\n          0.7510373443983401,\n          0.8373983739837398,\n          0.7923076923076924\n        ],\n        "precision": [\n          0.8362068965517241,\n          0.7591240875912408,\n          0.7510373443983402,\n          0.8340080971659919,\n          0.7984496124031008\n        ],\n        "recall": [\n          0.782258064516129,\n          0.8125,\n          0.7510373443983402,\n          0.8408163265306122,\n          0.7862595419847328\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            194,\n            16,\n            9,\n            21,\n            8\n          ],\n          [\n            8,\n            208,\n            26,\n            4,\n            10\n          ],\n          [\n            1,\n            39,\n            181,\n            1,\n            19\n          ],\n          [\n            14,\n            4,\n            6,\n            206,\n            15\n          ],\n          [\n            15,\n            7,\n            19,\n            15,\n            206\n          ]\n        ],\n        "count": 1252\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.792332268370607,\n        "macro_f1": 0.7924369913269433,\n        "class_f1": [\n          0.8148148148148147,\n          0.7792207792207793,\n          0.7452631578947368,\n          0.8305785123966942,\n          0.7923076923076924\n        ],\n        "precision": [\n          0.8319327731092437,\n          0.7420494699646644,\n          0.7564102564102564,\n          0.8410041841004184,\n          0.7984496124031008\n        ],\n        "recall": [\n          0.7983870967741935,\n          0.8203125,\n          0.7344398340248963,\n          0.8204081632653061,\n          0.7862595419847328\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            198,\n            15,\n            7,\n            21,\n            7\n          ],\n          [\n            9,\n            210,\n            25,\n            3,\n            9\n          ],\n          [\n            1,\n            43,\n            177,\n            1,\n            19\n          ],\n          [\n            15,\n            7,\n            5,\n            201,\n            17\n          ],\n          [\n            15,\n            8,\n            20,\n            13,\n            206\n          ]\n        ],\n        "count": 1252\n      }\n    },\n    "stage3_aligned": {\n      "clean": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7947284345047924,\n        "macro_f1": 0.795185171769517,\n        "class_f1": [\n          0.8189300411522633,\n          0.7809523809523808,\n          0.7464503042596348,\n          0.8357588357588358,\n          0.7938342967244703\n        ],\n        "precision": [\n          0.8361344537815126,\n          0.7620817843866171,\n          0.7301587301587301,\n          0.8516949152542372,\n          0.8015564202334631\n        ],\n        "recall": [\n          0.8024193548387096,\n          0.80078125,\n          0.7634854771784232,\n          0.8204081632653061,\n          0.7862595419847328\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            199,\n            15,\n            9,\n            17,\n            8\n          ],\n          [\n            8,\n            205,\n            31,\n            2,\n            10\n          ],\n          [\n            1,\n            36,\n            184,\n            1,\n            19\n          ],\n          [\n            15,\n            7,\n            8,\n            201,\n            14\n          ],\n          [\n            15,\n            6,\n            20,\n            15,\n            206\n          ]\n        ],\n        "count": 1252\n      },\n      "lighting": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7947284345047924,\n        "macro_f1": 0.7947652590801082,\n        "class_f1": [\n          0.8108108108108107,\n          0.7863894139886579,\n          0.7484407484407485,\n          0.8373983739837398,\n          0.7907869481765836\n        ],\n        "precision": [\n          0.8369098712446352,\n          0.7619047619047619,\n          0.75,\n          0.8340080971659919,\n          0.7953667953667953\n        ],\n        "recall": [\n          0.7862903225806451,\n          0.8125,\n          0.7468879668049793,\n          0.8408163265306122,\n          0.7862595419847328\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            195,\n            15,\n            9,\n            21,\n            8\n          ],\n          [\n            8,\n            208,\n            26,\n            4,\n            10\n          ],\n          [\n            1,\n            39,\n            180,\n            1,\n            20\n          ],\n          [\n            14,\n            4,\n            6,\n            206,\n            15\n          ],\n          [\n            15,\n            7,\n            19,\n            15,\n            206\n          ]\n        ],\n        "count": 1252\n      },\n      "expanded": {\n        "status": "measured",\n        "labeled_count": 1252,\n        "accuracy": 0.7931309904153354,\n        "macro_f1": 0.7932563613616767,\n        "class_f1": [\n          0.8172484599589324,\n          0.7806691449814126,\n          0.7478991596638656,\n          0.8281573498964804,\n          0.7923076923076924\n        ],\n        "precision": [\n          0.8326359832635983,\n          0.7446808510638298,\n          0.7574468085106383,\n          0.8403361344537815,\n          0.7984496124031008\n        ],\n        "recall": [\n          0.8024193548387096,\n          0.8203125,\n          0.7385892116182573,\n          0.8163265306122449,\n          0.7862595419847328\n        ],\n        "support": [\n          248,\n          256,\n          241,\n          245,\n          262\n        ],\n        "confusion_matrix": [\n          [\n            199,\n            14,\n            7,\n            21,\n            7\n          ],\n          [\n            9,\n            210,\n            25,\n            3,\n            9\n          ],\n          [\n            1,\n            42,\n            178,\n            1,\n            19\n          ],\n          [\n            15,\n            8,\n            5,\n            200,\n            17\n          ],\n          [\n            15,\n            8,\n            20,\n            13,\n            206\n          ]\n        ],\n        "count": 1252\n      }\n    }\n  },\n  "identity": {\n    "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n    "evaluator_sha256": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6"\n  },\n  "environment": {\n    "python": "3.13.16",\n    "tensorflow": "2.20.0",\n    "keras": "3.13.2",\n    "numpy": "2.5.4",\n    "opencv": "4.14.0",\n    "matplotlib": "3.11.2",\n    "gpu": [\n      {\n        "compute_capability": [\n          8,\n          9\n        ],\n        "device_name": "NVIDIA L4"\n      }\n    ]\n  }\n}'}.items():
        (prior / name).write_bytes(body)
    plan_path = work / "final_test_plan.json"
    plan_path.write_bytes(b'{\n  "scope": "source_test_only_historical_checkpoint_independence_unverified",\n  "bundle_sha256": "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe",\n  "validation_sha256": "5bd16693270fa0962825588a95508412677da866e6d15a745b535bf5d84a32f4",\n  "selection_rule": "max worst(clean, previous expanded, stronger expanded) F1; clean; earlier arm",\n  "transform_seed": 1707,\n  "conditions": [\n    "clean",\n    "lighting",\n    "expanded",\n    "stage4_lighting",\n    "stage4_expanded"\n  ],\n  "git_commit": "08cb827aa3c6858e61c71faf7f6481f3a139530b",\n  "domains": {\n    "skin": {\n      "start_role": "basic",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/skin/1_training/original/best_model.keras",\n          "sha256": "1b146a6a53742226ec677865aa71bd842a28c9db9c3174b352dd8bc87270770b"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "skin/basic/best.keras",\n          "sha256": "4bca967a479f40745e2d48e81579d442831e955a6193934b5840364fc1e419bb"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "skin/best.keras",\n          "sha256": "9687fa503d8287b880e58e5014cae0a0efd47e750f5ececcc11d813367e7ce6b"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "skin/best.keras",\n          "sha256": "98d9de1af93d31cf2a07cb828d6c29b33b7166f7169da14ca1bc53b7bc2f6f3f"\n        },\n        "stage4_revised": {\n          "stage": 4,\n          "path": "skin/best.keras",\n          "sha256": "c62a76580a0d6658c934b993629a33605961b153d28ac1c6abc3205382815be2"\n        }\n      },\n      "selected_on_validation": "stage4_revised",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.9965963674000184,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage1_lighting": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage2_expanded": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage3_aligned": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        }\n      },\n      "reference_sha256": "1837fd554de1363d4695b01190d0625420e5fdde9775313d732fb93e69876172",\n      "manifest_sha256": "36f79b61a42e9ed634ffa7d4c490efefe5e5031a8bbdce1daf630b1e01e7e27d",\n      "test_count": 700,\n      "classes": [\n        "\xea\xb4\x91\xec\x84\xa0\xea\xb0\x81\xed\x99\x94\xec\xa6\x9d",\n        "\xea\xb8\xb0\xec\xa0\x80\xec\x84\xb8\xed\x8f\xac\xec\x95\x94",\n        "\xeb\xb3\xb4\xec\x9b\xac\xeb\xb3\x91",\n        "\xec\x82\xac\xeb\xa7\x88\xea\xb7\x80",\n        "\xec\xa7\x80\xeb\xa3\xa8\xea\xb0\x81\xed\x99\x94\xec\xa6\x9d",\n        "\xed\x8e\xb8\xed\x8f\x89\xec\x84\xb8\xed\x8f\xac\xec\x95\x94",\n        "\xed\x91\x9c\xed\x94\xbc\xeb\x82\xad\xec\xa2\x85",\n        "\xed\x94\xbc\xeb\xb6\x80\xec\x84\xac\xec\x9c\xa0\xec\xa2\x85",\n        "\xed\x98\x88\xea\xb4\x80\xec\xa2\x85",\n        "\xed\x9d\x91\xec\x83\x89\xec\xa0\x90"\n      ],\n      "source_sha256": "db6cf59087b68d4534ff857bdb0eacc1a1a09ed5703772fefc75f038d4570e4b",\n      "model_spec": {\n        "architecture": "b0",\n        "size": 224,\n        "label_smoothing": 0.0,\n        "initialization": "saved original-only early checkpoint",\n        "model": "results/skin/1_training/original/best_model.keras",\n        "model_sha256": "1b146a6a53742226ec677865aa71bd842a28c9db9c3174b352dd8bc87270770b"\n      },\n      "selection_validation_macro_f1": {\n        "stage4_revised": {\n          "clean": 0.9976017479221756,\n          "weak": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "initial": {\n          "clean": 0.9965963674000184,\n          "weak": 0.9976017479221756,\n          "expanded": 0.9935152536910259\n        },\n        "stage2_expanded": {\n          "clean": 0.9976017479221756,\n          "weak": 0.9976017479221756,\n          "expanded": 0.993168147190449\n        }\n      },\n      "stage4_validation_report_sha256": "446779d677cdff03f26d70d799618a01aacd7c9a5586749173925913460dffc5"\n    },\n    "web_skin": {\n      "start_role": "selected",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/web_skin/selected_models/v2/web_skin_model.keras",\n          "sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "web_skin/selected/best.keras",\n          "sha256": "beb992fa2bf51c28c7b2e760b1c7000bb5b65decef47c0e196ed07c9a62070e5"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "web_skin/best.keras",\n          "sha256": "5e181c1ff064eb1d501c57ccecba91703cfe59b14d02b258152b3ca737c46981"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "web_skin/best.keras",\n          "sha256": "9df65bdf905434e8671afdc034e832170c667193ba803213771d11d5afa5ff71"\n        },\n        "stage4_revised": {\n          "stage": 4,\n          "path": "web_skin/best.keras",\n          "sha256": "bbdfade5173c12d4076ab414c907821e21ee57fee2366f79d57adde4a8f74dc9"\n        }\n      },\n      "selected_on_validation": "stage4_revised",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.8471375958306038,\n          "lighting": 0.8264309294807919,\n          "expanded": 0.8330199818760329\n        },\n        "stage1_lighting": {\n          "clean": 0.8436210626904069,\n          "lighting": 0.8354160598851343,\n          "expanded": 0.8197582784927677\n        },\n        "stage2_expanded": {\n          "clean": 0.8452139236284577,\n          "lighting": 0.8414941635470983,\n          "expanded": 0.8297960345454877\n        },\n        "stage3_aligned": {\n          "clean": 0.8463380823691281,\n          "lighting": 0.8360852895678168,\n          "expanded": 0.8261149876826673\n        }\n      },\n      "reference_sha256": "239245cc7a0804c931ddd45b5b9408474fb8cce5838a00b00d65471b87221b4e",\n      "manifest_sha256": "7c8b41878883193e5dd340ab8a4b7f804a3ab7f37c8159093b663937d997a738",\n      "test_count": 400,\n      "classes": [\n        "\xea\xb1\xb4\xec\x84\xa0",\n        "\xec\x95\x84\xed\x86\xa0\xed\x94\xbc",\n        "\xec\x97\xac\xeb\x93\x9c\xeb\xa6\x84",\n        "\xec\xa0\x95\xec\x83\x81",\n        "\xec\xa3\xbc\xec\x82\xac"\n      ],\n      "source_sha256": "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d",\n      "model_spec": {\n        "architecture": "pmg",\n        "size": 256,\n        "label_smoothing": 0.0,\n        "reference_candidate_id": "public_candidate_v2_pmg_b0_256_ce_20260922_235840_093d10de",\n        "initialization": "saved selected checkpoint",\n        "model": "results/web_skin/selected_models/v2/web_skin_model.keras",\n        "model_sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4"\n      },\n      "selection_validation_macro_f1": {\n        "stage4_revised": {\n          "clean": 0.8440681425959449,\n          "weak": 0.8196219458629163,\n          "expanded": 0.8309329752560016\n        },\n        "initial": {\n          "clean": 0.8471375958306038,\n          "weak": 0.8330199818760329,\n          "expanded": 0.7951552177660287\n        },\n        "stage2_expanded": {\n          "clean": 0.8452139236284577,\n          "weak": 0.8297960345454877,\n          "expanded": 0.8041675654309934\n        }\n      },\n      "stage4_validation_report_sha256": "9b37f24de479709e9c53f832fd9995c27433ea90da74f52c8b1857812dd9f400"\n    },\n    "hair": {\n      "start_role": "selected",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/hair/selected_models/v2/hair_model.keras",\n          "sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "hair/selected/best.keras",\n          "sha256": "983001b0b116cd913b4cf1e349b47d11ab6246cf78fbaa607dc75b4578108b7f"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "hair/best.keras",\n          "sha256": "fa2c59ae6a0b1342b8c693014d0a951c59064a4c8439a313f2520e1ce8c224fd"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "hair/best.keras",\n          "sha256": "940b398c7bfdba033df58e99b2639eef8e8eabb0c8c19945dd8f89cfd265e1d7"\n        },\n        "stage4_revised": {\n          "stage": 4,\n          "path": "hair/best.keras",\n          "sha256": "ddb123669956f4fbabea791d7815da7aa16101a3ed331ef4cbc7503848f5ed00"\n        }\n      },\n      "selected_on_validation": "stage2_expanded",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.7948815320131865,\n          "lighting": 0.7932265946320327,\n          "expanded": 0.7812966127563242\n        },\n        "stage1_lighting": {\n          "clean": 0.7940808319964521,\n          "lighting": 0.790851219358226,\n          "expanded": 0.7717843351060846\n        },\n        "stage2_expanded": {\n          "clean": 0.7933089589962153,\n          "lighting": 0.7891828841548413,\n          "expanded": 0.7892573288809733\n        },\n        "stage3_aligned": {\n          "clean": 0.7924600237364962,\n          "lighting": 0.7891245054928616,\n          "expanded": 0.7884731261645151\n        }\n      },\n      "reference_sha256": "1a44a143fe6f5ef3021865b75f4797ffff16b1e21bae7dc7ca838eb70a09a81f",\n      "manifest_sha256": "c6320fa7d2ef25332d508b3da868d2731d61ce7c8cd3ed2ebc505e6cd0b54717",\n      "test_count": 1252,\n      "classes": [\n        "\xeb\xaa\xa8\xeb\x82\xad\xec\x82\xac\xec\x9d\xb4\xed\x99\x8d\xeb\xb0\x98",\n        "\xeb\xaf\xb8\xec\x84\xb8\xea\xb0\x81\xec\xa7\x88",\n        "\xeb\xb9\x84\xeb\x93\xac",\n        "\xed\x83\x88\xeb\xaa\xa8",\n        "\xed\x94\xbc\xec\xa7\x80\xea\xb3\xbc\xeb\x8b\xa4"\n      ],\n      "source_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156",\n      "model_spec": {\n        "architecture": "b1",\n        "size": 384,\n        "label_smoothing": 0.05,\n        "reference_candidate_id": "public_candidate_v2_b1_384_ls005_adam_20260921_155906_82311da4",\n        "initialization": "saved selected checkpoint",\n        "model": "results/hair/selected_models/v2/hair_model.keras",\n        "model_sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2",\n        "training_source_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156"\n      },\n      "selection_validation_macro_f1": {\n        "stage4_revised": {\n          "clean": 0.7926449140430689,\n          "weak": 0.774844874001947,\n          "expanded": 0.7882173427367334\n        },\n        "initial": {\n          "clean": 0.7957429931637543,\n          "weak": 0.7812966127563242,\n          "expanded": 0.7846213015924425\n        },\n        "stage2_expanded": {\n          "clean": 0.7924812646314018,\n          "weak": 0.7900578506535009,\n          "expanded": 0.7820517795657131\n        }\n      },\n      "stage4_validation_report_sha256": "e35ff6d066d37e2b4bd789eecf4591a5e50fd3a3b1635f1cf611f70aeab676e4"\n    }\n  },\n  "configuration_sha256": "ee9604c652ace8a6d7d0df5656b70ef7512f0ae40f7ef2211f081432b75d9366",\n  "predictor_code_sha256": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a",\n  "schema": "mediflow_stage4_only_source_test_v2",\n  "evaluate_arms": [\n    "stage4_revised"\n  ],\n  "input_policies": {\n    "skin": "legacy_resize",\n    "web_skin": "center_square",\n    "hair": "legacy_resize"\n  },\n  "previous_test": {\n    "plan_sha256": "bc4d235c1354b937f9f12c0927e6dd3c5e2d4e754ed442fc13a7962ddf0614c6",\n    "files": {\n      "evaluation_plan.json": "64942a94580c4ff954c434197a50519c471031349f5f83677a301642ff1c6707",\n      "summary.json": "c80430f58b85a17a626d3ed856825b66e221f7b63b54c6fd6aa2a009dfced29f",\n      "skin.json": "969c75a24b5879ef0ad3246cb00fffcbe13918e58513f79a0baaca8ce6090e92",\n      "web_skin.json": "8fc4eb46116c4a9c9c4722505e894bfaec9975051ace1b99f890f5254c1a11d1",\n      "hair.json": "1adbcdb98d2e3f3fab33512b8cb098915e0242ea80dd4ccfd2fa7923d7ef8f0f"\n    }\n  },\n  "stage4_study_sha256": "492ee01afb4579b6402fca29459c59d56618243589642fc022db18bbbe45a165",\n  "stage4_study_file_sha256": "de429b5a85d902645ac83b79f865591eaf2d8147d41f69b9937d98856e593767",\n  "stage4_drive_folder": "measured_strength_492ee01afb45",\n  "stage4_selection_sha256": "bb38d00addb783ce4eab29a8fbf3fc10319129646c32949f364b34b942790c01",\n  "stage4_profiles": {\n    "skin": {\n      "schema": "measured_camera_extended_v1",\n      "domain": "skin",\n      "lighting": {\n        "domain": "skin",\n        "profile_version": "bounded_source_target_strength_v1",\n        "brightness_offset": [\n          32.0,\n          32.0\n        ],\n        "contrast": [\n          0.8,\n          0.8\n        ],\n        "scale_x": [\n          1.0,\n          1.0\n        ],\n        "scale_y": [\n          1.0,\n          1.0\n        ],\n        "left_light": [\n          0.0,\n          0.0\n        ],\n        "evidence": {\n          "method": "Source class-balanced median to equal participant/condition Target quartiles",\n          "anchors": {\n            "y_p50": 83.40784072875977,\n            "contrast": 70.15270042419434\n          },\n          "coverage": {\n            "y_p50": {\n              "class_medians": [\n                96.70617294311523,\n                77.98558807373047,\n                79.60416030883789,\n                104.08260345458984,\n                73.82204055786133,\n                87.21152114868164,\n                109.7138786315918,\n                77.95131301879883,\n                55.02832794189453,\n                92.87034606933594\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 723,\n                  "measured": 723\n                },\n                {\n                  "total": 722,\n                  "measured": 722\n                },\n                {\n                  "total": 709,\n                  "measured": 709\n                },\n                {\n                  "total": 714,\n                  "measured": 714\n                },\n                {\n                  "total": 691,\n                  "measured": 691\n                },\n                {\n                  "total": 725,\n                  "measured": 725\n                },\n                {\n                  "total": 703,\n                  "measured": 703\n                },\n                {\n                  "total": 519,\n                  "measured": 519\n                },\n                {\n                  "total": 702,\n                  "measured": 702\n                },\n                {\n                  "total": 545,\n                  "measured": 545\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                50.152610778808594,\n                86.36319732666016,\n                54.131675720214844,\n                55.54798126220703,\n                74.786376953125,\n                84.3202133178711,\n                74.34446716308594,\n                67.56494903564453,\n                72.74045181274414,\n                44.280799865722656\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 723,\n                  "measured": 723\n                },\n                {\n                  "total": 722,\n                  "measured": 722\n                },\n                {\n                  "total": 709,\n                  "measured": 709\n                },\n                {\n                  "total": 714,\n                  "measured": 714\n                },\n                {\n                  "total": 691,\n                  "measured": 691\n                },\n                {\n                  "total": 725,\n                  "measured": 725\n                },\n                {\n                  "total": 703,\n                  "measured": 703\n                },\n                {\n                  "total": 519,\n                  "measured": 519\n                },\n                {\n                  "total": 702,\n                  "measured": 702\n                },\n                {\n                  "total": 545,\n                  "measured": 545\n                }\n              ]\n            }\n          }\n        },\n        "bounds": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        },\n        "held_back": [\n          "color: unmatched disease color",\n          "blur/noise: texture/scale confounding",\n          "hair mask: not validated and could change class evidence",\n          "geometry: no rescaling onto artificial padding; composition is an input policy"\n        ],\n        "photometric_operator": "bounded_additive_v3",\n        "input_policy": "legacy_resize",\n        "source_sha256": "db6cf59087b68d4534ff857bdb0eacc1a1a09ed5703772fefc75f038d4570e4b",\n        "reference_model_sha256": "0b2a757320a113dbc1b285adab8c7745b2f7edc2cb29cc402d06f7882edc9f7a",\n        "calibration_fingerprint": "659c8bb3edf406ba89fe782781b4fe9b7ac065cfc87411d26c895aff7a4b75ea",\n        "source_subset_sha256": "59766e93e57214d1922caa5579fe78f8790a56a9fa5cdc7b439c43a93efc529b",\n        "train_visual_review": "revised_train_replay_not_clinical_validation",\n        "review_note": "Same retained Train previews and actual calibration captures remeasured. Unlabeled descriptive audit; disease-feature preservation and accuracy unverified.",\n        "calibration_version": "direction_and_focus_audited_v1"\n      },\n      "color_a": [\n        -0.9609375,\n        0.0\n      ],\n      "color_b": [\n        0.0,\n        0.92578125\n      ],\n      "texture_gain": [\n        1.0,\n        1.0\n      ],\n      "blur_sigma_roi64": [\n        0.0,\n        0.0\n      ],\n      "blur_input_scale": 1.75,\n      "evidence": {\n        "chroma": {\n          "lab_a": {\n            "participant_changes": [\n              0.15625,\n              0.6875,\n              -0.515625,\n              -1.78125\n            ],\n            "range": [\n              -0.9609375,\n              0.0\n            ]\n          },\n          "lab_b": {\n            "participant_changes": [\n              -1.453125,\n              0.75,\n              0.09375,\n              -0.515625\n            ],\n            "range": [\n              0.0,\n              0.92578125\n            ]\n          }\n        },\n        "normalized_detail_ratios": [\n          1.5315814159199996,\n          0.9506550225759118,\n          0.93334946929935,\n          0.8952716478170912\n        ],\n        "focus_fits": [\n          {\n            "person": "P01",\n            "condition": "alt",\n            "sigma_roi64": 0.6000000000000001,\n            "reference_path": "skin/P01/S01/P01_S01_skin_alt_20261108T152804327485.png",\n            "compared_paths": [\n              "skin/P01/S01/P01_S01_skin_alt_20261108T152757692299.png",\n              "skin/P01/S01/P01_S01_skin_alt_20261108T152759879125.png",\n              "skin/P01/S01/P01_S01_skin_alt_20261108T152802271030.png",\n              "skin/P01/S01/P01_S01_skin_alt_20261108T152804327485.png",\n              "skin/P01/S01/P01_S01_skin_alt_20261108T152806871550.png"\n            ],\n            "normalized_detail_reference": 0.14585232734680176,\n            "normalized_detail_target_q25": 0.09714946150779724,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P01",\n            "condition": "ref",\n            "sigma_roi64": 0.55,\n            "reference_path": "skin/P01/S01/P01_S01_skin_ref_20261108T152702885605.png",\n            "compared_paths": [\n              "skin/P01/S01/P01_S01_skin_ref_20261108T152654406408.png",\n              "skin/P01/S01/P01_S01_skin_ref_20261108T152656085726.png",\n              "skin/P01/S01/P01_S01_skin_ref_20261108T152700575134.png",\n              "skin/P01/S01/P01_S01_skin_ref_20261108T152702885605.png",\n              "skin/P01/S01/P01_S01_skin_ref_20261108T152726981437.png"\n            ],\n            "normalized_detail_reference": 0.0943775475025177,\n            "normalized_detail_target_q25": 0.06878045946359634,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "alt",\n            "sigma_roi64": 0.55,\n            "reference_path": "skin/P02/S01/P02_S01_skin_ref_20261108T155710902205.png",\n            "compared_paths": [\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155705604551.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155707918391.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155710902205.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155714224403.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155717083061.png"\n            ],\n            "normalized_detail_reference": 0.1551624834537506,\n            "normalized_detail_target_q25": 0.11530275642871857,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "ref",\n            "sigma_roi64": 0.35000000000000003,\n            "reference_path": "skin/P02/S01/P02_S01_skin_ref_20261108T155640349143.png",\n            "compared_paths": [\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155640349143.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155643620743.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155647691054.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155648665676.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155649721131.png",\n              "skin/P02/S01/P02_S01_skin_ref_20261108T155650848730.png"\n            ],\n            "normalized_detail_reference": 0.12822462618350983,\n            "normalized_detail_target_q25": 0.12492126226425171,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "alt",\n            "sigma_roi64": 0.45,\n            "reference_path": "skin/P03/S01/P03_S01_skin_alt_20261108T160340079280.png",\n            "compared_paths": [\n              "skin/P03/S01/P03_S01_skin_alt_20261108T160327738793.png",\n              "skin/P03/S01/P03_S01_skin_alt_20261108T160329708329.png",\n              "skin/P03/S01/P03_S01_skin_alt_20261108T160331984806.png",\n              "skin/P03/S01/P03_S01_skin_alt_20261108T160335180900.png",\n              "skin/P03/S01/P03_S01_skin_alt_20261108T160340079280.png"\n            ],\n            "normalized_detail_reference": 0.1409766525030136,\n            "normalized_detail_target_q25": 0.1246052235364914,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "skin/P03/S01/P03_S01_skin_ref_20261108T160307547385.png",\n            "compared_paths": [\n              "skin/P03/S01/P03_S01_skin_ref_20261108T160300453688.png",\n              "skin/P03/S01/P03_S01_skin_ref_20261108T160302282924.png",\n              "skin/P03/S01/P03_S01_skin_ref_20261108T160304846565.png",\n              "skin/P03/S01/P03_S01_skin_ref_20261108T160307547385.png",\n              "skin/P03/S01/P03_S01_skin_ref_20261108T160313317627.png"\n            ],\n            "normalized_detail_reference": 0.14165307581424713,\n            "normalized_detail_target_q25": 0.12634940445423126,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "alt",\n            "sigma_roi64": 0.6000000000000001,\n            "reference_path": "skin/P04/S01/P04_S01_skin_alt_20261108T161536584800.png",\n            "compared_paths": [\n              "skin/P04/S01/P04_S01_skin_alt_20261108T161524502585.png",\n              "skin/P04/S01/P04_S01_skin_alt_20261108T161527705996.png",\n              "skin/P04/S01/P04_S01_skin_alt_20261108T161531183868.png",\n              "skin/P04/S01/P04_S01_skin_alt_20261108T161533594835.png",\n              "skin/P04/S01/P04_S01_skin_alt_20261108T161536584800.png"\n            ],\n            "normalized_detail_reference": 0.11807460337877274,\n            "normalized_detail_target_q25": 0.08239267021417618,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "ref",\n            "sigma_roi64": 0.7000000000000001,\n            "reference_path": "skin/P04/S01/P04_S01_skin_ref_20261108T161422697239.png",\n            "compared_paths": [\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161420722968.png",\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161422697239.png",\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161424373063.png",\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161429837877.png",\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161432163032.png",\n              "skin/P04/S01/P04_S01_skin_ref_20261108T161435199939.png"\n            ],\n            "normalized_detail_reference": 0.12333618104457855,\n            "normalized_detail_target_q25": 0.07842539250850677,\n            "fit_at_search_limit": false\n          }\n        ],\n        "source_train_statistics": {\n          "width": {\n            "n": 6753,\n            "mean": 512.0,\n            "var": 0.0,\n            "q05": 512.0,\n            "q25": 512.0,\n            "q50": 512.0,\n            "q75": 512.0,\n            "q95": 512.0\n          },\n          "height": {\n            "n": 6753,\n            "mean": 512.0,\n            "var": 0.0,\n            "q05": 512.0,\n            "q25": 512.0,\n            "q50": 512.0,\n            "q75": 512.0,\n            "q95": 512.0\n          },\n          "aspect": {\n            "n": 6753,\n            "mean": 1.0,\n            "var": 0.0,\n            "q05": 1.0,\n            "q25": 1.0,\n            "q50": 1.0,\n            "q75": 1.0,\n            "q95": 1.0\n          },\n          "frame_y_mean": {\n            "n": 6753,\n            "mean": 114.47958635284726,\n            "var": 87.24607925917684,\n            "q05": 98.81668624877929,\n            "q25": 108.35001754760742,\n            "q50": 114.64883804321289,\n            "q75": 120.7322006225586,\n            "q95": 129.66388626098632\n          },\n          "y_mean": {\n            "n": 6753,\n            "mean": 90.30464909543784,\n            "var": 314.8961629560137,\n            "q05": 59.69416427612305,\n            "q25": 77.81758880615234,\n            "q50": 91.09374237060547,\n            "q75": 103.08687591552734,\n            "q95": 118.37844696044921\n          },\n          "y_std": {\n            "n": 6753,\n            "mean": 27.32309096134558,\n            "var": 59.87403125311148,\n            "q05": 15.252275657653808,\n            "q25": 21.501598358154297,\n            "q50": 27.000465393066406,\n            "q75": 32.88024139404297,\n            "q95": 40.44438552856445\n          },\n          "y_p10": {\n            "n": 6753,\n            "mean": 58.886400914315765,\n            "var": 380.82845967317195,\n            "q05": 29.144909973144532,\n            "q25": 43.101741790771484,\n            "q50": 58.33773231506348,\n            "q75": 74.27768020629883,\n            "q95": 90.3705628967285\n          },\n          "y_p50": {\n            "n": 6753,\n            "mean": 86.80831658029916,\n            "var": 406.0092309856499,\n            "q05": 51.69489097595215,\n            "q25": 72.77033233642578,\n            "q50": 88.37580490112305,\n            "q75": 101.51531600952148,\n            "q95": 117.89541549682617\n          },\n          "y_p90": {\n            "n": 6753,\n            "mean": 125.98163479816716,\n            "var": 397.3568733147368,\n            "q05": 93.61960815429688,\n            "q25": 114.22539291381838,\n            "q50": 125.44330825805665,\n            "q75": 137.827294921875,\n            "q95": 160.09986907958984\n          },\n          "contrast": {\n            "n": 6753,\n            "mean": 67.09523358282188,\n            "var": 458.93342413074026,\n            "q05": 34.41724853515625,\n            "q25": 50.24510955810547,\n            "q50": 66.11698150634766,\n            "q75": 82.55947875976562,\n            "q95": 103.58233337402343\n          },\n          "very_bright": {\n            "n": 6753,\n            "mean": 1.7931845105878867e-05,\n            "var": 1.0437930924882913e-08,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 7.971938775510203e-05\n          },\n          "very_dark": {\n            "n": 6753,\n            "mean": 0.00015975751140082868,\n            "var": 2.2884004608760086e-06,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.00015943877551020407\n          },\n          "detail": {\n            "n": 6753,\n            "mean": 5.901556804200269,\n            "var": 6.615591828532237,\n            "q05": 2.219536018371582,\n            "q25": 3.957143783569336,\n            "q50": 5.602933883666992,\n            "q75": 7.533182144165039,\n            "q95": 10.604405212402344\n          },\n          "lab_a": {\n            "n": 6753,\n            "mean": 18.31927245298386,\n            "var": 76.4908259028224,\n            "q05": 5.015625,\n            "q25": 11.890625,\n            "q50": 17.34375,\n            "q75": 24.9921875,\n            "q95": 33.5\n          },\n          "lab_b": {\n            "n": 6753,\n            "mean": 13.335503665037761,\n            "var": 30.128186772458058,\n            "q05": 3.8125,\n            "q25": 9.875,\n            "q50": 13.390625,\n            "q75": 16.6875,\n            "q95": 22.734375\n          },\n          "lighting_range": {\n            "n": 6753,\n            "mean": 26.869026558886063,\n            "var": 282.25989583423865,\n            "q05": 6.5154876708984375,\n            "q25": 14.311271667480469,\n            "q50": 23.121639251708984,\n            "q75": 35.64080047607422,\n            "q95": 60.97312698364257\n          },\n          "r_p10": {\n            "n": 6753,\n            "mean": 92.25889119671201,\n            "var": 738.8185138834799,\n            "q05": 37.77306228637696,\n            "q25": 78.43979797363282,\n            "q50": 96.28571319580078,\n            "q75": 111.57449417114258,\n            "q95": 130.64938873291015\n          },\n          "r_p50": {\n            "n": 6753,\n            "mean": 121.62911342624804,\n            "var": 429.5406325190565,\n            "q05": 79.82091979980468,\n            "q25": 110.66836547851562,\n            "q50": 124.61734771728516,\n            "q75": 135.83163452148438,\n            "q95": 150.2193817138672\n          },\n          "r_p90": {\n            "n": 6753,\n            "mean": 153.09953683395307,\n            "var": 266.527168117746,\n            "q05": 125.96499908447267,\n            "q25": 142.97449798583983,\n            "q50": 153.19898529052733,\n            "q75": 163.71429443359375,\n            "q95": 179.5838781738281\n          },\n          "g_p10": {\n            "n": 6753,\n            "mean": 44.17473054689141,\n            "var": 422.4624981550124,\n            "q05": 14.131632347106935,\n            "q25": 27.714284896850586,\n            "q50": 42.1275520324707,\n            "q75": 59.944896697998054,\n            "q95": 78.92408203125\n          },\n          "g_p50": {\n            "n": 6753,\n            "mean": 73.4532289986573,\n            "var": 547.4737761948525,\n            "q05": 30.594898223876953,\n            "q25": 57.85714340209961,\n            "q50": 74.06887817382812,\n            "q75": 89.97193908691406,\n            "q95": 111.20306320190429\n          },\n          "g_p90": {\n            "n": 6753,\n            "mean": 117.08673418831088,\n            "var": 548.7626369136294,\n            "q05": 80.03285888671877,\n            "q25": 103.84948806762698,\n            "q50": 116.17857360839844,\n            "q75": 130.30510864257815,\n            "q95": 158.40153076171876\n          },\n          "b_p10": {\n            "n": 6753,\n            "mean": 38.451087429021065,\n            "var": 226.14975422575247,\n            "q05": 16.55806118011475,\n            "q25": 27.408163642883302,\n            "q50": 36.846427536010744,\n            "q75": 47.909695053100585,\n            "q95": 65.53969284057614\n          },\n          "b_p50": {\n            "n": 6753,\n            "mean": 65.42372289799097,\n            "var": 291.69216518860287,\n            "q05": 38.85867347717285,\n            "q25": 52.68877601623535,\n            "q50": 64.63010025024414,\n            "q75": 76.96683883666992,\n            "q95": 94.93622589111327\n          },\n          "b_p90": {\n            "n": 6753,\n            "mean": 107.51134333289255,\n            "var": 428.51867497457596,\n            "q05": 75.50520294189452,\n            "q25": 93.04846801757814,\n            "q50": 106.0,\n            "q75": 120.47193756103518,\n            "q95": 144.04938293457027\n          }\n        },\n        "target_statistics": {\n          "width": {\n            "n": 42,\n            "mean": 640.0,\n            "var": 0.0,\n            "q05": 640.0,\n            "q25": 640.0,\n            "q50": 640.0,\n            "q75": 640.0,\n            "q95": 640.0\n          },\n          "height": {\n            "n": 42,\n            "mean": 480.0,\n            "var": 0.0,\n            "q05": 480.0,\n            "q25": 480.0,\n            "q50": 480.0,\n            "q75": 480.0,\n            "q95": 480.0\n          },\n          "aspect": {\n            "n": 42,\n            "mean": 1.3333333333333333,\n            "var": 0.0,\n            "q05": 1.3333333333333333,\n            "q25": 1.3333333333333333,\n            "q50": 1.3333333333333333,\n            "q75": 1.3333333333333333,\n            "q95": 1.3333333333333333\n          },\n          "frame_y_mean": {\n            "n": 42,\n            "mean": 141.80286993117556,\n            "var": 4.310451009794682,\n            "q05": 138.94495605468748,\n            "q25": 140.46587158203124,\n            "q50": 141.25586100260415,\n            "q75": 143.70915283203126,\n            "q95": 145.35050097656247\n          },\n          "y_mean": {\n            "n": 42,\n            "mean": 155.47330329531715,\n            "var": 5.008440635400553,\n            "q05": 152.89668960571288,\n            "q25": 153.63165283203125,\n            "q50": 154.96517181396484,\n            "q75": 157.4436798095703,\n            "q95": 159.37026138305663\n          },\n          "y_std": {\n            "n": 42,\n            "mean": 9.428914728618803,\n            "var": 2.829139449333622,\n            "q05": 7.127552080154419,\n            "q25": 8.250247478485107,\n            "q50": 9.326010704040527,\n            "q75": 10.292496919631958,\n            "q95": 13.22414555549621\n          },\n          "y_p10": {\n            "n": 42,\n            "mean": 144.35247751871745,\n            "var": 5.533187754240107,\n            "q05": 140.94701385498047,\n            "q25": 142.9382297515869,\n            "q50": 143.66613082885743,\n            "q75": 146.12437324523927,\n            "q95": 148.2756336212158\n          },\n          "y_p50": {\n            "n": 42,\n            "mean": 154.95250393095472,\n            "var": 6.147326018129242,\n            "q05": 151.74092483520508,\n            "q25": 153.26583099365234,\n            "q50": 154.2252426147461,\n            "q75": 157.12484169006348,\n            "q95": 159.48736534118652\n          },\n          "y_p90": {\n            "n": 42,\n            "mean": 167.9121224176316,\n            "var": 8.209270375926142,\n            "q05": 162.84856636047365,\n            "q25": 165.90727386474612,\n            "q50": 168.01725158691406,\n            "q75": 169.62824020385742,\n            "q95": 172.11037048339844\n          },\n          "contrast": {\n            "n": 42,\n            "mean": 23.559644426618302,\n            "var": 9.826000824754386,\n            "q05": 19.020677947998045,\n            "q25": 21.123985290527344,\n            "q50": 23.768699645996094,\n            "q75": 26.311141967773438,\n            "q95": 27.667374420166013\n          },\n          "very_bright": {\n            "n": 42,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "very_dark": {\n            "n": 42,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "detail": {\n            "n": 42,\n            "mean": 2.5531061859357926,\n            "var": 0.515576507486669,\n            "q05": 1.3356806814670563,\n            "q25": 2.046028733253479,\n            "q50": 2.5456701517105103,\n            "q75": 3.2130183577537537,\n            "q95": 3.5456066250801084\n          },\n          "lab_a": {\n            "n": 42,\n            "mean": 2.4094122023809526,\n            "var": 1.3908315299589888,\n            "q05": 0.5265625,\n            "q25": 1.58984375,\n            "q50": 2.1953125,\n            "q75": 3.20703125,\n            "q95": 4.363281249999999\n          },\n          "lab_b": {\n            "n": 42,\n            "mean": 30.124441964285715,\n            "var": 5.113811363168314,\n            "q05": 27.596875,\n            "q25": 28.54296875,\n            "q50": 30.4765625,\n            "q75": 31.60546875,\n            "q95": 33.81015625\n          },\n          "lighting_range": {\n            "n": 42,\n            "mean": 4.645108904157366,\n            "var": 5.420522390871126,\n            "q05": 2.0537261962890625,\n            "q25": 2.7186660766601562,\n            "q50": 4.137176513671875,\n            "q75": 5.598941802978516,\n            "q95": 8.73028793334961\n          },\n          "r_p10": {\n            "n": 42,\n            "mean": 169.24217747279576,\n            "var": 14.438573350575775,\n            "q05": 164.11377334594727,\n            "q25": 165.52041244506836,\n            "q50": 168.3040802001953,\n            "q75": 172.53316116333008,\n            "q95": 175.86770347595214\n          },\n          "r_p50": {\n            "n": 42,\n            "mean": 178.81146712530227,\n            "var": 15.664006627342044,\n            "q05": 174.13571166992188,\n            "q25": 174.9285774230957,\n            "q50": 178.6556167602539,\n            "q75": 182.05101776123047,\n            "q95": 185.90255012512208\n          },\n          "r_p90": {\n            "n": 42,\n            "mean": 190.1097420102074,\n            "var": 15.070765466794745,\n            "q05": 183.20698570251466,\n            "q25": 188.30867385864258,\n            "q50": 189.88265228271484,\n            "q75": 192.12499618530273,\n            "q95": 196.22515365600586\n          },\n          "g_p10": {\n            "n": 42,\n            "mean": 142.0577498299735,\n            "var": 6.571455264308846,\n            "q05": 138.49030227661132,\n            "q25": 140.42857360839844,\n            "q50": 141.3484680175781,\n            "q75": 143.60025291442872,\n            "q95": 146.84218849182128\n          },\n          "g_p50": {\n            "n": 42,\n            "mean": 153.0891640072777,\n            "var": 6.768668294293346,\n            "q05": 149.02831802368163,\n            "q25": 151.22066688537598,\n            "q50": 152.82142639160156,\n            "q75": 154.6364803314209,\n            "q95": 157.30229568481445\n          },\n          "g_p90": {\n            "n": 42,\n            "mean": 166.57378619966053,\n            "var": 9.316482722943652,\n            "q05": 161.804598236084,\n            "q25": 164.26811180114748,\n            "q50": 166.43214645385743,\n            "q75": 168.77551651000977,\n            "q95": 171.23229400634764\n          },\n          "b_p10": {\n            "n": 42,\n            "mean": 88.87910610380628,\n            "var": 23.33520545854283,\n            "q05": 82.05377723693849,\n            "q25": 85.81964302062988,\n            "q50": 87.89948806762695,\n            "q75": 91.59413394927978,\n            "q95": 96.33178791046141\n          },\n          "b_p50": {\n            "n": 42,\n            "mean": 102.36479586646671,\n            "var": 16.101650183758434,\n            "q05": 97.44540672302246,\n            "q25": 99.79591369628906,\n            "q50": 102.03571319580078,\n            "q75": 105.5076551437378,\n            "q95": 107.72602291107178\n          },\n          "b_p90": {\n            "n": 42,\n            "mean": 118.02538942609515,\n            "var": 19.96475796479221,\n            "q05": 111.144030418396,\n            "q25": 115.1709213256836,\n            "q50": 118.38265609741211,\n            "q75": 121.53239612579345,\n            "q95": 123.92857360839844\n          }\n        }\n      },\n      "engineering_bounds": {\n        "chroma_limit_lab": 3,\n        "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n        "texture_gain_limits": [\n          1.0,\n          1.0\n        ],\n        "blur_max_roi64": 0.0,\n        "lighting": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        }\n      },\n      "limitations": [\n        "Within-person variation includes pose/content; not an isolated camera calibration",\n        "Texture and focus proxies are not independent physical measurements",\n        "Global disease colors/textures are not matched to unlabeled participants",\n        "Four participants are not a general camera population or independent Target Test",\n        "Source/Target brightness and contrast also contain disease and composition differences",\n        "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n        "Bounds are engineering choices, not validated physical camera calibration",\n        "This profile changes direction and calibration policy as well as amplitude",\n        "The existing Source Test has already been inspected; select on Validation only",\n        "Revised chroma direction remains confounded with disease, person and body region",\n        "Consistency across omissions is a sensitivity check, not independent validation",\n        "Focus gates are content-dependent proxies; no physical camera equivalence established"\n      ],\n      "previous_visual_review": {\n        "profile_version": "quarantined_within_person_bounded_v4",\n        "train_visual_review": "approved",\n        "review_note": "Inspected class-stratified retained Train previews and worst observed clipping examples. No gross lesion-boundary loss or artificial geometry observed at preview scale. All retained preview metrics and parameter corners were calculated. Some highlights approach bounds; this is a limited pilot approval, not image information or clinical accuracy validation. No face identity assessment performed. Source ROI missingness does not set brightness; within-person Target lighting differences are used. Whole Source/Target appearance matching is not claimed."\n      },\n      "calibration_version": "direction_and_focus_audited_v1",\n      "calibration_evidence": {\n        "source_role": "retained_original_train",\n        "target_role": "unlabeled_calibration",\n        "source_class_balanced_anchors": {\n          "y_p50": 83.40784072875977,\n          "contrast": 70.15270042419434\n        },\n        "source_measurement_coverage": {\n          "y_p50": {\n            "class_medians": [\n              96.70617294311523,\n              77.98558807373047,\n              79.60416030883789,\n              104.08260345458984,\n              73.82204055786133,\n              87.21152114868164,\n              109.7138786315918,\n              77.95131301879883,\n              55.02832794189453,\n              92.87034606933594\n            ],\n            "coverage_by_class": [\n              {\n                "total": 723,\n                "measured": 723\n              },\n              {\n                "total": 722,\n                "measured": 722\n              },\n              {\n                "total": 709,\n                "measured": 709\n              },\n              {\n                "total": 714,\n                "measured": 714\n              },\n              {\n                "total": 691,\n                "measured": 691\n              },\n              {\n                "total": 725,\n                "measured": 725\n              },\n              {\n                "total": 703,\n                "measured": 703\n              },\n              {\n                "total": 519,\n                "measured": 519\n              },\n              {\n                "total": 702,\n                "measured": 702\n              },\n              {\n                "total": 545,\n                "measured": 545\n              }\n            ]\n          },\n          "contrast": {\n            "class_medians": [\n              50.152610778808594,\n              86.36319732666016,\n              54.131675720214844,\n              55.54798126220703,\n              74.786376953125,\n              84.3202133178711,\n              74.34446716308594,\n              67.56494903564453,\n              72.74045181274414,\n              44.280799865722656\n            ],\n            "coverage_by_class": [\n              {\n                "total": 723,\n                "measured": 723\n              },\n              {\n                "total": 722,\n                "measured": 722\n              },\n              {\n                "total": 709,\n                "measured": 709\n              },\n              {\n                "total": 714,\n                "measured": 714\n              },\n              {\n                "total": 691,\n                "measured": 691\n              },\n              {\n                "total": 725,\n                "measured": 725\n              },\n              {\n                "total": 703,\n                "measured": 703\n              },\n              {\n                "total": 519,\n                "measured": 519\n              },\n              {\n                "total": 702,\n                "measured": 702\n              },\n              {\n                "total": 545,\n                "measured": 545\n              }\n            ]\n          }\n        },\n        "target_equal_participant_condition_medians": {\n          "y_p50": [\n            155.1225357055664,\n            157.67981719970703,\n            151.86930084228516,\n            153.47462463378906,\n            152.83219146728516,\n            155.95436096191406,\n            155.48592376708984,\n            156.50391387939453\n          ],\n          "contrast": [\n            23.243118286132812,\n            24.954025268554688,\n            21.043075561523438,\n            26.485572814941406,\n            23.47039794921875,\n            25.83660888671875,\n            19.079574584960938,\n            20.986839294433594\n          ]\n        },\n        "raw_brightness_offsets": [\n          69.90617561340332,\n          72.68390846252441\n        ],\n        "raw_contrast_ratios": [\n          0.29976061317090036,\n          0.35885534014900783\n        ],\n        "brightness_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ],\n        "hair_global_contrast": "identity; coverage and content remain confounded",\n        "revision_policy": "direction_and_focus_audited_v1"\n      },\n      "revision_evidence": {\n        "color_direction": {\n          "a": {\n            "source_after_lighting_class_balanced_anchor": 15.72265625,\n            "target_equal_group_medians": {\n              "P01/alt": 3.046875,\n              "P01/ref": 2.90625,\n              "P02/ref": 1.5703125,\n              "P02/alt": 2.234375,\n              "P03/alt": 1.828125,\n              "P03/ref": 2.3125,\n              "P04/alt": 1.640625,\n              "P04/ref": 3.359375\n            },\n            "full_direction": -1,\n            "leave_one_person_out_directions": {\n              "P01": -1,\n              "P02": -1,\n              "P03": -1,\n              "P04": -1\n            },\n            "direction_stable": true,\n            "within_person_absolute_q75_limit": 0.9609375,\n            "proposed_range": [\n              -0.9609375,\n              0.0\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          },\n          "b": {\n            "source_after_lighting_class_balanced_anchor": 11.990234375,\n            "target_equal_group_medians": {\n              "P01/alt": 30.046875,\n              "P01/ref": 31.453125,\n              "P02/ref": 27.828125,\n              "P02/alt": 28.59375,\n              "P03/alt": 31.0,\n              "P03/ref": 30.890625,\n              "P04/alt": 31.78125,\n              "P04/ref": 32.3203125\n            },\n            "full_direction": 1,\n            "leave_one_person_out_directions": {\n              "P01": 1,\n              "P02": 1,\n              "P03": 1,\n              "P04": 1\n            },\n            "direction_stable": true,\n            "within_person_absolute_q75_limit": 0.92578125,\n            "proposed_range": [\n              0.0,\n              0.92578125\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          }\n        },\n        "color_choice": "both",\n        "focus_choice": "none",\n        "source_preview_count": 223,\n        "target_count": 42,\n        "seeds": [\n          42,\n          707,\n          1707\n        ],\n        "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n      },\n      "previous_engineering_bounds": {\n        "measured_change_fraction": 1.0,\n        "chroma_limit_lab": 3,\n        "texture_gain_limits": [\n          0.85,\n          1.15\n        ],\n        "focus_fit_grid_roi64": [\n          0,\n          1.5,\n          0.05\n        ],\n        "blur_max_roi64": 0.75,\n        "brightness_offset_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ]\n      }\n    },\n    "web_skin": {\n      "schema": "measured_camera_extended_v1",\n      "domain": "web_skin",\n      "lighting": {\n        "domain": "web_skin",\n        "profile_version": "bounded_source_target_strength_v1",\n        "brightness_offset": [\n          -32.0,\n          -32.0\n        ],\n        "contrast": [\n          0.8,\n          1.0152967817078993\n        ],\n        "scale_x": [\n          1.0,\n          1.0\n        ],\n        "scale_y": [\n          1.0,\n          1.0\n        ],\n        "left_light": [\n          24.0,\n          24.0\n        ],\n        "evidence": {\n          "method": "Source class-balanced median to equal participant/condition Target quartiles",\n          "anchors": {\n            "y_p50": 167.1248779296875,\n            "contrast": 27.026893615722656\n          },\n          "coverage": {\n            "y_p50": {\n              "class_medians": [\n                164.75593948364258,\n                166.1425018310547,\n                167.1248779296875,\n                193.63400268554688,\n                168.03762435913086\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 714,\n                  "measured": 356\n                },\n                {\n                  "total": 712,\n                  "measured": 595\n                },\n                {\n                  "total": 687,\n                  "measured": 643\n                },\n                {\n                  "total": 720,\n                  "measured": 719\n                },\n                {\n                  "total": 702,\n                  "measured": 670\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                27.026893615722656,\n                24.09050750732422,\n                22.20507049560547,\n                28.38959503173828,\n                30.2370548248291\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 714,\n                  "measured": 356\n                },\n                {\n                  "total": 712,\n                  "measured": 595\n                },\n                {\n                  "total": 687,\n                  "measured": 643\n                },\n                {\n                  "total": 720,\n                  "measured": 719\n                },\n                {\n                  "total": 702,\n                  "measured": 670\n                }\n              ]\n            }\n          }\n        },\n        "bounds": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        },\n        "held_back": [\n          "color: unmatched disease color",\n          "blur/noise: texture/scale confounding",\n          "hair mask: not validated and could change class evidence",\n          "geometry: no rescaling onto artificial padding; composition is an input policy"\n        ],\n        "photometric_operator": "bounded_additive_v3",\n        "input_policy": "center_square",\n        "source_sha256": "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d",\n        "reference_model_sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4",\n        "calibration_fingerprint": "20dc5244d16d7e28a2eede6ad323a97a5db98fba2f5bb06420b0941dd1e481df",\n        "source_subset_sha256": "4377568a2b483f5b2efa38349f5469e658c74923c15dac924e1b3606ad4e0993",\n        "train_visual_review": "revised_train_replay_not_clinical_validation",\n        "review_note": "Same retained Train previews and actual calibration captures remeasured. Unlabeled descriptive audit; disease-feature preservation and accuracy unverified.",\n        "calibration_version": "direction_and_focus_audited_v1"\n      },\n      "color_a": [\n        0.0,\n        0.0\n      ],\n      "color_b": [\n        -1.9033203125,\n        0.0\n      ],\n      "texture_gain": [\n        0.85,\n        0.9145815899839365\n      ],\n      "blur_sigma_roi64": [\n        0.0,\n        0.4\n      ],\n      "blur_input_scale": 1.9375,\n      "evidence": {\n        "chroma": {\n          "lab_a": {\n            "participant_changes": [\n              -0.53125,\n              -0.1640625,\n              -2.2734375,\n              -0.97265625\n            ],\n            "range": [\n              0.0,\n              0.0\n            ]\n          },\n          "lab_b": {\n            "participant_changes": [\n              -2.703125,\n              1.63671875,\n              -0.265625,\n              0.32421875\n            ],\n            "range": [\n              -1.9033203125,\n              0.0\n            ]\n          }\n        },\n        "normalized_detail_ratios": [\n          0.9041781600546774,\n          0.7888216863779229,\n          0.8610762805143408,\n          0.9457918797717134\n        ],\n        "focus_fits": [\n          {\n            "person": "P01",\n            "condition": "alt",\n            "sigma_roi64": 0.35000000000000003,\n            "reference_path": "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150143660730.png",\n            "compared_paths": [\n              "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150146243557.png",\n              "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150145477935.png",\n              "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150143660730.png"\n            ],\n            "normalized_detail_reference": 0.04072413966059685,\n            "normalized_detail_target_q25": 0.03876550309360027,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P01",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150117359310.png",\n            "compared_paths": [\n              "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150117359310.png",\n              "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150121743228.png",\n              "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150120048313.png"\n            ],\n            "normalized_detail_reference": 0.042286016047000885,\n            "normalized_detail_target_q25": 0.03858809173107147,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "alt",\n            "sigma_roi64": 0.35000000000000003,\n            "reference_path": "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155549996145.png",\n            "compared_paths": [\n              "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155546795247.png",\n              "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155549996145.png",\n              "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155547726859.png"\n            ],\n            "normalized_detail_reference": 0.03815416991710663,\n            "normalized_detail_target_q25": 0.03712081536650658,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155523943430.png",\n            "compared_paths": [\n              "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155520318017.png",\n              "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155523943430.png",\n              "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155520566900.png"\n            ],\n            "normalized_detail_reference": 0.032488320022821426,\n            "normalized_detail_target_q25": 0.02975787501782179,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "alt",\n            "sigma_roi64": 0.4,\n            "reference_path": "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160213812778.png",\n            "compared_paths": [\n              "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160217426731.png",\n              "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160213812778.png",\n              "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160214841649.png"\n            ],\n            "normalized_detail_reference": 0.04655195027589798,\n            "normalized_detail_target_q25": 0.04139178246259689,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160141027206.png",\n            "compared_paths": [\n              "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160139114564.png",\n              "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160135334160.png",\n              "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160141027206.png"\n            ],\n            "normalized_detail_reference": 0.049063827842473984,\n            "normalized_detail_target_q25": 0.04606209881603718,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "alt",\n            "sigma_roi64": 0.30000000000000004,\n            "reference_path": "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161326825741.png",\n            "compared_paths": [\n              "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161326825741.png",\n              "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161323856383.png",\n              "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161325776274.png"\n            ],\n            "normalized_detail_reference": 0.045572228729724884,\n            "normalized_detail_target_q25": 0.04498937726020813,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161259907934.png",\n            "compared_paths": [\n              "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161258616795.png",\n              "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161259907934.png",\n              "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161317619759.png"\n            ],\n            "normalized_detail_reference": 0.058970071375370026,\n            "normalized_detail_target_q25": 0.05298882909119129,\n            "fit_at_search_limit": false\n          }\n        ],\n        "source_train_statistics": {\n          "width": {\n            "n": 3535,\n            "mean": 1024.0,\n            "var": 0.0,\n            "q05": 1024.0,\n            "q25": 1024.0,\n            "q50": 1024.0,\n            "q75": 1024.0,\n            "q95": 1024.0\n          },\n          "height": {\n            "n": 3535,\n            "mean": 1024.0,\n            "var": 0.0,\n            "q05": 1024.0,\n            "q25": 1024.0,\n            "q50": 1024.0,\n            "q75": 1024.0,\n            "q95": 1024.0\n          },\n          "aspect": {\n            "n": 3535,\n            "mean": 1.0,\n            "var": 0.0,\n            "q05": 1.0,\n            "q25": 1.0,\n            "q50": 1.0,\n            "q75": 1.0,\n            "q95": 1.0\n          },\n          "frame_y_mean": {\n            "n": 3535,\n            "mean": 122.4840412606613,\n            "var": 174.63957111668591,\n            "q05": 100.79979181289673,\n            "q25": 113.60491180419922,\n            "q50": 122.60625743865967,\n            "q75": 131.76815176010132,\n            "q95": 144.24287633895872\n          },\n          "face_w": {\n            "n": 2983,\n            "mean": 0.7038756730849816,\n            "var": 0.010719641525178694,\n            "q05": 0.4990234375,\n            "q25": 0.658203125,\n            "q50": 0.734375,\n            "q75": 0.77490234375,\n            "q95": 0.818359375\n          },\n          "face_h": {\n            "n": 2983,\n            "mean": 0.7038782920926919,\n            "var": 0.010721216919545137,\n            "q05": 0.4990234375,\n            "q25": 0.658203125,\n            "q50": 0.734375,\n            "q75": 0.775390625,\n            "q95": 0.818359375\n          },\n          "face_x": {\n            "n": 2983,\n            "mean": 0.4952846403054811,\n            "var": 0.0005910972975787344,\n            "q05": 0.46875,\n            "q25": 0.4873046875,\n            "q50": 0.49755859375,\n            "q75": 0.50537109375,\n            "q95": 0.51904296875\n          },\n          "face_y": {\n            "n": 2983,\n            "mean": 0.49806962762948376,\n            "var": 0.002680517827346623,\n            "q05": 0.41943359375,\n            "q25": 0.45947265625,\n            "q50": 0.5009765625,\n            "q75": 0.5302734375,\n            "q95": 0.57958984375\n          },\n          "y_mean": {\n            "n": 2983,\n            "mean": 171.60286215700003,\n            "var": 340.5626552189052,\n            "q05": 142.94990234375,\n            "q25": 160.08666229248047,\n            "q50": 170.08668518066406,\n            "q75": 182.89017486572266,\n            "q95": 204.19006652832033\n          },\n          "y_std": {\n            "n": 2983,\n            "mean": 13.824026793711065,\n            "var": 28.611544611011293,\n            "q05": 6.904114818572998,\n            "q25": 10.07047176361084,\n            "q50": 12.990439414978027,\n            "q75": 16.47877597808838,\n            "q95": 23.480698585510254\n          },\n          "y_p10": {\n            "n": 2983,\n            "mean": 153.8254084425647,\n            "var": 433.1489020992675,\n            "q05": 119.6391700744629,\n            "q25": 141.92668838500975,\n            "q50": 153.04942779541017,\n            "q75": 165.74359970092775,\n            "q95": 190.13076248168946\n          },\n          "y_p50": {\n            "n": 2983,\n            "mean": 172.59530727630727,\n            "var": 360.93161945805053,\n            "q05": 143.72868576049805,\n            "q25": 160.80518341064453,\n            "q50": 170.97125244140625,\n            "q75": 184.3146858215332,\n            "q95": 206.0211311340332\n          },\n          "y_p90": {\n            "n": 2983,\n            "mean": 188.0014141879464,\n            "var": 336.62722341229323,\n            "q05": 159.26213500976564,\n            "q25": 176.1097557067871,\n            "q50": 186.53913269042968,\n            "q75": 200.18789443969726,\n            "q95": 219.2217042541504\n          },\n          "contrast": {\n            "n": 2983,\n            "mean": 28.567295564248024,\n            "var": 145.80308910326926,\n            "q05": 14.091455078125,\n            "q25": 20.190940856933594,\n            "q50": 26.279129028320312,\n            "q75": 33.947147369384766,\n            "q95": 51.409949493408206\n          },\n          "very_bright": {\n            "n": 2983,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "very_dark": {\n            "n": 2983,\n            "mean": 2.257461191420293e-06,\n            "var": 1.5196662733752344e-08,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "detail": {\n            "n": 2983,\n            "mean": 0.49890150976165293,\n            "var": 0.09083875806631915,\n            "q05": 0.12404881790280342,\n            "q25": 0.262665294110775,\n            "q50": 0.44187019765377045,\n            "q75": 0.6757000982761383,\n            "q95": 1.082399696111679\n          },\n          "lab_a": {\n            "n": 2983,\n            "mean": 14.891709269192088,\n            "var": 23.005682807687716,\n            "q05": 7.0,\n            "q25": 12.0,\n            "q50": 14.671875,\n            "q75": 17.81640625,\n            "q95": 23.07734375\n          },\n          "lab_b": {\n            "n": 2983,\n            "mean": 20.73640473097553,\n            "var": 50.364169986750426,\n            "q05": 7.71875,\n            "q25": 16.07421875,\n            "q50": 21.6875,\n            "q75": 26.0,\n            "q95": 30.9671875\n          },\n          "lighting_range": {\n            "n": 2983,\n            "mean": 30.582603060715638,\n            "var": 227.63935227652632,\n            "q05": 11.45445556640625,\n            "q25": 19.742637634277344,\n            "q50": 27.928878784179688,\n            "q75": 38.418434143066406,\n            "q95": 58.480264282226564\n          },\n          "r_p10": {\n            "n": 2983,\n            "mean": 195.33571069393227,\n            "var": 558.134331727566,\n            "q05": 153.7725,\n            "q25": 182.25,\n            "q50": 196.0,\n            "q75": 209.25,\n            "q95": 235.46\n          },\n          "r_p50": {\n            "n": 2983,\n            "mean": 210.31352665102247,\n            "var": 460.8270383672918,\n            "q05": 173.275,\n            "q25": 197.0625,\n            "q50": 210.0,\n            "q75": 223.8125,\n            "q95": 246.875\n          },\n          "r_p90": {\n            "n": 2983,\n            "mean": 221.05195273214886,\n            "var": 391.97067943356865,\n            "q05": 187.0,\n            "q25": 208.25,\n            "q50": 221.0,\n            "q75": 235.5,\n            "q95": 252.5\n          },\n          "g_p10": {\n            "n": 2983,\n            "mean": 140.27132919879315,\n            "var": 466.92841134922116,\n            "q05": 104.975,\n            "q25": 127.25,\n            "q50": 139.75,\n            "q75": 153.6875,\n            "q95": 177.5225\n          },\n          "g_p50": {\n            "n": 2983,\n            "mean": 160.78247569560844,\n            "var": 389.1172563835024,\n            "q05": 131.0,\n            "q25": 148.0,\n            "q50": 159.25,\n            "q75": 173.1875,\n            "q95": 195.25\n          },\n          "g_p90": {\n            "n": 2983,\n            "mean": 178.013794837412,\n            "var": 373.13305894818654,\n            "q05": 148.6825,\n            "q25": 165.5,\n            "q50": 176.70000000000002,\n            "q75": 191.0,\n            "q95": 211.25\n          },\n          "b_p10": {\n            "n": 2983,\n            "mean": 112.22356687898088,\n            "var": 468.43108514864485,\n            "q05": 78.055,\n            "q25": 98.0,\n            "q50": 111.125,\n            "q75": 126.0,\n            "q95": 149.09\n          },\n          "b_p50": {\n            "n": 2983,\n            "mean": 134.49773717733825,\n            "var": 425.0373524056142,\n            "q05": 102.0,\n            "q25": 120.75,\n            "q50": 133.75,\n            "q75": 148.6875,\n            "q95": 168.625\n          },\n          "b_p90": {\n            "n": 2983,\n            "mean": 155.19481226952732,\n            "var": 436.5251266660647,\n            "q05": 120.75,\n            "q25": 141.7,\n            "q50": 155.575,\n            "q75": 168.8,\n            "q95": 189.5\n          },\n          "side_light": {\n            "n": 2983,\n            "mean": -2.789991009846814,\n            "var": 216.62861919520685,\n            "q05": -27.705178833007814,\n            "q25": -11.624557495117188,\n            "q50": -2.094879150390625,\n            "q75": 7.0645599365234375,\n            "q95": 19.366043090820312\n          }\n        },\n        "target_statistics": {\n          "width": {\n            "n": 42,\n            "mean": 720.0,\n            "var": 0.0,\n            "q05": 720.0,\n            "q25": 720.0,\n            "q50": 720.0,\n            "q75": 720.0,\n            "q95": 720.0\n          },\n          "height": {\n            "n": 42,\n            "mean": 720.0,\n            "var": 0.0,\n            "q05": 720.0,\n            "q25": 720.0,\n            "q50": 720.0,\n            "q75": 720.0,\n            "q95": 720.0\n          },\n          "aspect": {\n            "n": 42,\n            "mean": 1.0,\n            "var": 0.0,\n            "q05": 1.0,\n            "q25": 1.0,\n            "q50": 1.0,\n            "q75": 1.0,\n            "q95": 1.0\n          },\n          "frame_y_mean": {\n            "n": 42,\n            "mean": 117.96577284501763,\n            "var": 168.44828486838938,\n            "q05": 97.55988040123458,\n            "q25": 106.15853973765432,\n            "q50": 121.94068769290124,\n            "q75": 125.36228298611111,\n            "q95": 141.28436178626544\n          },\n          "native_width": {\n            "n": 42,\n            "mean": 1280.0,\n            "var": 0.0,\n            "q05": 1280.0,\n            "q25": 1280.0,\n            "q50": 1280.0,\n            "q75": 1280.0,\n            "q95": 1280.0\n          },\n          "native_height": {\n            "n": 42,\n            "mean": 720.0,\n            "var": 0.0,\n            "q05": 720.0,\n            "q25": 720.0,\n            "q50": 720.0,\n            "q75": 720.0,\n            "q95": 720.0\n          },\n          "face_w": {\n            "n": 42,\n            "mean": 0.47837301587301584,\n            "var": 0.002169876438229613,\n            "q05": 0.39458333333333334,\n            "q25": 0.4472222222222222,\n            "q50": 0.48333333333333334,\n            "q75": 0.5083333333333333,\n            "q95": 0.5541666666666666\n          },\n          "face_h": {\n            "n": 42,\n            "mean": 0.47837301587301584,\n            "var": 0.002169876438229613,\n            "q05": 0.39458333333333334,\n            "q25": 0.4472222222222222,\n            "q50": 0.48333333333333334,\n            "q75": 0.5083333333333333,\n            "q95": 0.5541666666666666\n          },\n          "face_x": {\n            "n": 42,\n            "mean": 0.5650462962962963,\n            "var": 0.003975043785518322,\n            "q05": 0.4625694444444445,\n            "q25": 0.5145833333333333,\n            "q50": 0.5916666666666667,\n            "q75": 0.6246527777777777,\n            "q95": 0.6442361111111111\n          },\n          "face_y": {\n            "n": 42,\n            "mean": 0.5393849206349207,\n            "var": 0.005078981245275889,\n            "q05": 0.4222222222222222,\n            "q25": 0.46215277777777775,\n            "q50": 0.5625,\n            "q75": 0.5934027777777777,\n            "q95": 0.6263194444444444\n          },\n          "y_mean": {\n            "n": 42,\n            "mean": 108.4215832664853,\n            "var": 471.73526077400675,\n            "q05": 81.75565910339355,\n            "q25": 97.35028457641602,\n            "q50": 101.1187858581543,\n            "q75": 124.03936767578125,\n            "q95": 144.38405075073243\n          },\n          "y_std": {\n            "n": 42,\n            "mean": 12.71238576798212,\n            "var": 27.17134695743536,\n            "q05": 6.027664399147033,\n            "q25": 7.170705556869507,\n            "q50": 12.186700344085693,\n            "q75": 17.962547779083252,\n            "q95": 19.73787727355957\n          },\n          "y_p10": {\n            "n": 42,\n            "mean": 92.96419746762231,\n            "var": 309.3590120859504,\n            "q05": 73.76079399108887,\n            "q25": 76.78439102172851,\n            "q50": 89.62044029235841,\n            "q75": 106.92601470947267,\n            "q95": 123.44529014587403\n          },\n          "y_p50": {\n            "n": 42,\n            "mean": 106.60154070172992,\n            "var": 415.21468403277703,\n            "q05": 81.44257717132568,\n            "q25": 95.33713245391846,\n            "q50": 100.76734733581543,\n            "q75": 115.55965232849121,\n            "q95": 139.64438972473144\n          },\n          "y_p90": {\n            "n": 42,\n            "mean": 126.24842383975076,\n            "var": 824.4180391542692,\n            "q05": 88.9665768814087,\n            "q25": 109.57465324401855,\n            "q50": 118.08759078979493,\n            "q75": 155.39628486633302,\n            "q95": 172.2517123413086\n          },\n          "contrast": {\n            "n": 42,\n            "mean": 23.63250796000163,\n            "var": 31.77594674984333,\n            "q05": 15.582783126831055,\n            "q25": 18.345415115356445,\n            "q50": 24.028310775756836,\n            "q75": 27.55977439880371,\n            "q95": 32.18445816040039\n          },\n          "very_bright": {\n            "n": 42,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "very_dark": {\n            "n": 42,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "detail": {\n            "n": 42,\n            "mean": 0.10692421950045086,\n            "var": 0.00038114416721783337,\n            "q05": 0.07504560090601445,\n            "q25": 0.09076076932251453,\n            "q50": 0.10582415945827961,\n            "q75": 0.12221345119178295,\n            "q95": 0.13948970828205345\n          },\n          "lab_a": {\n            "n": 42,\n            "mean": 15.566778273809524,\n            "var": 1.9094418549483594,\n            "q05": 13.469140625,\n            "q25": 14.6953125,\n            "q50": 15.45703125,\n            "q75": 16.59375,\n            "q95": 17.8703125\n          },\n          "lab_b": {\n            "n": 42,\n            "mean": 13.899925595238095,\n            "var": 13.184434402016015,\n            "q05": 8.048046875,\n            "q25": 10.611328125,\n            "q50": 13.7265625,\n            "q75": 17.736328125,\n            "q95": 19.006249999999998\n          },\n          "lighting_range": {\n            "n": 42,\n            "mean": 31.957427796863374,\n            "var": 247.98203200329277,\n            "q05": 12.51398811340332,\n            "q25": 17.754268646240234,\n            "q50": 29.24612808227539,\n            "q75": 45.80515670776367,\n            "q95": 54.02673950195312\n          },\n          "r_p10": {\n            "n": 42,\n            "mean": 124.67250046502977,\n            "var": 344.8074200952891,\n            "q05": 103.9420166015625,\n            "q25": 106.6775146484375,\n            "q50": 121.81464843750001,\n            "q75": 137.7626953125,\n            "q95": 155.14917480468748\n          },\n          "r_p50": {\n            "n": 42,\n            "mean": 138.02106584821428,\n            "var": 424.9906201200421,\n            "q05": 112.015625,\n            "q25": 126.3297119140625,\n            "q50": 131.5458984375,\n            "q75": 147.0126953125,\n            "q95": 171.9563720703125\n          },\n          "r_p90": {\n            "n": 42,\n            "mean": 155.45787527901786,\n            "var": 878.8976339353021,\n            "q05": 118.23587890625,\n            "q25": 137.4837890625,\n            "q50": 147.3919921875,\n            "q75": 181.68759765625,\n            "q95": 204.9484033203125\n          },\n          "g_p10": {\n            "n": 42,\n            "mean": 81.36427408854168,\n            "var": 282.89784631271215,\n            "q05": 62.83041015625,\n            "q25": 67.665625,\n            "q50": 76.99370117187499,\n            "q75": 95.308740234375,\n            "q95": 110.8648828125\n          },\n          "g_p50": {\n            "n": 42,\n            "mean": 95.46513439360119,\n            "var": 395.1578419555882,\n            "q05": 71.2409912109375,\n            "q25": 85.3331298828125,\n            "q50": 88.64208984375,\n            "q75": 103.992431640625,\n            "q95": 127.4594970703125\n          },\n          "g_p90": {\n            "n": 42,\n            "mean": 115.66137230282737,\n            "var": 778.3190722293864,\n            "q05": 79.87362304687501,\n            "q25": 97.6484375,\n            "q50": 108.1435546875,\n            "q75": 145.5100341796875,\n            "q95": 159.31440429687498\n          },\n          "b_p10": {\n            "n": 42,\n            "mean": 68.77881556919644,\n            "var": 385.041113648366,\n            "q05": 46.137138671875,\n            "q25": 47.9603515625,\n            "q50": 70.10048828125,\n            "q75": 81.8400390625,\n            "q95": 101.8196240234375\n          },\n          "b_p50": {\n            "n": 42,\n            "mean": 82.75233677455357,\n            "var": 532.924830329256,\n            "q05": 54.620068359375,\n            "q25": 63.442626953125,\n            "q50": 82.416259765625,\n            "q75": 92.9210205078125,\n            "q95": 120.99169921875\n          },\n          "b_p90": {\n            "n": 42,\n            "mean": 104.50118815104166,\n            "var": 955.9511582215247,\n            "q05": 63.3044384765625,\n            "q25": 88.7904296875,\n            "q50": 97.46328125,\n            "q75": 134.455322265625,\n            "q95": 153.45194824218748\n          },\n          "side_light": {\n            "n": 42,\n            "mean": 13.002259753999256,\n            "var": 216.15893591643612,\n            "q05": -6.440453720092774,\n            "q25": 0.09168052673339844,\n            "q50": 10.742233276367188,\n            "q75": 27.831308364868164,\n            "q95": 30.24348678588867\n          }\n        }\n      },\n      "engineering_bounds": {\n        "chroma_limit_lab": 3,\n        "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n        "texture_gain_limits": [\n          0.85,\n          0.9145815899839365\n        ],\n        "blur_max_roi64": 0.4,\n        "lighting": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        }\n      },\n      "limitations": [\n        "Within-person variation includes pose/content; not an isolated camera calibration",\n        "Texture and focus proxies are not independent physical measurements",\n        "Global disease colors/textures are not matched to unlabeled participants",\n        "Four participants are not a general camera population or independent Target Test",\n        "Source/Target brightness and contrast also contain disease and composition differences",\n        "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n        "Bounds are engineering choices, not validated physical camera calibration",\n        "This profile changes direction and calibration policy as well as amplitude",\n        "The existing Source Test has already been inspected; select on Validation only",\n        "Revised chroma direction remains confounded with disease, person and body region",\n        "Consistency across omissions is a sensitivity check, not independent validation",\n        "Focus gates are content-dependent proxies; no physical camera equivalence established"\n      ],\n      "previous_visual_review": {\n        "profile_version": "quarantined_within_person_bounded_v4",\n        "train_visual_review": "approved",\n        "review_note": "Inspected class-stratified retained Train previews and worst observed clipping examples. No gross lesion-boundary loss or artificial geometry observed at preview scale. All retained preview metrics and parameter corners were calculated. Some highlights approach bounds; this is a limited pilot approval, not image information or clinical accuracy validation. No face identity assessment performed. Source ROI missingness does not set brightness; within-person Target lighting differences are used. Whole Source/Target appearance matching is not claimed."\n      },\n      "calibration_version": "direction_and_focus_audited_v1",\n      "calibration_evidence": {\n        "source_role": "retained_original_train",\n        "target_role": "unlabeled_calibration",\n        "source_class_balanced_anchors": {\n          "y_p50": 167.1248779296875,\n          "contrast": 27.026893615722656\n        },\n        "source_measurement_coverage": {\n          "y_p50": {\n            "class_medians": [\n              164.75593948364258,\n              166.1425018310547,\n              167.1248779296875,\n              193.63400268554688,\n              168.03762435913086\n            ],\n            "coverage_by_class": [\n              {\n                "total": 714,\n                "measured": 356\n              },\n              {\n                "total": 712,\n                "measured": 595\n              },\n              {\n                "total": 687,\n                "measured": 643\n              },\n              {\n                "total": 720,\n                "measured": 719\n              },\n              {\n                "total": 702,\n                "measured": 670\n              }\n            ]\n          },\n          "contrast": {\n            "class_medians": [\n              27.026893615722656,\n              24.09050750732422,\n              22.20507049560547,\n              28.38959503173828,\n              30.2370548248291\n            ],\n            "coverage_by_class": [\n              {\n                "total": 714,\n                "measured": 356\n              },\n              {\n                "total": 712,\n                "measured": 595\n              },\n              {\n                "total": 687,\n                "measured": 643\n              },\n              {\n                "total": 720,\n                "measured": 719\n              },\n              {\n                "total": 702,\n                "measured": 670\n              }\n            ]\n          }\n        },\n        "target_equal_participant_condition_medians": {\n          "y_p50": [\n            114.7862548828125,\n            82.22571182250977,\n            139.51790618896484,\n            100.60310363769531,\n            136.71737670898438,\n            102.7280044555664,\n            95.67724990844727,\n            85.4756031036377\n          ],\n          "contrast": [\n            26.097904205322266,\n            16.046756744384766,\n            31.983932495117188,\n            21.72176170349121,\n            31.467559814453125,\n            24.71625518798828,\n            24.18914794921875,\n            15.956871032714844\n          ]\n        },\n        "raw_brightness_offsets": [\n          -73.99803972244263,\n          -46.85584259033203\n        ],\n        "raw_contrast_ratios": [\n          0.7512150953191121,\n          1.0152967817078993\n        ],\n        "brightness_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ],\n        "hair_global_contrast": "identity; coverage and content remain confounded",\n        "revision_policy": "direction_and_focus_audited_v1"\n      },\n      "revision_evidence": {\n        "color_direction": {\n          "a": {\n            "source_after_lighting_class_balanced_anchor": 16.5234375,\n            "target_equal_group_medians": {\n              "P01/alt": 15.0234375,\n              "P01/ref": 15.40625,\n              "P02/alt": 16.546875,\n              "P02/ref": 16.80859375,\n              "P03/alt": 15.6796875,\n              "P03/ref": 17.90625,\n              "P04/alt": 13.53125,\n              "P04/ref": 14.421875\n            },\n            "full_direction": 0,\n            "leave_one_person_out_directions": {\n              "P01": 0,\n              "P02": -1,\n              "P03": -1,\n              "P04": 0\n            },\n            "direction_stable": false,\n            "within_person_absolute_q75_limit": 1.2978515625,\n            "proposed_range": [\n              0.0,\n              0.0\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          },\n          "b": {\n            "source_after_lighting_class_balanced_anchor": 20.734375,\n            "target_equal_group_medians": {\n              "P01/alt": 12.8984375,\n              "P01/ref": 15.53125,\n              "P02/alt": 9.8515625,\n              "P02/ref": 8.171875,\n              "P03/alt": 13.625,\n              "P03/ref": 13.890625,\n              "P04/alt": 18.84375,\n              "P04/ref": 18.484375\n            },\n            "full_direction": -1,\n            "leave_one_person_out_directions": {\n              "P01": -1,\n              "P02": -1,\n              "P03": -1,\n              "P04": -1\n            },\n            "direction_stable": true,\n            "within_person_absolute_q75_limit": 1.9033203125,\n            "proposed_range": [\n              -1.9033203125,\n              0.0\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          }\n        },\n        "color_choice": "both",\n        "focus_choice": "both",\n        "source_preview_count": 118,\n        "target_count": 42,\n        "seeds": [\n          42,\n          707,\n          1707\n        ],\n        "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n      },\n      "previous_engineering_bounds": {\n        "measured_change_fraction": 1.0,\n        "chroma_limit_lab": 3,\n        "texture_gain_limits": [\n          0.85,\n          1.15\n        ],\n        "focus_fit_grid_roi64": [\n          0,\n          1.5,\n          0.05\n        ],\n        "blur_max_roi64": 0.75,\n        "brightness_offset_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ]\n      }\n    },\n    "hair": {\n      "schema": "measured_camera_extended_v1",\n      "domain": "hair",\n      "lighting": {\n        "domain": "hair",\n        "profile_version": "bounded_source_target_strength_v1",\n        "brightness_offset": [\n          15.043113708496094,\n          26.26080894470215\n        ],\n        "contrast": [\n          1.0,\n          1.0\n        ],\n        "scale_x": [\n          1.0,\n          1.0\n        ],\n        "scale_y": [\n          1.0,\n          1.0\n        ],\n        "left_light": [\n          0.0,\n          0.0\n        ],\n        "evidence": {\n          "method": "Source class-balanced median to equal participant/condition Target quartiles",\n          "anchors": {\n            "y_p50": 160.440185546875,\n            "contrast": 70.79170608520508\n          },\n          "coverage": {\n            "y_p50": {\n              "class_medians": [\n                150.06185150146484,\n                160.9347801208496,\n                160.440185546875,\n                157.4539794921875,\n                165.97833251953125\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 1990,\n                  "measured": 1990\n                },\n                {\n                  "total": 2050,\n                  "measured": 2050\n                },\n                {\n                  "total": 1929,\n                  "measured": 1929\n                },\n                {\n                  "total": 1965,\n                  "measured": 1965\n                },\n                {\n                  "total": 2098,\n                  "measured": 2098\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                58.59663391113281,\n                70.79170608520508,\n                83.7009048461914,\n                35.322113037109375,\n                71.07597732543945\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 1990,\n                  "measured": 1990\n                },\n                {\n                  "total": 2050,\n                  "measured": 2050\n                },\n                {\n                  "total": 1929,\n                  "measured": 1929\n                },\n                {\n                  "total": 1965,\n                  "measured": 1965\n                },\n                {\n                  "total": 2098,\n                  "measured": 2098\n                }\n              ]\n            }\n          }\n        },\n        "bounds": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        },\n        "held_back": [\n          "color: unmatched disease color",\n          "blur/noise: texture/scale confounding",\n          "hair mask: not validated and could change class evidence",\n          "geometry: no rescaling onto artificial padding; composition is an input policy",\n          "contrast/highlights: hair coverage and scalp visibility changed"\n        ],\n        "source_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156",\n        "model_sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2",\n        "calibration_fingerprint": "2d2ab5106e7f079147d4cd1e2e6e0ce1c27e8494d554085b4ad4947e3cd2b22c",\n        "input_policy": "legacy_resize",\n        "train_visual_review": "revised_train_replay_not_clinical_validation",\n        "selected_from": "verified_train_and_target_calibration",\n        "photometric_operator": "bounded_additive_v3",\n        "review_note": "Same retained Train previews and actual calibration captures remeasured. Unlabeled descriptive audit; disease-feature preservation and accuracy unverified.",\n        "calibration_version": "direction_and_focus_audited_v1"\n      },\n      "color_a": [\n        0.0,\n        0.0\n      ],\n      "color_b": [\n        0.0,\n        0.0\n      ],\n      "texture_gain": [\n        1.0,\n        1.0\n      ],\n      "blur_sigma_roi64": [\n        0.0,\n        0.0\n      ],\n      "blur_input_scale": 3.0,\n      "evidence": {\n        "chroma": {\n          "lab_a": {\n            "participant_changes": [\n              -1.265625,\n              0.59375,\n              -0.8125,\n              -1.140625\n            ],\n            "range": [\n              0.0,\n              0.0\n            ]\n          },\n          "lab_b": {\n            "participant_changes": [\n              -0.640625,\n              -2.21875,\n              0.390625,\n              -0.359375\n            ],\n            "range": [\n              0.0,\n              0.0\n            ]\n          }\n        },\n        "normalized_detail_ratios": [\n          0.7956155523836431,\n          0.903396231148771,\n          0.8178386205760373,\n          1.00576106234326\n        ],\n        "focus_fits": [\n          {\n            "person": "P01",\n            "condition": "alt",\n            "sigma_roi64": 0.5,\n            "reference_path": "hair/P01/S01/P01_S01_hair_alt_20261108T153239675963.png",\n            "compared_paths": [\n              "hair/P01/S01/P01_S01_hair_alt_20261108T153159444214.png",\n              "hair/P01/S01/P01_S01_hair_alt_20261108T153239675963.png",\n              "hair/P01/S01/P01_S01_hair_alt_20261108T153151051139.png"\n            ],\n            "normalized_detail_reference": 0.06333822011947632,\n            "normalized_detail_target_q25": 0.048252301290631294,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P01",\n            "condition": "ref",\n            "sigma_roi64": 0.7000000000000001,\n            "reference_path": "hair/P01/S01/P01_S01_hair_ref_20261108T153045788746.png",\n            "compared_paths": [\n              "hair/P01/S01/P01_S01_hair_ref_20261108T153045788746.png",\n              "hair/P01/S01/P01_S01_hair_ref_20261108T153121869582.png",\n              "hair/P01/S01/P01_S01_hair_ref_20261108T153051029359.png"\n            ],\n            "normalized_detail_reference": 0.06347747892141342,\n            "normalized_detail_target_q25": 0.04593171179294586,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "alt",\n            "sigma_roi64": 0.5,\n            "reference_path": "hair/P02/S01/P02_S01_hair_alt_20261108T155856007554.png",\n            "compared_paths": [\n              "hair/P02/S01/P02_S01_hair_alt_20261108T155902899143.png",\n              "hair/P02/S01/P02_S01_hair_alt_20261108T155856007554.png",\n              "hair/P02/S01/P02_S01_hair_alt_20261108T155847429457.png"\n            ],\n            "normalized_detail_reference": 0.0768783763051033,\n            "normalized_detail_target_q25": 0.06241852976381779,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P02",\n            "condition": "ref",\n            "sigma_roi64": 0.4,\n            "reference_path": "hair/P02/S01/P02_S01_hair_ref_20261108T155823882222.png",\n            "compared_paths": [\n              "hair/P02/S01/P02_S01_hair_ref_20261108T155823882222.png",\n              "hair/P02/S01/P02_S01_hair_ref_20261108T155807961945.png",\n              "hair/P02/S01/P02_S01_hair_ref_20261108T155819402127.png"\n            ],\n            "normalized_detail_reference": 0.07383783161640167,\n            "normalized_detail_target_q25": 0.06689455173909664,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "alt",\n            "sigma_roi64": 0.65,\n            "reference_path": "hair/P03/S01/P03_S01_hair_alt_20261108T160524791795.png",\n            "compared_paths": [\n              "hair/P03/S01/P03_S01_hair_alt_20261108T160524791795.png",\n              "hair/P03/S01/P03_S01_hair_alt_20261108T160528077321.png",\n              "hair/P03/S01/P03_S01_hair_alt_20261108T160534515213.png"\n            ],\n            "normalized_detail_reference": 0.04810452461242676,\n            "normalized_detail_target_q25": 0.034384097903966904,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P03",\n            "condition": "ref",\n            "sigma_roi64": 0.8,\n            "reference_path": "hair/P03/S01/P03_S01_hair_ref_20261108T160432797868.png",\n            "compared_paths": [\n              "hair/P03/S01/P03_S01_hair_ref_20261108T160447415708.png",\n              "hair/P03/S01/P03_S01_hair_ref_20261108T160414107183.png",\n              "hair/P03/S01/P03_S01_hair_ref_20261108T160432797868.png"\n            ],\n            "normalized_detail_reference": 0.08795812726020813,\n            "normalized_detail_target_q25": 0.05506112985312939,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "alt",\n            "sigma_roi64": 0.4,\n            "reference_path": "hair/P04/S01/P04_S01_hair_alt_20261108T161821606621.png",\n            "compared_paths": [\n              "hair/P04/S01/P04_S01_hair_alt_20261108T161818945430.png",\n              "hair/P04/S01/P04_S01_hair_alt_20261108T161821606621.png",\n              "hair/P04/S01/P04_S01_hair_alt_20261108T161755328976.png"\n            ],\n            "normalized_detail_reference": 0.0599982924759388,\n            "normalized_detail_target_q25": 0.05631589889526367,\n            "fit_at_search_limit": false\n          },\n          {\n            "person": "P04",\n            "condition": "ref",\n            "sigma_roi64": 0.75,\n            "reference_path": "hair/P04/S01/P04_S01_hair_ref_20261108T161741046903.png",\n            "compared_paths": [\n              "hair/P04/S01/P04_S01_hair_ref_20261108T161656311169.png",\n              "hair/P04/S01/P04_S01_hair_ref_20261108T161717832940.png",\n              "hair/P04/S01/P04_S01_hair_ref_20261108T161741046903.png"\n            ],\n            "normalized_detail_reference": 0.06583444774150848,\n            "normalized_detail_target_q25": 0.04240403510630131,\n            "fit_at_search_limit": false\n          }\n        ],\n        "source_train_statistics": {\n          "width": {\n            "n": 10032,\n            "mean": 640.0,\n            "var": 0.0,\n            "q05": 640.0,\n            "q25": 640.0,\n            "q50": 640.0,\n            "q75": 640.0,\n            "q95": 640.0\n          },\n          "height": {\n            "n": 10032,\n            "mean": 480.0,\n            "var": 0.0,\n            "q05": 480.0,\n            "q25": 480.0,\n            "q50": 480.0,\n            "q75": 480.0,\n            "q95": 480.0\n          },\n          "aspect": {\n            "n": 10032,\n            "mean": 1.3333333333333333,\n            "var": 0.0,\n            "q05": 1.3333333333333333,\n            "q25": 1.3333333333333333,\n            "q50": 1.3333333333333333,\n            "q75": 1.3333333333333333,\n            "q95": 1.3333333333333333\n          },\n          "frame_y_mean": {\n            "n": 10032,\n            "mean": 151.29554827449297,\n            "var": 45.89208796768109,\n            "q05": 140.40998828125,\n            "q25": 146.89392415364583,\n            "q50": 151.2693359375,\n            "q75": 155.68573893229166,\n            "q95": 162.4778154296875\n          },\n          "y_mean": {\n            "n": 10032,\n            "mean": 155.6826136944967,\n            "var": 130.13233446255578,\n            "q05": 136.8667984008789,\n            "q25": 147.92795181274414,\n            "q50": 155.83558654785156,\n            "q75": 163.58118057250977,\n            "q95": 174.12792510986327\n          },\n          "y_std": {\n            "n": 10032,\n            "mean": 29.953676802880075,\n            "var": 73.51660025014611,\n            "q05": 15.744802856445313,\n            "q25": 23.93495750427246,\n            "q50": 30.09425640106201,\n            "q75": 35.900136947631836,\n            "q95": 44.008992195129395\n          },\n          "y_p10": {\n            "n": 10032,\n            "mean": 118.73763829463977,\n            "var": 560.5050425503916,\n            "q05": 73.80268222808837,\n            "q25": 102.88904209136963,\n            "q50": 123.861079788208,\n            "q75": 137.00472869873047,\n            "q95": 150.05238952636716\n          },\n          "y_p50": {\n            "n": 10032,\n            "mean": 158.98320664249158,\n            "var": 145.1415514473159,\n            "q05": 139.57371711730957,\n            "q25": 150.6834716796875,\n            "q50": 158.9146385192871,\n            "q75": 167.0275421142578,\n            "q95": 179.19177665710447\n          },\n          "y_p90": {\n            "n": 10032,\n            "mean": 186.38956678153227,\n            "var": 287.0178060840724,\n            "q05": 160.2778648376465,\n            "q25": 173.7174224853516,\n            "q50": 185.64989242553713,\n            "q75": 197.79513931274414,\n            "q95": 215.30248764038083\n          },\n          "contrast": {\n            "n": 10032,\n            "mean": 67.65192802785116,\n            "var": 869.3538685090921,\n            "q05": 26.21061782836914,\n            "q25": 44.13491439819336,\n            "q50": 63.77436447143555,\n            "q75": 88.79049301147461,\n            "q95": 120.14251708984374\n          },\n          "very_bright": {\n            "n": 10032,\n            "mean": 5.6784435639287606e-08,\n            "var": 1.537151928674455e-12,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "very_dark": {\n            "n": 10032,\n            "mean": 0.0,\n            "var": 0.0,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.0,\n            "q75": 0.0,\n            "q95": 0.0\n          },\n          "detail": {\n            "n": 10032,\n            "mean": 7.9486075215980385,\n            "var": 6.836524631497128,\n            "q05": 3.89287109375,\n            "q25": 6.066689610481262,\n            "q50": 7.8241846561431885,\n            "q75": 9.658596277236938,\n            "q95": 12.415809583663938\n          },\n          "lab_a": {\n            "n": 10032,\n            "mean": 1.548149048046252,\n            "var": 93.81593158000446,\n            "q05": -11.36640625,\n            "q25": -5.875,\n            "q50": -0.546875,\n            "q75": 8.275390625,\n            "q95": 19.3125\n          },\n          "lab_b": {\n            "n": 10032,\n            "mean": 6.404843563098086,\n            "var": 47.66745941111487,\n            "q05": -4.296875,\n            "q25": 1.71875,\n            "q50": 6.078125,\n            "q75": 10.546875,\n            "q95": 18.710156249999983\n          },\n          "lighting_range": {\n            "n": 10032,\n            "mean": 20.021714983184182,\n            "var": 145.52417187959475,\n            "q05": 5.645606994628906,\n            "q25": 11.416168212890625,\n            "q50": 17.636642456054688,\n            "q75": 25.657127380371094,\n            "q95": 42.85272674560545\n          },\n          "r_p10": {\n            "n": 10032,\n            "mean": 122.83078115849592,\n            "var": 843.9024052340699,\n            "q05": 72.25,\n            "q25": 102.3291681289673,\n            "q50": 125.08958549499512,\n            "q75": 144.0,\n            "q95": 167.6666717529297\n          },\n          "r_p50": {\n            "n": 10032,\n            "mean": 167.46787159979058,\n            "var": 243.81335583496457,\n            "q05": 143.625,\n            "q25": 156.58333587646484,\n            "q50": 166.5833282470703,\n            "q75": 177.5416660308838,\n            "q95": 194.83333511352538\n          },\n          "r_p90": {\n            "n": 10032,\n            "mean": 189.73281665053665,\n            "var": 229.85646193840952,\n            "q05": 165.92083053588868,\n            "q25": 179.375,\n            "q50": 188.9104209899903,\n            "q75": 199.5833282470703,\n            "q95": 215.875\n          },\n          "g_p10": {\n            "n": 10032,\n            "mean": 117.43019380143575,\n            "var": 560.2681108291806,\n            "q05": 73.98125114440919,\n            "q25": 101.54166412353516,\n            "q50": 120.58958549499512,\n            "q75": 135.05416870117188,\n            "q95": 151.6854171752929\n          },\n          "g_p50": {\n            "n": 10032,\n            "mean": 157.09712873729603,\n            "var": 315.9570630042419,\n            "q05": 127.93958282470703,\n            "q25": 144.0416717529297,\n            "q50": 157.4166717529297,\n            "q75": 170.0416660308838,\n            "q95": 185.9166717529297\n          },\n          "g_p90": {\n            "n": 10032,\n            "mean": 187.1766518092422,\n            "var": 489.9436178101222,\n            "q05": 151.18270950317392,\n            "q25": 170.3333282470703,\n            "q50": 187.75,\n            "q75": 203.21562080383305,\n            "q95": 223.20145492553718\n          },\n          "b_p10": {\n            "n": 10032,\n            "mean": 104.9410540757567,\n            "var": 536.7216484610998,\n            "q05": 64.375,\n            "q25": 89.625,\n            "q50": 106.29166412353516,\n            "q75": 121.375,\n            "q95": 140.7916717529297\n          },\n          "b_p50": {\n            "n": 10032,\n            "mean": 147.03057508643545,\n            "var": 382.9799729408593,\n            "q05": 115.95833244323731,\n            "q25": 133.25,\n            "q50": 146.625,\n            "q75": 160.42187881469727,\n            "q95": 180.0\n          },\n          "b_p90": {\n            "n": 10032,\n            "mean": 182.3092101091784,\n            "var": 615.5828331807069,\n            "q05": 141.0,\n            "q25": 164.66667060852052,\n            "q50": 182.8479209899903,\n            "q75": 200.375,\n            "q95": 222.91250457763664\n          },\n          "dark_ridge": {\n            "n": 10032,\n            "mean": 0.12943713039281243,\n            "var": 0.0035930118071247218,\n            "q05": 0.036717393663194445,\n            "q25": 0.0853610568576389,\n            "q50": 0.125244140625,\n            "q75": 0.16943359375,\n            "q95": 0.23448350694444445\n          }\n        },\n        "target_statistics": {\n          "width": {\n            "n": 40,\n            "mean": 640.0,\n            "var": 0.0,\n            "q05": 640.0,\n            "q25": 640.0,\n            "q50": 640.0,\n            "q75": 640.0,\n            "q95": 640.0\n          },\n          "height": {\n            "n": 40,\n            "mean": 480.0,\n            "var": 0.0,\n            "q05": 480.0,\n            "q25": 480.0,\n            "q50": 480.0,\n            "q75": 480.0,\n            "q95": 480.0\n          },\n          "aspect": {\n            "n": 40,\n            "mean": 1.3333333333333333,\n            "var": 0.0,\n            "q05": 1.3333333333333333,\n            "q25": 1.3333333333333333,\n            "q50": 1.3333333333333333,\n            "q75": 1.3333333333333333,\n            "q95": 1.3333333333333333\n          },\n          "frame_y_mean": {\n            "n": 40,\n            "mean": 129.3877598470052,\n            "var": 227.04159751607412,\n            "q05": 108.03277636718751,\n            "q25": 127.07451578776042,\n            "q50": 133.569970703125,\n            "q75": 137.4266650390625,\n            "q95": 142.24065006510418\n          },\n          "y_mean": {\n            "n": 40,\n            "mean": 150.14063701629638,\n            "var": 448.04424967062334,\n            "q05": 105.40777053833008,\n            "q25": 142.20550918579102,\n            "q50": 154.28058624267578,\n            "q75": 165.46496200561523,\n            "q95": 171.9572799682617\n          },\n          "y_std": {\n            "n": 40,\n            "mean": 63.410143756866454,\n            "var": 84.45266875594243,\n            "q05": 47.934397315979005,\n            "q25": 59.69694900512695,\n            "q50": 64.1294059753418,\n            "q75": 69.64577865600586,\n            "q95": 77.99204025268554\n          },\n          "y_p10": {\n            "n": 40,\n            "mean": 41.933959406614306,\n            "var": 958.1521278850385,\n            "q05": 2.6368062615394594,\n            "q25": 18.843915605545057,\n            "q50": 32.772714614868164,\n            "q75": 61.99440174102783,\n            "q95": 109.34296669006349\n          },\n          "y_p50": {\n            "n": 40,\n            "mean": 173.0009801864624,\n            "var": 645.5022337511291,\n            "q05": 124.96294689178467,\n            "q25": 163.11760330200195,\n            "q50": 181.49776458740234,\n            "q75": 188.64525032043457,\n            "q95": 193.22096557617186\n          },\n          "y_p90": {\n            "n": 40,\n            "mean": 210.17519733428952,\n            "var": 295.3613638653209,\n            "q05": 183.68099594116222,\n            "q25": 208.27590675354006,\n            "q50": 213.28709335327147,\n            "q75": 216.74776496887208,\n            "q95": 233.09896827697756\n          },\n          "contrast": {\n            "n": 40,\n            "mean": 168.2412368774414,\n            "var": 1074.1118161923218,\n            "q05": 99.46612167358398,\n            "q25": 158.83081817626953,\n            "q50": 173.03665924072266,\n            "q75": 189.48519897460938,\n            "q95": 211.66148376464844\n          },\n          "very_bright": {\n            "n": 40,\n            "mean": 0.004967583550347222,\n            "var": 0.00026568657040228086,\n            "q05": 0.0,\n            "q25": 0.0,\n            "q50": 0.00029839409722222225,\n            "q75": 0.002163357204861111,\n            "q95": 0.018929036458333288\n          },\n          "very_dark": {\n            "n": 40,\n            "mean": 0.04101901584201389,\n            "var": 0.0017355358596017327,\n            "q05": 0.0003350151909722223,\n            "q25": 0.014112684461805556,\n            "q50": 0.023491753472222224,\n            "q75": 0.05970594618055555,\n            "q95": 0.14106445312499996\n          },\n          "detail": {\n            "n": 40,\n            "mean": 10.617775154113769,\n            "var": 15.142611563012437,\n            "q05": 4.960670614242554,\n            "q25": 8.365247249603271,\n            "q50": 10.384588241577148,\n            "q75": 11.909588098526001,\n            "q95": 16.43147802352905\n          },\n          "lab_a": {\n            "n": 40,\n            "mean": -3.1611328125,\n            "var": 1.1246114730834962,\n            "q05": -4.60703125,\n            "q25": -3.75,\n            "q50": -3.328125,\n            "q75": -2.48046875,\n            "q95": -1.2265625000000004\n          },\n          "lab_b": {\n            "n": 40,\n            "mean": 14.0974609375,\n            "var": 7.331787071228026,\n            "q05": 10.19765625,\n            "q25": 11.80078125,\n            "q50": 13.8203125,\n            "q75": 16.4296875,\n            "q95": 18.154687499999998\n          },\n          "lighting_range": {\n            "n": 40,\n            "mean": 56.37495832443237,\n            "var": 835.1875440829211,\n            "q05": 22.528936004638673,\n            "q25": 34.36559295654297,\n            "q50": 54.25605773925781,\n            "q75": 70.88351821899414,\n            "q95": 116.78360519409178\n          },\n          "r_p10": {\n            "n": 40,\n            "mean": 48.72812503933907,\n            "var": 966.3664832597153,\n            "q05": 7.367291672229768,\n            "q25": 26.8125,\n            "q50": 39.375,\n            "q75": 67.0510410308838,\n            "q95": 113.98833583831788\n          },\n          "r_p50": {\n            "n": 40,\n            "mean": 179.26197872161865,\n            "var": 661.7000408236996,\n            "q05": 132.64375,\n            "q25": 169.18749618530273,\n            "q50": 188.64583587646484,\n            "q75": 194.12499618530273,\n            "q95": 200.77708358764647\n          },\n          "r_p90": {\n            "n": 40,\n            "mean": 214.31541709899903,\n            "var": 282.4515446650902,\n            "q05": 187.3104118347168,\n            "q25": 210.65625,\n            "q50": 216.91041564941415,\n            "q75": 221.28125381469727,\n            "q95": 240.01041717529296\n          },\n          "g_p10": {\n            "n": 40,\n            "mean": 41.64218744665385,\n            "var": 1019.8722548788097,\n            "q05": 0.48687498867512474,\n            "q25": 17.690624952316284,\n            "q50": 32.35416603088379,\n            "q75": 62.35000019073487,\n            "q95": 110.67562660217285\n          },\n          "g_p50": {\n            "n": 40,\n            "mean": 175.17812471389772,\n            "var": 661.0801369134093,\n            "q05": 126.98125,\n            "q25": 164.6041717529297,\n            "q50": 184.14583206176758,\n            "q75": 191.56250762939453,\n            "q95": 195.47708282470703\n          },\n          "g_p90": {\n            "n": 40,\n            "mean": 212.95385398864747,\n            "var": 299.5880323434888,\n            "q05": 185.92291641235352,\n            "q25": 211.34374618530273,\n            "q50": 216.0833282470703,\n            "q75": 219.43750381469727,\n            "q95": 235.725\n          },\n          "b_p10": {\n            "n": 40,\n            "mean": 25.2124998831749,\n            "var": 711.475031435818,\n            "q05": 0.0,\n            "q25": 4.603124964237219,\n            "q50": 14.979166507720947,\n            "q75": 41.250000953674316,\n            "q95": 88.94229175567627\n          },\n          "b_p50": {\n            "n": 40,\n            "mean": 145.1359375,\n            "var": 583.7990686353174,\n            "q05": 100.77291908264161,\n            "q25": 138.78125,\n            "q50": 150.83333587646484,\n            "q75": 161.79166412353516,\n            "q95": 166.6375\n          },\n          "b_p90": {\n            "n": 40,\n            "mean": 186.04812545776372,\n            "var": 340.0548890277736,\n            "q05": 159.0110397338869,\n            "q25": 181.09375,\n            "q50": 188.01458740234384,\n            "q75": 197.07291793823242,\n            "q95": 209.48958816528318\n          },\n          "dark_ridge": {\n            "n": 40,\n            "mean": 0.14239705403645836,\n            "var": 0.00478912869307362,\n            "q05": 0.05533989800347222,\n            "q25": 0.09922960069444445,\n            "q50": 0.1377902560763889,\n            "q75": 0.1697048611111111,\n            "q95": 0.24119873046874996\n          }\n        }\n      },\n      "engineering_bounds": {\n        "chroma_limit_lab": 3,\n        "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n        "texture_gain_limits": [\n          1.0,\n          1.0\n        ],\n        "blur_max_roi64": 0.0,\n        "lighting": {\n          "max_brightness_offset": 32,\n          "contrast": [\n            0.8,\n            1.25\n          ]\n        }\n      },\n      "limitations": [\n        "Within-person variation includes pose/content; not an isolated camera calibration",\n        "Texture and focus proxies are not independent physical measurements",\n        "Global disease colors/textures are not matched to unlabeled participants",\n        "Four participants are not a general camera population or independent Target Test",\n        "Source/Target brightness and contrast also contain disease and composition differences",\n        "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n        "Bounds are engineering choices, not validated physical camera calibration",\n        "This profile changes direction and calibration policy as well as amplitude",\n        "The existing Source Test has already been inspected; select on Validation only",\n        "Revised chroma direction remains confounded with disease, person and body region",\n        "Consistency across omissions is a sensitivity check, not independent validation",\n        "Focus gates are content-dependent proxies; no physical camera equivalence established"\n      ],\n      "previous_visual_review": {\n        "profile_version": "measured_bounded_v3",\n        "train_visual_review": "approved",\n        "review_note": "Codex inspected all 120 fixed Train before/after previews (pages 1-15). No gross hair/scale boundary loss or artificial geometry observed at preview resolution. Bounded brightness operator retained for a limited pilot, not a clinical validation. The full Source leakage audit is repeated before training; model inputs and augmentation use float32 RGB 0-255."\n      },\n      "calibration_version": "direction_and_focus_audited_v1",\n      "calibration_evidence": {\n        "source_role": "retained_original_train",\n        "target_role": "unlabeled_calibration",\n        "source_class_balanced_anchors": {\n          "y_p50": 160.440185546875,\n          "contrast": 70.79170608520508\n        },\n        "source_measurement_coverage": {\n          "y_p50": {\n            "class_medians": [\n              150.06185150146484,\n              160.9347801208496,\n              160.440185546875,\n              157.4539794921875,\n              165.97833251953125\n            ],\n            "coverage_by_class": [\n              {\n                "total": 1990,\n                "measured": 1990\n              },\n              {\n                "total": 2050,\n                "measured": 2050\n              },\n              {\n                "total": 1929,\n                "measured": 1929\n              },\n              {\n                "total": 1965,\n                "measured": 1965\n              },\n              {\n                "total": 2098,\n                "measured": 2098\n              }\n            ]\n          },\n          "contrast": {\n            "class_medians": [\n              58.59663391113281,\n              70.79170608520508,\n              83.7009048461914,\n              35.322113037109375,\n              71.07597732543945\n            ],\n            "coverage_by_class": [\n              {\n                "total": 1990,\n                "measured": 1990\n              },\n              {\n                "total": 2050,\n                "measured": 2050\n              },\n              {\n                "total": 1929,\n                "measured": 1929\n              },\n              {\n                "total": 1965,\n                "measured": 1965\n              },\n              {\n                "total": 2098,\n                "measured": 2098\n              }\n            ]\n          }\n        },\n        "target_equal_participant_condition_medians": {\n          "y_p50": [\n            188.3468475341797,\n            162.00440216064453,\n            179.97626495361328,\n            188.29984283447266,\n            181.40353393554688,\n            180.3260040283203,\n            186.1680450439453,\n            161.74720764160156\n          ],\n          "contrast": [\n            145.20181274414062,\n            173.9080047607422,\n            174.50856018066406,\n            177.37515258789062,\n            172.16531372070312,\n            183.07321166992188,\n            168.57965087890625,\n            189.6483917236328\n          ]\n        },\n        "raw_brightness_offsets": [\n          15.043113708496094,\n          26.26080894470215\n        ],\n        "raw_contrast_ratios": [\n          2.4193356465249507,\n          2.5257149070993528\n        ],\n        "brightness_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ],\n        "hair_global_contrast": "identity; coverage and content remain confounded",\n        "revision_policy": "direction_and_focus_audited_v1"\n      },\n      "revision_evidence": {\n        "color_direction": {\n          "a": {\n            "source_after_lighting_class_balanced_anchor": -2.96875,\n            "target_equal_group_medians": {\n              "P01/alt": -2.984375,\n              "P01/ref": -1.671875,\n              "P02/alt": -3.625,\n              "P02/ref": -4.25,\n              "P03/alt": -3.640625,\n              "P03/ref": -2.828125,\n              "P04/alt": -3.640625,\n              "P04/ref": -2.46875\n            },\n            "full_direction": 0,\n            "leave_one_person_out_directions": {\n              "P01": -1,\n              "P02": 0,\n              "P03": 0,\n              "P04": 0\n            },\n            "direction_stable": false,\n            "within_person_absolute_q75_limit": 1.171875,\n            "proposed_range": [\n              0.0,\n              0.0\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          },\n          "b": {\n            "source_after_lighting_class_balanced_anchor": 5.4453125,\n            "target_equal_group_medians": {\n              "P01/alt": 12.46875,\n              "P01/ref": 13.109375,\n              "P02/alt": 10.671875,\n              "P02/ref": 12.875,\n              "P03/alt": 16.328125,\n              "P03/ref": 15.9375,\n              "P04/alt": 14.5625,\n              "P04/ref": 14.9375\n            },\n            "full_direction": 1,\n            "leave_one_person_out_directions": {\n              "P01": 1,\n              "P02": 1,\n              "P03": 1,\n              "P04": 1\n            },\n            "direction_stable": true,\n            "within_person_absolute_q75_limit": 1.03515625,\n            "proposed_range": [\n              0.0,\n              1.03515625\n            ],\n            "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n          }\n        },\n        "color_choice": "none",\n        "focus_choice": "none",\n        "source_preview_count": 120,\n        "target_count": 40,\n        "seeds": [\n          42,\n          707,\n          1707\n        ],\n        "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n      },\n      "previous_engineering_bounds": {\n        "measured_change_fraction": 1.0,\n        "chroma_limit_lab": 3,\n        "texture_gain_limits": [\n          0.85,\n          1.15\n        ],\n        "focus_fit_grid_roi64": [\n          0,\n          1.5,\n          0.05\n        ],\n        "blur_max_roi64": 0.75,\n        "brightness_offset_limit": 32,\n        "contrast_limits": [\n          0.8,\n          1.25\n        ]\n      }\n    }\n  },\n  "selection_scope": "Fixed from Stage 4 Validation comparison among initial, stage2 and stage4; stage1 and stage3 are historical comparison-only, not newly evaluated.",\n  "prior_source_test_inspected": true,\n  "test_used_for_model_selection": false,\n  "refitted_stage4_excluded": true,\n  "runtime_sha256": {\n    "adaptation_stage4_test.py": "5d0d3874df6235ee07358f8d20566d2ad3029da675b4b8f1eb83ae3d160b17c4",\n    "adaptation_final_test.py": "e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6",\n    "adaptation_stage1.py": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a",\n    "camera_extended.py": "7a1cd1e25c025398cb06baab4b6675f0d7d6a11bd3d9f56d35424a23a6e49da1",\n    "domain_adaptation_experiment.py": "216d01c860cf977d7a8f9093605a90a5f4cda495c4d4f363b5862e8b69d63d2b"\n  },\n  "builder_sha256": "400cc1e73975df32adde2b1d46ab82adfb2bf4aad261b99bdf9ec735b2fcdff4"\n}\n')
    for name, expected in {'adaptation_stage4_test.py': '5d0d3874df6235ee07358f8d20566d2ad3029da675b4b8f1eb83ae3d160b17c4', 'adaptation_final_test.py': 'e8b943f8a9a81fe1178caa04f493b4ff6a1d04ccfe9a8d3e31970d52d3dee1b6', 'adaptation_stage1.py': '6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a', 'camera_extended.py': '7a1cd1e25c025398cb06baab4b6675f0d7d6a11bd3d9f56d35424a23a6e49da1', 'domain_adaptation_experiment.py': '216d01c860cf977d7a8f9093605a90a5f4cda495c4d4f363b5862e8b69d63d2b'}.items():
        staged = runtime / "src/mediflow_datasets" / name
        if hashlib.sha256(staged.read_bytes()).hexdigest() != expected:
            raise ValueError(f"평가 코드 사본 검사 실패: {name}")
    print("평가 코드 사본 검사 완료", flush=True)
    env = {**os.environ, "PYTHONPATH": str(runtime / "src"), "PYTHONIOENCODING": "utf-8",
           "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1", "MPLBACKEND": "Agg"}
    launch = """from pathlib import Path
import sys
from mediflow_datasets import domain_adaptation_batch as batch
from mediflow_datasets.adaptation_stage1_colab import run_live
import subprocess
work, runtime = map(Path, sys.argv[1:3])
print("Python·TensorFlow 환경과 GPU 확인 중", flush=True)
python, versions = batch.prepare_python(work, executor=run_live)
print("확인된 실행 환경", versions, flush=True)
if not versions["gpu"]:
    raise RuntimeError("Colab에서 GPU 런타임을 선택하고 다시 실행하세요.")
env = batch.python_environment(python, runtime)
check = subprocess.run([str(python), "-c", "import matplotlib"], env=env, capture_output=True)
if check.returncode:
    subprocess.run([str(python), "-m", "pip", "install", "matplotlib>=3.8,<4"], env=env, check=True)
run_live(
    [str(python), "-m", "mediflow_datasets.adaptation_stage4_test", *sys.argv[3:]],
    env, work / "test.log"
)
"""
    command = [sys.executable, "-u", "-c", launch, str(work), str(runtime),
               "--runtime", str(runtime), "--stages", *map(str, stages),
               "--datasets", str(project / "datasets"), "--plan", str(plan_path),
               "--output", str(out), "--work", str(work)]
    with subprocess.Popen(
        command, env=env, start_new_session=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1,
    ) as process, log_path.open("x", encoding="utf-8") as log:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            if return_code:
                raise RuntimeError(
                    f"최종 Test 실행 실패 · 종료 코드 {return_code} · "
                    f"위의 실제 오류 또는 Drive 로그를 확인하세요. {log_path}"
                )
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
            for diagnostic in work.glob("*.log"):
                shutil.copyfile(diagnostic, log_root / f"{run_id}_{diagnostic.name}")
    local_zip = Path("/content/MediFlow_Stage4_Test_Results_20261111.zip")
    shutil.make_archive(str(work / "results"), "zip", root_dir=out)
    shutil.copyfile(work / "results.zip", local_zip)
    shutil.copyfile(local_zip, out.with_suffix(".zip"))
print("완료:", out / "test_comparison.csv")
import pandas as pd
from IPython.display import Image, display
from google.colab import files
display(pd.read_csv(out / "test_comparison.csv", dtype=str, keep_default_na=False))
display(Image(filename=str(out / "plots/test_comparison.png")))
display(Image(filename=str(out / "plots/stage4_confusion_matrices.png")))
files.download(str(local_zip))
